# RSNA Knee — RGB 2.5D voisins DICOM, 320

**Shard 9/12** (CPU, internet off). **1777 series**, ~10.6 Go.

UID **figés** depuis `data/crop130_rgb25d/manifest_rgb25d_proto_part*.csv` (6 slots v20b, contraste exact). Même recette que le proto 224, seul le resize passe à **320** :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 %
2. bande profondeur **20–80 %**
3. linspace **24** centres dans la bande
4. pour chaque centre `k`, **`k-1, k, k+1` dans le volume DICOM**
5. resize **320×320**, uint8, percentiles 1–99

Shape `(24, 3, 320, 320)`, canaux R=k-1, G=k, B=k+1.

Fichiers dans `/kaggle/working/` :
- `volumes_crop130_rgb25d_320_part9.zip`
- `manifest_rgb25d_320_part9.csv`
- `RECIPE.json`

Download : `python -X utf8 kaggle_DICOM_exctract/download_rgb25d_320.py` → `data/crop130_rgb25d_320/`.

In [ ]:
SHARD_ID = 9
N_SHARDS = 12

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 320
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:
import json
jobs = json.loads('[{"study":"1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585","series":"1.2.826.0.1.3680043.8.498.27845969797953115400959653384511165237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10013663742400736029415768216599146542","series":"1.2.826.0.1.3680043.8.498.11982164807463634778369544669639739403","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10018749256316287542884435387067561101","series":"1.2.826.0.1.3680043.8.498.42759405933119955877969739566391261556","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10025765742726180988537955593315007559","series":"1.2.826.0.1.3680043.8.498.52503045794545563216216974185473269584","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10029168398037940681915299760479434938","series":"1.2.826.0.1.3680043.8.498.61060692464231753798040762152369852425","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10048842414095994814596043262132965018","series":"1.2.826.0.1.3680043.8.498.41751407285593261301937942061885125240","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10059200627175320481920009076697103801","series":"1.2.826.0.1.3680043.8.498.99708703036078965966876737788881385029","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10066945398485042741116657288959323922","series":"1.2.826.0.1.3680043.8.498.11547431245652234248420888650946548426","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10081117556977868957150377234052259018","series":"1.2.826.0.1.3680043.8.498.11993387778168096411963985619648870481","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10087350795575215080051569638065974347","series":"1.2.826.0.1.3680043.8.498.96012886433868321293915735968338210998","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10090136633727522725960302994815319097","series":"1.2.826.0.1.3680043.8.498.26739695880351722615408236206223622714","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149","series":"1.2.826.0.1.3680043.8.498.12451507871565834868863189854063726441","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10096216489317602387766193675918767557","series":"1.2.826.0.1.3680043.8.498.46174887318465327763817364318575843548","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10112440805412493412191181061985753636","series":"1.2.826.0.1.3680043.8.498.89837468161028677597270115982511521187","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10120841513315425967980213160839994882","series":"1.2.826.0.1.3680043.8.498.36258630042254758729305084838122618231","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10123751739767586025744123799948754343","series":"1.2.826.0.1.3680043.8.498.98027629871036248874002203265341743237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10128954897472795975323353442305671586","series":"1.2.826.0.1.3680043.8.498.18470434175523388200744086637879974300","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10132169498032330070492330913783717864","series":"1.2.826.0.1.3680043.8.498.69214460339246844021907171343113321393","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10136486417446492428821313562961937182","series":"1.2.826.0.1.3680043.8.498.11173210030954491208464370452342123067","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10144735099702632881322032871591573824","series":"1.2.826.0.1.3680043.8.498.71635123504520367676841601899689926352","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10148037169326559689992758180208780221","series":"1.2.826.0.1.3680043.8.498.75936723106952608430637802524046218935","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10149967031679082348901055550415519790","series":"1.2.826.0.1.3680043.8.498.26551486547969393077464620776076183298","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10155611590208164695256621422536643320","series":"1.2.826.0.1.3680043.8.498.11081869777466496980946756456652430317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10159551371305936155888341171022480536","series":"1.2.826.0.1.3680043.8.498.27634092237325443249951719414199451510","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10162720374291180196973998234794414415","series":"1.2.826.0.1.3680043.8.498.29133697488265813918029705241253190966","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269","series":"1.2.826.0.1.3680043.8.498.39687308593816556791491064151179747304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10182565688010898600749222617373997197","series":"1.2.826.0.1.3680043.8.498.32157658084222604910876248410927477104","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10189392162056736163704940747328177364","series":"1.2.826.0.1.3680043.8.498.11740732272602659379392281696328258247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10197565004992937722841731691775578390","series":"1.2.826.0.1.3680043.8.498.80281164867492700984572159185963507377","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10201197020583611347810070328854760801","series":"1.2.826.0.1.3680043.8.498.69801199489032763877908268931609114736","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10207134048598925821261298801127950516","series":"1.2.826.0.1.3680043.8.498.72901826894628879008397330239124974612","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10211034309703220222647821582770542129","series":"1.2.826.0.1.3680043.8.498.11112142671354810696616635369245080090","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10218458664774126578027318307405130821","series":"1.2.826.0.1.3680043.8.498.29388585808164899891636914715535480012","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10239418182819270860819533058081041576","series":"1.2.826.0.1.3680043.8.498.80447059383586328104459885231756893726","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10241639721703882165429374175491428414","series":"1.2.826.0.1.3680043.8.498.66329718510489698682846385479529824620","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10247930792182001003497189021027466833","series":"1.2.826.0.1.3680043.8.498.21502069353013131801996504092255889015","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10254828365033984708960944586564394286","series":"1.2.826.0.1.3680043.8.498.64923769131146081353672711856062427976","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10256441455107860474283716992657597873","series":"1.2.826.0.1.3680043.8.498.33844238698794294558665505131940955613","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10262024906324456863812446480254600405","series":"1.2.826.0.1.3680043.8.498.85985794862127095585192707633430420116","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10273861326753707436672556008991334542","series":"1.2.826.0.1.3680043.8.498.12244627660297417879533300891039965883","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10274956886032850418742861294277331382","series":"1.2.826.0.1.3680043.8.498.10240055476834460412982667755942563040","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10284446538566255572004044331831825072","series":"1.2.826.0.1.3680043.8.498.12060642123164043431257858441129259459","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10299618611626190927173051117228647808","series":"1.2.826.0.1.3680043.8.498.25101442181336970122086565575227563009","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10304242666970626179776896768140580106","series":"1.2.826.0.1.3680043.8.498.21647436319077947464496009531361339719","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10309477218703786877612946261946268132","series":"1.2.826.0.1.3680043.8.498.73686837107266161783005896227858611804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10314240078685804110739316196999571466","series":"1.2.826.0.1.3680043.8.498.54670494116226982905211664727016694020","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10321556388278642027434955207617064548","series":"1.2.826.0.1.3680043.8.498.12947525228617236458008333275468710927","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10322724865012212544754724404226583315","series":"1.2.826.0.1.3680043.8.498.12870978401576394708550583634587068822","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10327608571317884761792196888985963332","series":"1.2.826.0.1.3680043.8.498.50801897070902191483468881621106649366","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10336221220793312503236159680373829726","series":"1.2.826.0.1.3680043.8.498.63813877352290123090095847150581409875","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10348611327224519953673858098392135936","series":"1.2.826.0.1.3680043.8.498.25334951916774168418069841113346563539","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10351168527688244230248760520733020605","series":"1.2.826.0.1.3680043.8.498.12882412442996270853359137017204218578","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10352011938574988727803498915608467179","series":"1.2.826.0.1.3680043.8.498.51328118613639433991332641995495843243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10361030765327974510262894062047216433","series":"1.2.826.0.1.3680043.8.498.81709598870575871288492890141932594994","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10374791734880310643587680407133671218","series":"1.2.826.0.1.3680043.8.498.55818112171837421187961781130015730897","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10382074558506016193900889176239337801","series":"1.2.826.0.1.3680043.8.498.40621827593274408107813177177150954786","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10387383212921524066516723428665512341","series":"1.2.826.0.1.3680043.8.498.32384701845788175400093680690979011647","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10391178814963564201551003358372148419","series":"1.2.826.0.1.3680043.8.498.91137285493899938814643274473978270866","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10401813253995823397883215400000450724","series":"1.2.826.0.1.3680043.8.498.59679591899172017478992721044886020146","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10407865232593182855530878478701744409","series":"1.2.826.0.1.3680043.8.498.85632429287138635196815844563432536695","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10415616591430643294142401482460086713","series":"1.2.826.0.1.3680043.8.498.18924856789055532756179119124569633981","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10419958821528108176389641942168280311","series":"1.2.826.0.1.3680043.8.498.41358991973522613871019529966438935484","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10422536395266111565753575369125069689","series":"1.2.826.0.1.3680043.8.498.43107589186828718052106886327986901933","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10425131723456547056365404016184663035","series":"1.2.826.0.1.3680043.8.498.77571392570892735939256892039248691247","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10432568348356281020515782178064869805","series":"1.2.826.0.1.3680043.8.498.57617951351340684183238841816773211696","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10442828926117258624043497279351962738","series":"1.2.826.0.1.3680043.8.498.43029114884517979884664605067493305460","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10447920208680921688438738178539475509","series":"1.2.826.0.1.3680043.8.498.75868812501900137621930248498636947349","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10455589411958266725060942213473374667","series":"1.2.826.0.1.3680043.8.498.10299525200880174330696187663982648634","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10470282959537049979906878778884086884","series":"1.2.826.0.1.3680043.8.498.39392911873677781144558060911235384662","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10475397432011699698577978597417345091","series":"1.2.826.0.1.3680043.8.498.64411944758384363138097958943147049343","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10492484093480313596330475837639172625","series":"1.2.826.0.1.3680043.8.498.96808970977450883057717727514921209301","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10495855084258410825408867079175646590","series":"1.2.826.0.1.3680043.8.498.76256858144845895774350808484652066510","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10502440129091559437492859441158882078","series":"1.2.826.0.1.3680043.8.498.48935765974368981722519684278093867559","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10511712617129216774539441890488742560","series":"1.2.826.0.1.3680043.8.498.70872280662254251771947915537752865182","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10519566075312736546346469231439794627","series":"1.2.826.0.1.3680043.8.498.12327973997698835621835033415742695767","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10522585237725235866013737575867341946","series":"1.2.826.0.1.3680043.8.498.23620199704815482140329754787371560009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10532708800468111850607752497432656694","series":"1.2.826.0.1.3680043.8.498.42660913846820227996641942093345390404","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10534260101860670685224894870019533364","series":"1.2.826.0.1.3680043.8.498.50581877347630891477845107780255805465","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10538502578022388392340022120166213828","series":"1.2.826.0.1.3680043.8.498.77644949118635082299832575995816199519","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10540980111087702906123811593906850156","series":"1.2.826.0.1.3680043.8.498.10660852664314576512773973710885619647","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10541227283184832783436242773084724216","series":"1.2.826.0.1.3680043.8.498.12724367611095945089332592654664231065","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10546313646515875953943156661172963953","series":"1.2.826.0.1.3680043.8.498.81163844184126176866270673061790307369","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10563016328082326759266280856730347260","series":"1.2.826.0.1.3680043.8.498.64167731514143465445799706258993480719","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10578013533179563448221888658506023084","series":"1.2.826.0.1.3680043.8.498.25888761947436322463902046472744688442","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10587511485561679190576188107774736852","series":"1.2.826.0.1.3680043.8.498.24147134651890455991969616143909533778","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10590148272001480024086117269799629816","series":"1.2.826.0.1.3680043.8.498.58258467705489100594537722261484038632","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10593694750925047629619718537436931628","series":"1.2.826.0.1.3680043.8.498.70123357669415805656737336925519503721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10600654987896381983520458496207741798","series":"1.2.826.0.1.3680043.8.498.13312095336842228726408531221265860304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10601812195848580582490508608655011196","series":"1.2.826.0.1.3680043.8.498.55474364078751475991666428812678996421","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10614508096225944703643482200835890514","series":"1.2.826.0.1.3680043.8.498.31572433113041186383228189623522004802","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10619486897279334206912764495067756449","series":"1.2.826.0.1.3680043.8.498.13395310871581406766805713529946597000","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10626071553432761206594400919097854593","series":"1.2.826.0.1.3680043.8.498.17282817463363779415928145052087653412","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10627019153481144392726933328057818971","series":"1.2.826.0.1.3680043.8.498.10346088613010110474605983256744286881","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10631238417904652761254062374051500506","series":"1.2.826.0.1.3680043.8.498.10524474659425920579843251311720556593","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10648346793256473462304051626410900340","series":"1.2.826.0.1.3680043.8.498.10001478771378979438868923030424928430","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10652685418462611085839874321267337079","series":"1.2.826.0.1.3680043.8.498.26985152860198593202676684376555208355","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10656388982978239967297620214873636996","series":"1.2.826.0.1.3680043.8.498.92873796631048229621656544554966617630","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10657584461509809047185519742966035356","series":"1.2.826.0.1.3680043.8.498.98399571241325115894203977486280947659","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10661650555947788912511478165000737040","series":"1.2.826.0.1.3680043.8.498.12024607756616740176927191712472776231","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10674673641872871321890995010616205087","series":"1.2.826.0.1.3680043.8.498.23247160794046439464701551526864300130","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10684020894033412667820478260168097071","series":"1.2.826.0.1.3680043.8.498.69646977943104855483908141936648005100","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10688670034893539180306173203903350286","series":"1.2.826.0.1.3680043.8.498.10408937669544138983401006187848394234","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10704240846075023334288326143979277080","series":"1.2.826.0.1.3680043.8.498.86801080204035534557614265996532522259","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10715956446507906169485719013515169244","series":"1.2.826.0.1.3680043.8.498.24662433267749616403811651347481592042","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10726624821670417064567481545750543282","series":"1.2.826.0.1.3680043.8.498.11763075007074530771528191590522142214","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10731612202208916303411280624156291171","series":"1.2.826.0.1.3680043.8.498.10560919746737620100880157330531452775","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10739653650865735867752564520952904802","series":"1.2.826.0.1.3680043.8.498.13729517091028270137956278647758250669","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10748806810865802591313438958975301375","series":"1.2.826.0.1.3680043.8.498.25134828673023620236499920910521636569","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10751346650129497348457481341700957445","series":"1.2.826.0.1.3680043.8.498.37447209175207308675029027317449518955","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10759288702627176691392048851622294054","series":"1.2.826.0.1.3680043.8.498.70683346493489193287066262182621693165","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10761658878615069270304502752374858610","series":"1.2.826.0.1.3680043.8.498.86100924875282251700683411967946424393","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10765617676329232046176300833850901740","series":"1.2.826.0.1.3680043.8.498.70040316994257134083028919382038280463","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10770877369291591553624215762368014357","series":"1.2.826.0.1.3680043.8.498.25573131331180306717123935696050392797","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10782963448708192069959366583555573334","series":"1.2.826.0.1.3680043.8.498.34349038239150700819574176393895222956","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10790689419760453870458170966064168002","series":"1.2.826.0.1.3680043.8.498.13770697509796699263744103378604200918","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10808663384878038371710260386807468588","series":"1.2.826.0.1.3680043.8.498.30298513951851818330174016569330112867","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10818980124981334596808438908811623528","series":"1.2.826.0.1.3680043.8.498.10727988469507447447342962788705581536","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10830288389793651695251406408666084306","series":"1.2.826.0.1.3680043.8.498.12421766865602641995571623155131343218","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10839126901366636822676830560774505865","series":"1.2.826.0.1.3680043.8.498.77134810976271971886385069328216524007","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10843917293035279597622029625809500972","series":"1.2.826.0.1.3680043.8.498.12839036332986557908481042284511065518","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10850742579000782417673172155504794301","series":"1.2.826.0.1.3680043.8.498.96853634365128226174332512295817151859","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10854012268828835845581966634225522931","series":"1.2.826.0.1.3680043.8.498.67583574379506841942044502463310044656","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10855290714402117092990651362697310270","series":"1.2.826.0.1.3680043.8.498.94978801886429346268220177480964043900","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10862481204237376580557516731952961213","series":"1.2.826.0.1.3680043.8.498.84418401551151739243426828752272137047","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10876177342063734507664188801203192617","series":"1.2.826.0.1.3680043.8.498.10210630698099464281664914547742562311","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10886583574130697346788286647281863932","series":"1.2.826.0.1.3680043.8.498.30106165779171049567812589919187638977","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10896245499713818848442117831482755673","series":"1.2.826.0.1.3680043.8.498.72085367543686823170454459454354339610","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10903003118372337016190215211789983264","series":"1.2.826.0.1.3680043.8.498.48609555377186108014484244796203313624","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10918580785296934228715514366408269433","series":"1.2.826.0.1.3680043.8.498.12449993287894667086973661976428586805","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10920269679387171659375091645187772303","series":"1.2.826.0.1.3680043.8.498.92455600591402675895558733986191140913","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10927716750119165925963563081261605359","series":"1.2.826.0.1.3680043.8.498.12289016128082253333432104317533511551","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10931111730564885182398163970692599398","series":"1.2.826.0.1.3680043.8.498.56890150707652366259277436073998436960","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10937181824204907065071760564745305722","series":"1.2.826.0.1.3680043.8.498.58719356744521106310581371640024901831","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10938719716535758543863717421521822626","series":"1.2.826.0.1.3680043.8.498.12234139798209022789162297307721406210","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10951328287594724738476663200798896670","series":"1.2.826.0.1.3680043.8.498.65470143831951705738988853613175609183","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10960770430559425421609885784366918575","series":"1.2.826.0.1.3680043.8.498.22331187374917580606556952211545398417","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10965573028231658307306828073340279415","series":"1.2.826.0.1.3680043.8.498.13340397223679645014143653923900034031","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10981062990502278795793220223076241424","series":"1.2.826.0.1.3680043.8.498.66992019675602202786980608747457444594","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10992875360571368325079907908221899699","series":"1.2.826.0.1.3680043.8.498.11611180622191992113742388040254567928","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994854354532517072359864903910225174","series":"1.2.826.0.1.3680043.8.498.12795566765509597739832248221669986244","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10995980880227536691827030731644023480","series":"1.2.826.0.1.3680043.8.498.35944846554473368813529371807566252796","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10998405409474759941744305665540376339","series":"1.2.826.0.1.3680043.8.498.98108336645013977100056131173594421075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11010004366885474301784150582039137111","series":"1.2.826.0.1.3680043.8.498.39621882836253405688047901677860731512","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11014091506864318694851337553197477906","series":"1.2.826.0.1.3680043.8.498.11214645231035703782350271821706221616","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11043081829401721095803441219553837608","series":"1.2.826.0.1.3680043.8.498.10882733772712847894551644317305234150","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11045459770905584149950259201414074545","series":"1.2.826.0.1.3680043.8.498.12416195563204440297932626779551277054","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11050604958275820465127935558416408008","series":"1.2.826.0.1.3680043.8.498.19051614206442309771354206847266835699","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11057883757434391643878168168845592475","series":"1.2.826.0.1.3680043.8.498.42336455299410288049765595714665192973","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11067200656721609390997819562401839124","series":"1.2.826.0.1.3680043.8.498.62394717994656841104031592597687522235","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11077406555624926104628200252710914930","series":"1.2.826.0.1.3680043.8.498.14296237954597402119768864897549965304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11084010833221855396433326769153950681","series":"1.2.826.0.1.3680043.8.498.58408971589922585653403754932244436552","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11091958567356981151034727948765010881","series":"1.2.826.0.1.3680043.8.498.10834164999259035141561523209903199560","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11110575656494405149601490027444484170","series":"1.2.826.0.1.3680043.8.498.10257076873470624137022887521892653390","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11113442398569287228475456000446876768","series":"1.2.826.0.1.3680043.8.498.11409028635806471488503520295909259503","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11123446342116837882811834332727990911","series":"1.2.826.0.1.3680043.8.498.91360426418842124443136227460128866285","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11126383719852491587081347886000714259","series":"1.2.826.0.1.3680043.8.498.12194927839973852494088528023258144722","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11129751904857491327872353316480050290","series":"1.2.826.0.1.3680043.8.498.52933658544984806032332845440403330923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11135115047200279264914390528987153419","series":"1.2.826.0.1.3680043.8.498.10341114575649140662688968041392278844","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11148939654362681630811787929668902046","series":"1.2.826.0.1.3680043.8.498.10596815023177683963582370050913352236","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11150193626240150189232364321568347937","series":"1.2.826.0.1.3680043.8.498.11376290476663840343620026650808953735","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11152585968424897353533125085307104033","series":"1.2.826.0.1.3680043.8.498.56386917286563827631089891953933694219","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11156884483511690263238401598901614096","series":"1.2.826.0.1.3680043.8.498.42544283891981713453189884565904554413","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11180457573346204459956051956213528997","series":"1.2.826.0.1.3680043.8.498.54786552374529179052792214898308872149","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11184968672176462852561645279750399747","series":"1.2.826.0.1.3680043.8.498.10011730488681862782320879441610733914","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11191015483427903713800997920940844550","series":"1.2.826.0.1.3680043.8.498.11228334482669520231597488596538738992","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11201741528059502111642265469347871878","series":"1.2.826.0.1.3680043.8.498.10005488803102962734659960996323792804","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11208107516750574657838495761140109258","series":"1.2.826.0.1.3680043.8.498.11531847212425259717754748760078151813","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11210361780779474361817208167009080083","series":"1.2.826.0.1.3680043.8.498.11874005327436999463204950933612097829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11214158717183172156320144105486247892","series":"1.2.826.0.1.3680043.8.498.78314978491876402312622528533451128022","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11219813299300915347039206446596901205","series":"1.2.826.0.1.3680043.8.498.10097583488988702954892237769096852455","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11223691171233118900160265287691267884","series":"1.2.826.0.1.3680043.8.498.73978734951733065314479810509883006002","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11229566047228169850473554486278603754","series":"1.2.826.0.1.3680043.8.498.83448427760610848396038387142125548246","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11244115315611630362929576340631670750","series":"1.2.826.0.1.3680043.8.498.88940532792591133438525237004929250015","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11254890841796835340329521299102281678","series":"1.2.826.0.1.3680043.8.498.72872783340662865226615382588525834289","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11260248089108026217929330361529065683","series":"1.2.826.0.1.3680043.8.498.54787395205905927653372682222456459313","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11268635423021790546063903457498005204","series":"1.2.826.0.1.3680043.8.498.10894577743852742621049937211870178566","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11275711698468745807758222802302685022","series":"1.2.826.0.1.3680043.8.498.34024429425164200376657098397836244463","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11276732020240247755618413418003401291","series":"1.2.826.0.1.3680043.8.498.22704040737009228293586060161533680132","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11282162920338602001456704018617788412","series":"1.2.826.0.1.3680043.8.498.81161507413957611238203708504335524160","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11287173314232027941677698384842764593","series":"1.2.826.0.1.3680043.8.498.97727977007706440977998910031122085491","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11294885816514924203337081832122619605","series":"1.2.826.0.1.3680043.8.498.82205514735791281341075415998940550295","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11296867854260286881418279496022384461","series":"1.2.826.0.1.3680043.8.498.20472254924515250809261425283365124437","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11302750434639976191333636496428935632","series":"1.2.826.0.1.3680043.8.498.11115904669609656163554793356410837754","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11314687816975146963598226343179928616","series":"1.2.826.0.1.3680043.8.498.10205605656623725155254975716115771457","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11316091466685605758061778045311969672","series":"1.2.826.0.1.3680043.8.498.54867374618942840055090661031449521891","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11334776960319816844935217444817103072","series":"1.2.826.0.1.3680043.8.498.13253546289221638866537209428208481931","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11338962847958062668991874160027887372","series":"1.2.826.0.1.3680043.8.498.10284219920313233626530492412896345428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11347466940974045182979684641300336177","series":"1.2.826.0.1.3680043.8.498.10789355882816885717918913864882914799","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11362547519125634779422991321337283241","series":"1.2.826.0.1.3680043.8.498.11072900579273686816542271419051832511","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11371400503360468962117579439852064607","series":"1.2.826.0.1.3680043.8.498.11003185826694804750408801855308460404","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11385845157446557781575804962818981994","series":"1.2.826.0.1.3680043.8.498.92442317956383258431006193112066625203","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11391095418955817124325938917797687542","series":"1.2.826.0.1.3680043.8.498.30665664024270743540827409424611467525","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11396451979734570993257791565721573849","series":"1.2.826.0.1.3680043.8.498.78284067733305502320081372457470854664","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11406381777983128446583783158882827226","series":"1.2.826.0.1.3680043.8.498.94900963408614275545850479493846564388","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11414215447313658600502835138277811260","series":"1.2.826.0.1.3680043.8.498.51280662836201853898250384089387648730","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11425683153901568541658945026861110110","series":"1.2.826.0.1.3680043.8.498.11760170619287103047972020800407031546","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11432815311029789072253610823554308183","series":"1.2.826.0.1.3680043.8.498.19313221420935126422247853244614177577","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11440217845998767853986099058965644463","series":"1.2.826.0.1.3680043.8.498.83689987005188717255706495839875078685","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11448326350892044215419380818679679579","series":"1.2.826.0.1.3680043.8.498.59090367960816296916888646841894643104","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11451448731901996559834481853733404099","series":"1.2.826.0.1.3680043.8.498.13209595503436144849689636548046859626","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11458488293461896930750018852127694875","series":"1.2.826.0.1.3680043.8.498.89120061241761105090554766908061925785","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11465586454035959390661499120751232686","series":"1.2.826.0.1.3680043.8.498.10633409006104408150388546028137841961","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11468812519969607938527116557104410121","series":"1.2.826.0.1.3680043.8.498.18570506313772761907578099172720027844","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11477000011453869751231224714283666904","series":"1.2.826.0.1.3680043.8.498.44584804607178495108742238012249165756","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11487799220518102363643849548556099719","series":"1.2.826.0.1.3680043.8.498.94242711639662046774772011317383589709","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11495133447993934376048707763538781434","series":"1.2.826.0.1.3680043.8.498.51083474795911549960586969875470018125","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11498239309891927151456629915927889165","series":"1.2.826.0.1.3680043.8.498.44333626639192528668537439970447179070","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11498845504486569595587386245872511981","series":"1.2.826.0.1.3680043.8.498.90924773257875016151787718474536008332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11506642732332295874076832453198901115","series":"1.2.826.0.1.3680043.8.498.12862978792298440711285743144662292295","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11515987024418648685676369132061127664","series":"1.2.826.0.1.3680043.8.498.93212518927519672590269285516108577286","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11516842786271998149681118182571061813","series":"1.2.826.0.1.3680043.8.498.58420552737742755437024509896155835294","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11521971289581057766555206729274639736","series":"1.2.826.0.1.3680043.8.498.11944848771632898890165082868184614896","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11532517984322509215429681648257299355","series":"1.2.826.0.1.3680043.8.498.56616439686446505902493514719594694205","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11543622448213658856759993374276266341","series":"1.2.826.0.1.3680043.8.498.24994233152492471699719170469997876803","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11546127923715190914128277599849591183","series":"1.2.826.0.1.3680043.8.498.79348145937688099854927598677633897221","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11548839502674800051973767797346682106","series":"1.2.826.0.1.3680043.8.498.28330099531306802330024106027906915266","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840","series":"1.2.826.0.1.3680043.8.498.50613286207082214222722525784789239290","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11569355546217846610131708883985793793","series":"1.2.826.0.1.3680043.8.498.11117813791943887442385944811679271270","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11571057273590444914079281238002921868","series":"1.2.826.0.1.3680043.8.498.19061733740401589340572749195072157689","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11577630811395858629711985689258647804","series":"1.2.826.0.1.3680043.8.498.45840560047288531178825971853634299139","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11583098598455610459898849451117846552","series":"1.2.826.0.1.3680043.8.498.67786655409077435006084218306936839272","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11587989960888698769912899464470116193","series":"1.2.826.0.1.3680043.8.498.15153271126343505840811217172714577420","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11592838054859712596515534341207971997","series":"1.2.826.0.1.3680043.8.498.49300433637361899333825241291273056203","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11596665780617412984766822528862741584","series":"1.2.826.0.1.3680043.8.498.10639884401600325010872465044892259231","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11598606533748611505510113033734692457","series":"1.2.826.0.1.3680043.8.498.12364031433876658179598449149032650447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11604298807403466096392316117803661479","series":"1.2.826.0.1.3680043.8.498.47499797457270069091222834017904760807","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11610585947047502123824835405421129552","series":"1.2.826.0.1.3680043.8.498.10970277964295063071222770470057779937","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11619883329923252065839671636992770849","series":"1.2.826.0.1.3680043.8.498.12436874165753986225828177253673244436","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11641274367861998748475337224297076397","series":"1.2.826.0.1.3680043.8.498.12215965632419209253018952993249706634","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11642228346306175407288158335830376197","series":"1.2.826.0.1.3680043.8.498.99398409625332602121579540436010035600","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11646465814751330102194657154003685957","series":"1.2.826.0.1.3680043.8.498.11813444535260394738083629164638342706","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11652570369213315569275956969935617843","series":"1.2.826.0.1.3680043.8.498.53239288414909246397196090599656070981","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11659861228342609979638095779959188093","series":"1.2.826.0.1.3680043.8.498.11673115964094616981195876864329153496","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11669120316759278775792190599036288128","series":"1.2.826.0.1.3680043.8.498.11779081258913722248829282962083655362","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11679044101697033975478370934058085847","series":"1.2.826.0.1.3680043.8.498.91835485445936746474384517063389043501","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11685904538570360040472064238371379681","series":"1.2.826.0.1.3680043.8.498.29285171537443598250674656421974291682","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11691245281415120685024733512803575288","series":"1.2.826.0.1.3680043.8.498.10113905717355398248066166787033506891","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11703195077261884009442719578918319962","series":"1.2.826.0.1.3680043.8.498.48249390839637582581399193723118922327","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11716605112302071535035323666308656381","series":"1.2.826.0.1.3680043.8.498.69790702451836499596995339154892122839","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11720535027022778971695130536590302635","series":"1.2.826.0.1.3680043.8.498.12009385404157810395052920094335138751","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11722185671504770687850569082348083612","series":"1.2.826.0.1.3680043.8.498.35140784814003809251998205412202314234","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11723945299462627737346907048285143063","series":"1.2.826.0.1.3680043.8.498.39794942415832934927320150633900484674","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11738872955471637985947490962985266753","series":"1.2.826.0.1.3680043.8.498.26287728381094133720832434033501259444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11753067309373104365558887398441364695","series":"1.2.826.0.1.3680043.8.498.68556704078503881943701255553130842225","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11770690655120964721059321709525979950","series":"1.2.826.0.1.3680043.8.498.92548045540265493493728463985272230666","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11776647434143676427722616249587614803","series":"1.2.826.0.1.3680043.8.498.82220404992393429349124950924483669149","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11783597160745641720078978406968189488","series":"1.2.826.0.1.3680043.8.498.51205293320304541716324027135430691247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11786582806577871618290342375465595269","series":"1.2.826.0.1.3680043.8.498.24721329136275451740013498668431826553","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11791567830371683660578052737702299124","series":"1.2.826.0.1.3680043.8.498.56995528357123612499606782528791798876","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11798183157109435753679723596499976762","series":"1.2.826.0.1.3680043.8.498.37059536119189433268020615776630709383","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11803762390999881908101130422708589537","series":"1.2.826.0.1.3680043.8.498.27213480613400664391015941426892754808","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11805131119810457174456565128522037991","series":"1.2.826.0.1.3680043.8.498.12470951213358719797605708255629896161","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11812607848520072826145426247813670653","series":"1.2.826.0.1.3680043.8.498.71069389332890648418954738424823877658","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11830915612184936576804778995164042200","series":"1.2.826.0.1.3680043.8.498.25103657622005289146529443280273050216","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11851285389689341495961233129719325417","series":"1.2.826.0.1.3680043.8.498.34953035672207213080853502978756070396","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11860731961337988611847638299761007498","series":"1.2.826.0.1.3680043.8.498.10896906005904497807375713481501894384","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11868154141292577776796381772987549697","series":"1.2.826.0.1.3680043.8.498.12587906820354793918744585217334954300","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11871142388984150456249558619289234011","series":"1.2.826.0.1.3680043.8.498.12637357087017427078555292164142753511","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11880385031047315524154761508473409762","series":"1.2.826.0.1.3680043.8.498.34484325801268820618795461261029216978","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11892908237449514945544380023820626319","series":"1.2.826.0.1.3680043.8.498.95517996854410374345447038244893855844","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11900355716044995266104285198121445744","series":"1.2.826.0.1.3680043.8.498.11694107495552930251406682182652982806","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11904156585987383639134967886361403369","series":"1.2.826.0.1.3680043.8.498.57160647868950580578198540266472160288","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11909779134152823550896593167929611759","series":"1.2.826.0.1.3680043.8.498.12717597798938193923048756018793257642","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11912618147674655465071922267060032851","series":"1.2.826.0.1.3680043.8.498.26546507790700727073127070578028218107","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581","series":"1.2.826.0.1.3680043.8.498.71284507741504130254010428167357512908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11920531053482540304184657218030242597","series":"1.2.826.0.1.3680043.8.498.39184697664509095524899574617515587968","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11926412946593048423086380882538180028","series":"1.2.826.0.1.3680043.8.498.92329289362168741989208703051662276509","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11934483804567594363179973225449686890","series":"1.2.826.0.1.3680043.8.498.93351769926829755139011186447426321596","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11937911933281010146897685679862930140","series":"1.2.826.0.1.3680043.8.498.27541778624369201952655638682100335357","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11944762815616453054528091721553484857","series":"1.2.826.0.1.3680043.8.498.33455730924801165535054190674478537259","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11957476321850340968244508976719889277","series":"1.2.826.0.1.3680043.8.498.54365352876095966171558784779554861056","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11964919539297578836040550782883323317","series":"1.2.826.0.1.3680043.8.498.87131673949236744357645621445387860532","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11979034558288454336812613626594510186","series":"1.2.826.0.1.3680043.8.498.11679027428076711523980385269728717512","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11980805687949950504930975019161587970","series":"1.2.826.0.1.3680043.8.498.98998064375679835891417951417719491596","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11991512352474459753052051071717622771","series":"1.2.826.0.1.3680043.8.498.99185669013745770421639253295675573108","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11999008448480965933785988718780864948","series":"1.2.826.0.1.3680043.8.498.86894744921414373182530417479118467950","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12006027138204983547873134458107233878","series":"1.2.826.0.1.3680043.8.498.71612316952780910425989000738816231009","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12010150865087936209475730225377000084","series":"1.2.826.0.1.3680043.8.498.10530455000474602584349591193709652863","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12015406188790105426257083591807220760","series":"1.2.826.0.1.3680043.8.498.54892982353539896301928105198437194101","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12034579945815402722591687609951730205","series":"1.2.826.0.1.3680043.8.498.12482607885143844244130943561085157178","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12041392088066705782960819598258792338","series":"1.2.826.0.1.3680043.8.498.91210841654570928178263041931092989708","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12045300645580015230470991493790324857","series":"1.2.826.0.1.3680043.8.498.16405010790840929644412572845533986393","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12054099347723802635102026371218576198","series":"1.2.826.0.1.3680043.8.498.28741009779146411817875168190881489063","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12064822036518837577612019712902340953","series":"1.2.826.0.1.3680043.8.498.99077228020457212663308522930032277692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12071998470084955703027522152040042364","series":"1.2.826.0.1.3680043.8.498.10205231417386112076783951928646565465","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12092744899746753313905432642371563786","series":"1.2.826.0.1.3680043.8.498.75004159226612438577724181459551206643","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12100254887364913022980951672127944515","series":"1.2.826.0.1.3680043.8.498.30253901671420957760016451229977123909","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12107680313133383587522762626929218051","series":"1.2.826.0.1.3680043.8.498.10636467190471602958350165476426474085","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12117377043894182951345966247899775612","series":"1.2.826.0.1.3680043.8.498.11920821684110085760040568662591852175","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12131866860554871145836544571379633595","series":"1.2.826.0.1.3680043.8.498.24105040540365292143712294544807756830","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12137739162059836759686062882514824614","series":"1.2.826.0.1.3680043.8.498.75236163172957717671276445264007423098","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12150502202634860887314001818431257267","series":"1.2.826.0.1.3680043.8.498.64599829814630877667915788027059760621","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12154188739734505793699974626991872621","series":"1.2.826.0.1.3680043.8.498.12813703960078945494493530082123341822","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12165893434332227221101266719291133169","series":"1.2.826.0.1.3680043.8.498.54084173905869134657293347020637606278","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12166235678892893545054546864889743251","series":"1.2.826.0.1.3680043.8.498.27453680718018620006971763224687427405","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12170753305862973517190756418865414984","series":"1.2.826.0.1.3680043.8.498.12710383538607339848876475758619547457","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12176466551281285838907773521868954581","series":"1.2.826.0.1.3680043.8.498.11440608443353712209397204914021665670","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12178921459424917661210327999672203211","series":"1.2.826.0.1.3680043.8.498.32720389905266006458714955048441139359","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12182309401920763498667366352681074302","series":"1.2.826.0.1.3680043.8.498.41859849503409748720896479303975666343","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12189863023888657759738596583562992408","series":"1.2.826.0.1.3680043.8.498.71756358449856403786654393784612932808","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12195090472992304712555822541659656360","series":"1.2.826.0.1.3680043.8.498.12217101994136498325795912273079861517","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12201598444731259697380878112783653796","series":"1.2.826.0.1.3680043.8.498.23421249629198798688959731442507154873","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12208986088767216690584621773389997968","series":"1.2.826.0.1.3680043.8.498.37255696679228054374930933633924446109","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12219658385745696203053079500246399481","series":"1.2.826.0.1.3680043.8.498.10915953378406788400444909605988234651","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12229628534316014336864328244217875465","series":"1.2.826.0.1.3680043.8.498.10105817235318405243669971031536329573","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12238753439953831519247789198174756615","series":"1.2.826.0.1.3680043.8.498.13107997959048040890528174594075361964","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12241413500128553133597113709525575494","series":"1.2.826.0.1.3680043.8.498.10336394951722636029707214007144773842","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12243764506921404196245986681613075136","series":"1.2.826.0.1.3680043.8.498.99985587288484035337375475542090751233","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12248983667596865960950671947508678392","series":"1.2.826.0.1.3680043.8.498.29579241097479460293092641745596479398","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12257119852150088212071954292747252243","series":"1.2.826.0.1.3680043.8.498.92915767184754446474397678142937734669","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12259896940707176389739553110492954399","series":"1.2.826.0.1.3680043.8.498.58758735759979515645166498422291012198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12264607993101717201407955391437731134","series":"1.2.826.0.1.3680043.8.498.56797125809501797997649624361477482211","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12266644932420958618394322959047940542","series":"1.2.826.0.1.3680043.8.498.15557662215814852313451357872646158661","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12271616365050313084031931009756086606","series":"1.2.826.0.1.3680043.8.498.10706420672981134613384474363591515684","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12273490180961458167930039242826008249","series":"1.2.826.0.1.3680043.8.498.31090520506231220065996076165531278625","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12280407492991430179266369843997653205","series":"1.2.826.0.1.3680043.8.498.83201257370138002029712999418154231298","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12285177271228289867211553575739205132","series":"1.2.826.0.1.3680043.8.498.26843705691144844596009192792253010661","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12286894057773964720268768529280193085","series":"1.2.826.0.1.3680043.8.498.12133369071833480601780123125563774301","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12295284437770380354725533247473017729","series":"1.2.826.0.1.3680043.8.498.39614418745356717231605133083393759599","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12300749089664733313051543117302244508","series":"1.2.826.0.1.3680043.8.498.14131238577320061897163600732116101228","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12303161232396569915032951761728884199","series":"1.2.826.0.1.3680043.8.498.29675617533071591444899906252391034222","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12307356432736970350794033171174403632","series":"1.2.826.0.1.3680043.8.498.25574137699429210804171749220745056378","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12314729913655626716039376606973474150","series":"1.2.826.0.1.3680043.8.498.25521359311635144562628899143377322773","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12328313245396930020974053819810033945","series":"1.2.826.0.1.3680043.8.498.39614718012624842553957788720145561913","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12333996498811456012744466468462396081","series":"1.2.826.0.1.3680043.8.498.98530197935627119211464312365440645027","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12345064031074704673835101507426902246","series":"1.2.826.0.1.3680043.8.498.12110383942603454634067187554198717371","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12348875283993064999589090039392140219","series":"1.2.826.0.1.3680043.8.498.46631780348349379939561692320036051549","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12349656752404066719917550972333542508","series":"1.2.826.0.1.3680043.8.498.12424893182534250163097466563424167895","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12355011301785011370279005103251815187","series":"1.2.826.0.1.3680043.8.498.69572127155379835976920233822006197141","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12365033663152692616611548717152384269","series":"1.2.826.0.1.3680043.8.498.64937545825526949492710418853440834252","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12373923338033469765189220784089350380","series":"1.2.826.0.1.3680043.8.498.58497372860271767633229487450164612512","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12379570725733005682132316461218878636","series":"1.2.826.0.1.3680043.8.498.85016185009008314878752308852529387450","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12381909868368799280462187344958674397","series":"1.2.826.0.1.3680043.8.498.93872401576710190986850041804439254721","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12386255059403233724886571725482313876","series":"1.2.826.0.1.3680043.8.498.13190848404463831037321369683329807672","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12387667155732116394489547491673280062","series":"1.2.826.0.1.3680043.8.498.82856471456735036799815914624119045257","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12401147012438969595202076588471168894","series":"1.2.826.0.1.3680043.8.498.10420678249796622582611365076781704534","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12408186961945791320086301369036829266","series":"1.2.826.0.1.3680043.8.498.52218847211899176691436330101998401555","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408945963789246646229147558387793270","series":"1.2.826.0.1.3680043.8.498.11610660430091574091305195786377167964","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12414869188691502885668778427120766321","series":"1.2.826.0.1.3680043.8.498.66190164467233565212885052399485462744","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12431272351851438889289190308550697909","series":"1.2.826.0.1.3680043.8.498.11707477725285854260104992917545210062","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12437154858167700438432674983971817217","series":"1.2.826.0.1.3680043.8.498.57642104824898031654709138419453341073","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945","series":"1.2.826.0.1.3680043.8.498.35003530834353727317166773901402690335","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12462498156569316578044317782899399656","series":"1.2.826.0.1.3680043.8.498.12793996618137267590049898391196565674","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12473533314363654795650919604551685896","series":"1.2.826.0.1.3680043.8.498.90998095476199715062524087547088636636","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12481913908649439189738319460342358360","series":"1.2.826.0.1.3680043.8.498.11587596693855513566699314135653151395","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12484700311993345120015691857089576898","series":"1.2.826.0.1.3680043.8.498.75940149982100115964289949729201591790","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12490034912643658601552092574630438642","series":"1.2.826.0.1.3680043.8.498.10735278525183587980413558885940199546","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12491747080684524905643593142141672951","series":"1.2.826.0.1.3680043.8.498.33341605923736659902359896812010146521","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12492431885298164225672926785397151525","series":"1.2.826.0.1.3680043.8.498.79595909109590807784330551039358115935","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12499208731138180053966892599139021038","series":"1.2.826.0.1.3680043.8.498.30371065324226976766512654032526830756","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12507957850103916926719601413279798696","series":"1.2.826.0.1.3680043.8.498.46646798262451287384747730914877208400","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12509944331769453135724043962410129357","series":"1.2.826.0.1.3680043.8.498.11996921362326487036723050314090553721","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12515322634403509077334846234426636136","series":"1.2.826.0.1.3680043.8.498.23686424991816550812860886301522498230","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12522620957277632939731546099134770342","series":"1.2.826.0.1.3680043.8.498.47960807605031985558512250557592275666","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12525752631743097782762806859112218620","series":"1.2.826.0.1.3680043.8.498.55488397086375850552533323684561870317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12531442175319757740745664049001806494","series":"1.2.826.0.1.3680043.8.498.18693421653173921663244447695960436510","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12534628800787651008876070514080610643","series":"1.2.826.0.1.3680043.8.498.33302786553882880641871317414131924920","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12548176380550230498070609559486512004","series":"1.2.826.0.1.3680043.8.498.19935676892610530712897117806150811121","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12553455350653672209409408595948190982","series":"1.2.826.0.1.3680043.8.498.26377885495328024912711340654735242016","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12559687473583953266143897736224215680","series":"1.2.826.0.1.3680043.8.498.27537728218629571523101260501810174177","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12569762036474456642024618586300354737","series":"1.2.826.0.1.3680043.8.498.90518704625447889148779670379345419217","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12573518342767030025851169814440863785","series":"1.2.826.0.1.3680043.8.498.60799201373214011013882145221313698139","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12576055921678075813718486704978381993","series":"1.2.826.0.1.3680043.8.498.82674395897986868671375564375095409997","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12580232188757225504172709417466515184","series":"1.2.826.0.1.3680043.8.498.60132801491926083582491700083257138231","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12606136253498337506682547306933254691","series":"1.2.826.0.1.3680043.8.498.57295782492759965190052958926881447387","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12607203615180268117341271176820357242","series":"1.2.826.0.1.3680043.8.498.95670931706266047331772172085105426379","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12607959449008110729566106776800628990","series":"1.2.826.0.1.3680043.8.498.25051043385554726329960468479103584806","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12619895582462244920124420363095282842","series":"1.2.826.0.1.3680043.8.498.13068960735263997742961432650984333625","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12625250919597111075072308824921101918","series":"1.2.826.0.1.3680043.8.498.74196124277568210323283203867551418706","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12630423349626505319792444053874011718","series":"1.2.826.0.1.3680043.8.498.49990488676008356643505434437507237044","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12632129640405661298623288977632682289","series":"1.2.826.0.1.3680043.8.498.10346030122694910196746254300805454660","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12640439197386154384526090653248054582","series":"1.2.826.0.1.3680043.8.498.29658523484253609458952570867059974903","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12644400073937544865441360809610886635","series":"1.2.826.0.1.3680043.8.498.73597374587725557347664342057503559815","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12655324218817524683138594937984186560","series":"1.2.826.0.1.3680043.8.498.39844418739615374856554968986934886340","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12660901572462562819575002681831361427","series":"1.2.826.0.1.3680043.8.498.89785896332575722355167901966132239653","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12674069717230126880288353271062347095","series":"1.2.826.0.1.3680043.8.498.45749292277574526757217552816194084557","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12676359576477452816896379524598090410","series":"1.2.826.0.1.3680043.8.498.74351597029797331207391259034086161633","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12690080124553846006638565600585469969","series":"1.2.826.0.1.3680043.8.498.41388534115296500695876891605672393845","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12692579161952587215276483587686562922","series":"1.2.826.0.1.3680043.8.498.20833718308044491115234940631267106835","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12700591955254168977239658917260628326","series":"1.2.826.0.1.3680043.8.498.63868482507169627116242651476363322519","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12704063145595847725421427571860589471","series":"1.2.826.0.1.3680043.8.498.12592652626460245987266493650004547090","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12715067947657150279232238280712066972","series":"1.2.826.0.1.3680043.8.498.95498040600327761359769867228162340150","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12717728703860353055063551162640833849","series":"1.2.826.0.1.3680043.8.498.63296028814368413917049125567156646664","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12719397512102459853075227679498459688","series":"1.2.826.0.1.3680043.8.498.11478205669820518412378116290859538760","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12725832336445568031849006413563892122","series":"1.2.826.0.1.3680043.8.498.50408416797312711032922922701093829164","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12728397615513347615111391662798012012","series":"1.2.826.0.1.3680043.8.498.46272492336412075361390422976764356518","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12754381146006839341912662615844567067","series":"1.2.826.0.1.3680043.8.498.62957845043894641655527446667033449578","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12761906443846433581503942530016459978","series":"1.2.826.0.1.3680043.8.498.13152195199457442085676820499797358337","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12765434812409174436720333864278447770","series":"1.2.826.0.1.3680043.8.498.91986112088220302734645950061293788384","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12771448321084229688831202344007889535","series":"1.2.826.0.1.3680043.8.498.45410919537579536942050220771610387031","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12775105605170844918620305197829216665","series":"1.2.826.0.1.3680043.8.498.46736650111491199405491732454419787960","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12785377697188564760194569072434828180","series":"1.2.826.0.1.3680043.8.498.81174581927771813577273859687156634576","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12785792890626338008419452097492192088","series":"1.2.826.0.1.3680043.8.498.12078737169515634929866484075781383568","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12798419553759200632295966180228523212","series":"1.2.826.0.1.3680043.8.498.11920092989239260957894386374535976453","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12800130397185143653180509040746675358","series":"1.2.826.0.1.3680043.8.498.11887297483714221287034895637968479422","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12801643268939133682891906705186571010","series":"1.2.826.0.1.3680043.8.498.38948437851580840845426108750128759451","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12814598731776085957663058326979908652","series":"1.2.826.0.1.3680043.8.498.88098646143891598014463787734481645188","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12817753008281742621758605755308036035","series":"1.2.826.0.1.3680043.8.498.79894314464418892431286722016372638972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12820051821104182459960036809430047569","series":"1.2.826.0.1.3680043.8.498.11823054595914389868960614605280295274","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12827120140904847104302376671076396695","series":"1.2.826.0.1.3680043.8.498.13351277218965625451721274331249587285","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12836702174475748025129787471026627810","series":"1.2.826.0.1.3680043.8.498.73159778967753527497341108541322386887","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12862782592303338109631225504988756108","series":"1.2.826.0.1.3680043.8.498.45865524702454965262690369732544918324","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12866937766198610064474201436305532400","series":"1.2.826.0.1.3680043.8.498.61765595978853136883068225840230704610","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12899995703999073968090810947564075608","series":"1.2.826.0.1.3680043.8.498.10050691989051417005090930183249549482","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12907457183137517669505067507754394039","series":"1.2.826.0.1.3680043.8.498.66844494028159914271256894733831386418","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12917504242141830416624247604384969664","series":"1.2.826.0.1.3680043.8.498.20783794220821074707237598157365054433","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12929915324982999692857962099866616265","series":"1.2.826.0.1.3680043.8.498.99794601079093793611579856041752546448","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12938500386086171684983091332136233036","series":"1.2.826.0.1.3680043.8.498.85605562052548645386522379927569199196","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12946789387115203718210904958275579544","series":"1.2.826.0.1.3680043.8.498.11385306681695165334034338134897238182","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12950639892345812884896649629409199101","series":"1.2.826.0.1.3680043.8.498.10911724933510725379203222416351482348","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12957752106697316015996777476239762111","series":"1.2.826.0.1.3680043.8.498.13347136996152572721642604572951615869","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12961116776122838195555132309258963046","series":"1.2.826.0.1.3680043.8.498.12926609058164804268497734713685153963","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12970315889871740202726744767179863179","series":"1.2.826.0.1.3680043.8.498.48632673060922085232888773153490096166","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803","series":"1.2.826.0.1.3680043.8.498.10259924209097075418304938918872474391","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12981413356524881858859912294078498486","series":"1.2.826.0.1.3680043.8.498.10677991600335293242218149077223298651","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12986028147682493711912242081052787346","series":"1.2.826.0.1.3680043.8.498.65060912892270216266719089209688172496","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12987428318535758055829100395092244687","series":"1.2.826.0.1.3680043.8.498.10719311413560785992590737290801120809","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13004048880335015585285100185910148808","series":"1.2.826.0.1.3680043.8.498.63087830739296630944274215418136223826","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13015502057686077045104167339480215528","series":"1.2.826.0.1.3680043.8.498.33862527046653526931396281608542449712","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13022266764623819477228425331705640702","series":"1.2.826.0.1.3680043.8.498.18065638657380268373829120268568856082","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13024250150960117282590397343184827466","series":"1.2.826.0.1.3680043.8.498.19324553511060818126216345662852719923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13026463185033527579856819607371375594","series":"1.2.826.0.1.3680043.8.498.72809864103903753394506293846002233056","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13027502247317650159940595684214701589","series":"1.2.826.0.1.3680043.8.498.12977093957566433052278562704608150577","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13034169434311535671044673510644711229","series":"1.2.826.0.1.3680043.8.498.26406505343485311811665060502150399288","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13042847643086819857036458569632034999","series":"1.2.826.0.1.3680043.8.498.57020178429856025451416068617066550431","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13062978180647549151946243290102173508","series":"1.2.826.0.1.3680043.8.498.11272757183639466239002585268951932080","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13068696303639759696458130704554709416","series":"1.2.826.0.1.3680043.8.498.30575049566215406867923012056226315801","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13073387272228394257811861905056479438","series":"1.2.826.0.1.3680043.8.498.11072522293527069439770688811820020741","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13078009378379325641560321462188164769","series":"1.2.826.0.1.3680043.8.498.73757298296230701186112457142479849650","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13079170182386765602129976624023222742","series":"1.2.826.0.1.3680043.8.498.51076221663397395375527319267723312243","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13087770868099843948960419968576882974","series":"1.2.826.0.1.3680043.8.498.34639861742039070892051229979926670784","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13092094949394905924922534008771583745","series":"1.2.826.0.1.3680043.8.498.54167704298114634010132650292508166475","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13096288495834275184930201615129174987","series":"1.2.826.0.1.3680043.8.498.29951782641847884555638667709652692595","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13102058401501684064017656674827527261","series":"1.2.826.0.1.3680043.8.498.12186172648842188822439642474804030051","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13113623966675790946816351311283701141","series":"1.2.826.0.1.3680043.8.498.49220585711938301157012714568359303257","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13114693814781737495494700902699619138","series":"1.2.826.0.1.3680043.8.498.91754552986738371359878537098765503346","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13118532243906060645114391749986421707","series":"1.2.826.0.1.3680043.8.498.26015462618859268745036649458589537775","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13126070481622692281046898445918832841","series":"1.2.826.0.1.3680043.8.498.93434320197687159733920561893282902130","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13128506994126383651503283021834372142","series":"1.2.826.0.1.3680043.8.498.25871201840147096680021158660366724521","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13130534736322715018024786213131944607","series":"1.2.826.0.1.3680043.8.498.10477330769927962496549556727741454978","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13133755791254469039849289186598412660","series":"1.2.826.0.1.3680043.8.498.11762443464520253324007966364684016351","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13136367866260644484250529322937554927","series":"1.2.826.0.1.3680043.8.498.86006021082632021053245639916099996960","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13137510961466386836387239510287919268","series":"1.2.826.0.1.3680043.8.498.47928757967785952235003776534844723699","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13143446880514535170343243531129864116","series":"1.2.826.0.1.3680043.8.498.84353279974316429119992159640494723672","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13147249459968468500804358644901642270","series":"1.2.826.0.1.3680043.8.498.27402348357955636493759174879530526976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13150794288945922409613136250303663502","series":"1.2.826.0.1.3680043.8.498.43806823961415152595597308690789448110","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13152690996267297597178587721720788387","series":"1.2.826.0.1.3680043.8.498.11030437094490549605690633300363200147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13158227087287104386790063637592969032","series":"1.2.826.0.1.3680043.8.498.22224322623098379179449294824933045860","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13162284470388627411596358366248494046","series":"1.2.826.0.1.3680043.8.498.38370237718563398520922427664639398704","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13163727085668231782866823012535268245","series":"1.2.826.0.1.3680043.8.498.71761806120847368021682684017862086841","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13172295193446466338347273836250944090","series":"1.2.826.0.1.3680043.8.498.67460396340738993628960518767605082210","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13173781757294564816839795307893228067","series":"1.2.826.0.1.3680043.8.498.13342305582597349550306755106645372295","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13179169600887013946657470830682778717","series":"1.2.826.0.1.3680043.8.498.12832668589575052468039888415165990574","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13199635020145400856987271814458645896","series":"1.2.826.0.1.3680043.8.498.69148246354070396396104408010728577660","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13207489724395276840990928372610055495","series":"1.2.826.0.1.3680043.8.498.86085023932489594356999903349351513446","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13211838518836535713919214396985811434","series":"1.2.826.0.1.3680043.8.498.74138381258344375258100239434788111310","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13214267416673058581197714011913363750","series":"1.2.826.0.1.3680043.8.498.55156073617146166116970681103640987908","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13226961394403558608712498872213808226","series":"1.2.826.0.1.3680043.8.498.80063878017475418334154255914427088030","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13232501202175905676335982676386953813","series":"1.2.826.0.1.3680043.8.498.12007154498049981923839921358511354010","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13240749152416621711688053465035013133","series":"1.2.826.0.1.3680043.8.498.68342443354064215536334021674641547875","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13246496699722524712272293738359002501","series":"1.2.826.0.1.3680043.8.498.28631485569552384681701721967275113293","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13262053361139017457941697384315610025","series":"1.2.826.0.1.3680043.8.498.12845572445033071726232491810642154022","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13262583253812134425581922368942846684","series":"1.2.826.0.1.3680043.8.498.57281729450916764067088924157682612971","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13267708980958529833316446819156908381","series":"1.2.826.0.1.3680043.8.498.41800231954367010670800444196246828267","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13273890409523605456488117581706115075","series":"1.2.826.0.1.3680043.8.498.10485776130942455433348897737615079963","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13284123714012951608998434802430211634","series":"1.2.826.0.1.3680043.8.498.94438864552446188486298870749561623004","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13288244567806030859347115222642155186","series":"1.2.826.0.1.3680043.8.498.18947691870703702715931272880892647669","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13291943370279119939499960846843879156","series":"1.2.826.0.1.3680043.8.498.92402087512705800298758705314173685647","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13297973188397662905314777604089528518","series":"1.2.826.0.1.3680043.8.498.70588147459360239526337693607246213267","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13312139694018789912028129989483016875","series":"1.2.826.0.1.3680043.8.498.11815694482487553001949474160944631615","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13321746459276191365724686301267421531","series":"1.2.826.0.1.3680043.8.498.12555622920124153625250106672331215980","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13328029743559327467301971917273397645","series":"1.2.826.0.1.3680043.8.498.63434831545249270920323125609932341891","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903","series":"1.2.826.0.1.3680043.8.498.96243991448267206821172339346422614701","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13341507176140340924941654608476073030","series":"1.2.826.0.1.3680043.8.498.51785847967322486703382929281312685159","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13353966209257569249620562863006006441","series":"1.2.826.0.1.3680043.8.498.93667436368515047325093283512907095904","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13358702230966891064267033765095124604","series":"1.2.826.0.1.3680043.8.498.58828126818782712635212464700777754875","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13363856251607349805111249441972460575","series":"1.2.826.0.1.3680043.8.498.61718286142737796659784907062776621983","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13367740087756379568826046059252678700","series":"1.2.826.0.1.3680043.8.498.30714271932798221997116625515992741439","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13371001997511542011416510364359588752","series":"1.2.826.0.1.3680043.8.498.46346960922617925557016803973257914818","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13380735447520796902327272696579348141","series":"1.2.826.0.1.3680043.8.498.55100700320946793207658200124466943944","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13391735462825209934041441767886593395","series":"1.2.826.0.1.3680043.8.498.10816123132131258755825325172416990464","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13399254446600380420635633478666850540","series":"1.2.826.0.1.3680043.8.498.52122119435277709075106821102410953522","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13405120554981098042415743645998258385","series":"1.2.826.0.1.3680043.8.498.46387973383400859113454385778279386915","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13455212888114594010716499984611216646","series":"1.2.826.0.1.3680043.8.498.71813330087786309399874373993184667855","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13460078631594032326006554321483799928","series":"1.2.826.0.1.3680043.8.498.10218141502823588443302657361190660596","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13463672277039459152171414002273376185","series":"1.2.826.0.1.3680043.8.498.89185873947480693202976022916858633175","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13575953449958751965232580597585303226","series":"1.2.826.0.1.3680043.8.498.12505387327303029720933331416999857712","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13616982533395291590655841043722813973","series":"1.2.826.0.1.3680043.8.498.11777157303005168483947648715317310210","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13737588520456064856233576393747389054","series":"1.2.826.0.1.3680043.8.498.76641180234093995861290716256405201378","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13829111847660458141056077151781118782","series":"1.2.826.0.1.3680043.8.498.25383265326186218979354058367514502303","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13915367834258386094001181500560177470","series":"1.2.826.0.1.3680043.8.498.20093052872219105135653739256134746748","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13975027824128345697756313869165838055","series":"1.2.826.0.1.3680043.8.498.12173425444226608580524508253175995216","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14032903253348079919799627243690484791","series":"1.2.826.0.1.3680043.8.498.97203556897154121487050140229763360139","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14117402902169664329719927863268363596","series":"1.2.826.0.1.3680043.8.498.92022340056329196871395150777384331875","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14188540532926067083914263496492255704","series":"1.2.826.0.1.3680043.8.498.26165725286684835006529288399895636661","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14236519693915284584208989450687591759","series":"1.2.826.0.1.3680043.8.498.37594293396971959133898932491671609168","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14325424371811230497993759959862738838","series":"1.2.826.0.1.3680043.8.498.60604924959383649099174880950462913602","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14377917653321813299237274871837695586","series":"1.2.826.0.1.3680043.8.498.75735256442094425540185771451704878848","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14472261871417480346162193127520918127","series":"1.2.826.0.1.3680043.8.498.10787918835808776047864163978831942378","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14479913217228604289584726096114526268","series":"1.2.826.0.1.3680043.8.498.61985151738410034547318763908318691314","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14495028403937132710814780922642512791","series":"1.2.826.0.1.3680043.8.498.56416928425843114538681629780343009591","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14679140388328648617386311272361396555","series":"1.2.826.0.1.3680043.8.498.11232121193216984303507359857101253221","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14770617961376953596780345652892307570","series":"1.2.826.0.1.3680043.8.498.97014020963265789671756736398385428643","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14806659119591146961784818305031900863","series":"1.2.826.0.1.3680043.8.498.26475201645582023239060565361381437664","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14958560978176121375035839707004667832","series":"1.2.826.0.1.3680043.8.498.12745481127900097143331668576381566974","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14959732862523218997149041503831001633","series":"1.2.826.0.1.3680043.8.498.10043334987354567111812324187317075137","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15025957400098336605043184930980421418","series":"1.2.826.0.1.3680043.8.498.10274525526979095809579714083485380730","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.15090991392324602559982474708414427621","series":"1.2.826.0.1.3680043.8.498.87446640076048533838687516880056283555","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15133818248365103660456294775831248835","series":"1.2.826.0.1.3680043.8.498.11956737332045131535346761143920883710","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15297444019822621504849851092382560980","series":"1.2.826.0.1.3680043.8.498.13374804614025648979830654979480187101","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15360150088598749206650025637914821738","series":"1.2.826.0.1.3680043.8.498.12098919197124193465801528240435546491","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15399233309269178192583032682669258885","series":"1.2.826.0.1.3680043.8.498.32090945954276474544358549546238142054","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15444316810805746453392291780593934328","series":"1.2.826.0.1.3680043.8.498.80319866330175800687139619195657049372","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15522600887743817464976568872439594397","series":"1.2.826.0.1.3680043.8.498.70237637953106689720205623668663069880","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309","series":"1.2.826.0.1.3680043.8.498.10799479496416725407987323179903258953","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15599161312748548709240395406977042093","series":"1.2.826.0.1.3680043.8.498.12763577002809043465286302208785360429","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15744573360087157139672078937679122840","series":"1.2.826.0.1.3680043.8.498.10274959727429382362338613778199794539","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15863325224755490561125835019647303556","series":"1.2.826.0.1.3680043.8.498.11444028267275645733208144447071593915","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16051798246004436152597344963060310360","series":"1.2.826.0.1.3680043.8.498.76853477556290826152278184679523288181","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16158812105029662172517682642229718094","series":"1.2.826.0.1.3680043.8.498.10699343372251311120926784632328947766","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16189833935983604051983090678102023832","series":"1.2.826.0.1.3680043.8.498.34915825240271993888031987711670161589","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16225091905018667993201414191711776943","series":"1.2.826.0.1.3680043.8.498.75278479092582319503064396031082719193","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16338296537380784459604897543069625390","series":"1.2.826.0.1.3680043.8.498.13034918818738639373300711871660196976","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16370303687278282143772106415664171358","series":"1.2.826.0.1.3680043.8.498.37580213447774736059976716750365000149","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16483470188406187000599644150600279596","series":"1.2.826.0.1.3680043.8.498.32717063457742403860326898610544301721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16534078612126477228267765508485297367","series":"1.2.826.0.1.3680043.8.498.13108592396007647454605469511623650142","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16581426215518472805200227992550139190","series":"1.2.826.0.1.3680043.8.498.16028635993652686272204428091886530475","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16657553974040122719239506008948241297","series":"1.2.826.0.1.3680043.8.498.25429459589078213600300517572376407064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16691383858003633107641297162554868325","series":"1.2.826.0.1.3680043.8.498.11531453819896281474836527253003452352","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16718788321951483456195507236392554704","series":"1.2.826.0.1.3680043.8.498.53691276667943103178282327560973858717","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16812524557513419866531695739014681414","series":"1.2.826.0.1.3680043.8.498.21104732279889003025090239093837594250","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16845451648760595922758166878121325861","series":"1.2.826.0.1.3680043.8.498.18696758505951714399637544337953742961","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16864649493021092944977766326505053153","series":"1.2.826.0.1.3680043.8.498.94160070623200539636420206772509316048","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17056997047488992455301411194938263817","series":"1.2.826.0.1.3680043.8.498.16520284988997479548559658970096517755","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17074746747536999918446136673966046653","series":"1.2.826.0.1.3680043.8.498.95230094684626752362274439647459907921","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17093025979426350838724553020541643719","series":"1.2.826.0.1.3680043.8.498.61836003381059395166907159610175855902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17146868851921828837871671912815843012","series":"1.2.826.0.1.3680043.8.498.11410237385070458262265431295039584639","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17241550277273207658303011467222594165","series":"1.2.826.0.1.3680043.8.498.53686120654406537336016944388907154138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17336223563862025807066700226971845206","series":"1.2.826.0.1.3680043.8.498.13318303750418016776038742575361219353","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17415936504684075438800895036740822145","series":"1.2.826.0.1.3680043.8.498.12815519309382967222114002806673992161","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17517950899674150719043097786728121608","series":"1.2.826.0.1.3680043.8.498.51964761684309178143134372850968161774","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17618734998753079190687719289815512319","series":"1.2.826.0.1.3680043.8.498.33931831252258535018600584911751668661","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17663005209256897892831319144686413853","series":"1.2.826.0.1.3680043.8.498.57847201874433701438006051707515630955","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17741907690913585163443565102293192244","series":"1.2.826.0.1.3680043.8.498.12953148585222632459882066976644636129","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17813878802252660525798053577039470087","series":"1.2.826.0.1.3680043.8.498.10036917905605875091613000644802338374","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711","series":"1.2.826.0.1.3680043.8.498.12647596472796287171744368208436290003","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17909590711893200514107254988453660857","series":"1.2.826.0.1.3680043.8.498.11728000183697805624297217875330929721","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17967978373588651520812399064902037467","series":"1.2.826.0.1.3680043.8.498.24570916710215008507038924693385711643","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18090147030578547920423840630147682826","series":"1.2.826.0.1.3680043.8.498.72962217867925374207888009689154612768","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18113123512696698441392990570166201548","series":"1.2.826.0.1.3680043.8.498.39790740478557049063210261109078803465","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18176586296602067352845573876750324761","series":"1.2.826.0.1.3680043.8.498.79194210985227202034577851019283308599","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18230847745299902998425838090653825745","series":"1.2.826.0.1.3680043.8.498.69423983144025250407627556775503272657","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18344563362014614046776037921812502569","series":"1.2.826.0.1.3680043.8.498.41815374979401517806123190969815775193","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18389676218363070462526838642303976332","series":"1.2.826.0.1.3680043.8.498.11534431000967639894324482743361512739","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18509793799833391995987656714798904411","series":"1.2.826.0.1.3680043.8.498.58248989247307318553924659820442012150","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.18529656610752887218162968403101035390","series":"1.2.826.0.1.3680043.8.498.10544235781347869990881403249137838640","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18625377919309205857120774497713011583","series":"1.2.826.0.1.3680043.8.498.11318656946120748277670218044331499067","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18671006890719350655167441375718348829","series":"1.2.826.0.1.3680043.8.498.13000269371805384187421716147182481053","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18721466442792788108431096912844719622","series":"1.2.826.0.1.3680043.8.498.86637852948879579970823158120867488181","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18792568294433896490648198192877693850","series":"1.2.826.0.1.3680043.8.498.11223305827176241657521766630037455445","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18825393854436144614202928260751418496","series":"1.2.826.0.1.3680043.8.498.99809851101430125933114584574703679111","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18899625751876258520147203904985857313","series":"1.2.826.0.1.3680043.8.498.57682925718865163233531210248121220852","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18968967345606818331567480514437075646","series":"1.2.826.0.1.3680043.8.498.86767382328069826065343657236383367794","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19104350033985745058396432133828748480","series":"1.2.826.0.1.3680043.8.498.12678072395759703101005901580377891869","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19178214677492694135853596920291058397","series":"1.2.826.0.1.3680043.8.498.78462747657939716221645842696624533127","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19194090255958016414082788444327981898","series":"1.2.826.0.1.3680043.8.498.12940304391711345415329946242012199841","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19291375342005405073321673638971145777","series":"1.2.826.0.1.3680043.8.498.25822706776264559233980587028689051936","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19374331384306309057311171252852774841","series":"1.2.826.0.1.3680043.8.498.11607048687495379082336247912983463775","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19486520578867272329833751210055639515","series":"1.2.826.0.1.3680043.8.498.81356769118713153234408825900380927325","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19516545000256238042886469152307943182","series":"1.2.826.0.1.3680043.8.498.12317242709606733645472930250079594397","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19592989024994864431870593504844309885","series":"1.2.826.0.1.3680043.8.498.18699541509667894207956088423983247090","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19706775737718777290660344285046025310","series":"1.2.826.0.1.3680043.8.498.19231300463584094673446492217921445642","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19788612491138642031531719942303619648","series":"1.2.826.0.1.3680043.8.498.11125896809962409070177601440001450709","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19830663206351645385520268848711462246","series":"1.2.826.0.1.3680043.8.498.43160259495251585792045484359794724807","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19845711038895023426012115550237130530","series":"1.2.826.0.1.3680043.8.498.88578417860866309097862590677041794147","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19895710326297085463251295629948896948","series":"1.2.826.0.1.3680043.8.498.60583398903499764660237792668260389662","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19965044457637713843792494926652048462","series":"1.2.826.0.1.3680043.8.498.52685041885783549821075131500406040399","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20063306448698185569553477478719533390","series":"1.2.826.0.1.3680043.8.498.47424846350075902218865591937182679099","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20134168618929769962391688049013971830","series":"1.2.826.0.1.3680043.8.498.70004411465192148499865921586061458323","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20192024727432864451309151144591371229","series":"1.2.826.0.1.3680043.8.498.21697502783870008318553188655699751524","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20204907530808218314127264658715054041","series":"1.2.826.0.1.3680043.8.498.71582789112069255276248231278643755020","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20269296702024202621734017056347194499","series":"1.2.826.0.1.3680043.8.498.66139495952807582968552901549768557704","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20411068889491114730824265411248922690","series":"1.2.826.0.1.3680043.8.498.11383882730339271507168748916042522040","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20472525526345683475357475395677168372","series":"1.2.826.0.1.3680043.8.498.56823044477545550460862438499834577304","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20509336519532058090925280050885010886","series":"1.2.826.0.1.3680043.8.498.44764317581588028150942867633280611354","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20522330861615570894210641293344468905","series":"1.2.826.0.1.3680043.8.498.35891815386087851729579765833722570550","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20640858366001462094044810443654811785","series":"1.2.826.0.1.3680043.8.498.87921238405775406307404865890308535943","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20733151481921261415295204209910573639","series":"1.2.826.0.1.3680043.8.498.64479606020576687895052550313216886087","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20801986011906346655458394198951999413","series":"1.2.826.0.1.3680043.8.498.33580467286243249295830435016521033051","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20859518005209632163032213488405671537","series":"1.2.826.0.1.3680043.8.498.19736299254652800512021750149964811616","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20953536352252551462901259297974956430","series":"1.2.826.0.1.3680043.8.498.64365712748715542032006881150396437696","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21012635480218712816069264326295410954","series":"1.2.826.0.1.3680043.8.498.12210838826196346475090170966302604203","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21116046804798247223618072736212944615","series":"1.2.826.0.1.3680043.8.498.26285063146397067613429770109504572905","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21142867813782396036287673290033987859","series":"1.2.826.0.1.3680043.8.498.43049019037221033147829380352862563560","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21312254534068992054745674608909722454","series":"1.2.826.0.1.3680043.8.498.53869872182175003893870931991100215708","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21389336408905524316915532623385847283","series":"1.2.826.0.1.3680043.8.498.67091124691782552281462784500257160092","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21413285047211074156709106352750752275","series":"1.2.826.0.1.3680043.8.498.11652508826162764717405568023693331243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21520719119040558609068952147312604007","series":"1.2.826.0.1.3680043.8.498.58929011086584440455776344968492716525","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21584859831565112159503971937497384030","series":"1.2.826.0.1.3680043.8.498.79144608468098640010333608501206701082","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21606196343708804451390109412118036330","series":"1.2.826.0.1.3680043.8.498.87482286975726375520487764868739044452","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21726746202429506433532477857404854137","series":"1.2.826.0.1.3680043.8.498.63865284035729416667656873528603696832","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21798342483930597563659425189400765869","series":"1.2.826.0.1.3680043.8.498.44820694480342764715632352215502856831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21874562538764236819894456417694074673","series":"1.2.826.0.1.3680043.8.498.68814524570679968520673929316106451854","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21937000442536121262190458446148124910","series":"1.2.826.0.1.3680043.8.498.38410944257644235186573918242080114073","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22007253197523827371600763959266047295","series":"1.2.826.0.1.3680043.8.498.11065269653219452630554118662721401505","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22027914323057444746912555313058666950","series":"1.2.826.0.1.3680043.8.498.30712196133548053027245391371969093834","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22057596860528658818833903685827585805","series":"1.2.826.0.1.3680043.8.498.44940057266941549625663772742466711244","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22119720417177816110156744770171496965","series":"1.2.826.0.1.3680043.8.498.73770285204847709873243435883019482639","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.22135282054716841452811687331325271982","series":"1.2.826.0.1.3680043.8.498.88420143504477494756792414560109585449","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22263857526032448247006468310072414129","series":"1.2.826.0.1.3680043.8.498.86345551044990407570724848906515356176","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22357754074308826479280023393343731000","series":"1.2.826.0.1.3680043.8.498.61441779241315732056186435558042660858","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22398731807962276404889164827238996005","series":"1.2.826.0.1.3680043.8.498.59729564999043654421787407448114361130","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22426390571898968732573184721547490379","series":"1.2.826.0.1.3680043.8.498.54930411992346495626836557487791913698","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22480218016816209183384996714060953859","series":"1.2.826.0.1.3680043.8.498.22345907772229366812094546073558707653","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22594249996322566888700984133307337128","series":"1.2.826.0.1.3680043.8.498.72234563342736558759082483249304894088","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22596004572382994883130705870731193046","series":"1.2.826.0.1.3680043.8.498.62543423010562898734410287052184934618","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22656222254273662937226093008714426490","series":"1.2.826.0.1.3680043.8.498.67567018630968509433831799907180981041","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22803020644232321685575897661362804423","series":"1.2.826.0.1.3680043.8.498.35602184194145711651724817788167559149","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22886251629508830700774506697670139848","series":"1.2.826.0.1.3680043.8.498.12951617795780537844603599003886257249","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22990786875064680564672188701015740170","series":"1.2.826.0.1.3680043.8.498.73924546614480369902106065840837099065","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23046618786812839762801821364158313134","series":"1.2.826.0.1.3680043.8.498.81665263308550258208125552671593703160","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23065868481290440008286663193810913414","series":"1.2.826.0.1.3680043.8.498.10193421103817217173710546319909670101","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23094733099955217022059584418468076048","series":"1.2.826.0.1.3680043.8.498.59755796796418869838454051956985710402","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23182005205627494484490011351673355568","series":"1.2.826.0.1.3680043.8.498.23917941350181989903401381737294391485","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23409201792139701921731647470152180932","series":"1.2.826.0.1.3680043.8.498.16775868422610301191015184270547399741","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23427198594727947883953289189443006288","series":"1.2.826.0.1.3680043.8.498.10261722270486719227047013261409695867","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23677916081296740815989149501889343663","series":"1.2.826.0.1.3680043.8.498.13352700998579499246844762703493557836","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23703817737293327679300330167234268929","series":"1.2.826.0.1.3680043.8.498.11124248686344041665830607150250842409","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23733147518259581649562651915997449088","series":"1.2.826.0.1.3680043.8.498.89720130515207778822887718965902789826","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23822313132245448173452915228147497501","series":"1.2.826.0.1.3680043.8.498.36706497608271950515608592380361786282","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23911658810234305013984976739924353704","series":"1.2.826.0.1.3680043.8.498.91222498624167450283311349753037909282","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23986051274386630509877205174270047762","series":"1.2.826.0.1.3680043.8.498.43123209844004057570859504710598562875","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24068239310155798723127213793623172530","series":"1.2.826.0.1.3680043.8.498.20320985074605834124636039331310559621","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24209643225939781032698894992032106781","series":"1.2.826.0.1.3680043.8.498.57942148712961695867429756478457786528","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24220732394554303343522053195582322788","series":"1.2.826.0.1.3680043.8.498.34686995036245794859476286697956883772","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24291465113360632954006568021942978765","series":"1.2.826.0.1.3680043.8.498.99500485107281028636575065459834119705","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.24313647347273496316728514161733307557","series":"1.2.826.0.1.3680043.8.498.12434657490155360428154867213760534763","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24478195415117310702720951368429189290","series":"1.2.826.0.1.3680043.8.498.72452756331848271661324866859899515059","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24529300482982370922479679056174585978","series":"1.2.826.0.1.3680043.8.498.10097114342487245269502915278041526907","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24588524807799377432185260702152371503","series":"1.2.826.0.1.3680043.8.498.11476099538316043507721997941987794331","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24679905857546029995247839380830345841","series":"1.2.826.0.1.3680043.8.498.98337037410761905899784479415257163881","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24714029982431301486050388345836960910","series":"1.2.826.0.1.3680043.8.498.99863158819314010420145665518660469887","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24826041178550854190310664677274404684","series":"1.2.826.0.1.3680043.8.498.59908021763865724301573574942804650740","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.24872746733270791851241663716306993824","series":"1.2.826.0.1.3680043.8.498.11469401967247091624769873573071113084","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25006652485159423287561656594171884310","series":"1.2.826.0.1.3680043.8.498.10955849333759736682723071223287641650","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25088333117273619121936521384053709674","series":"1.2.826.0.1.3680043.8.498.20278941921926715794372112843288860508","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25145588904873072866960435474159832047","series":"1.2.826.0.1.3680043.8.498.18117793850909664339646434167666996670","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25167060436407065195851060078175404789","series":"1.2.826.0.1.3680043.8.498.67565739549117807840277548171128667731","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25206096958520407302630747466352252249","series":"1.2.826.0.1.3680043.8.498.98849527979669448354181860044036440606","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25259588756496584739814764896903786709","series":"1.2.826.0.1.3680043.8.498.19601609373697687242941611684804006528","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25359770839263995354733835365510047109","series":"1.2.826.0.1.3680043.8.498.10278708218513111258444658592411019917","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25431207706481141354761246438973384670","series":"1.2.826.0.1.3680043.8.498.11061835977317429793697356754847694880","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25487225187305802150215261512092051199","series":"1.2.826.0.1.3680043.8.498.40560493664929699808179869232232883280","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25512240159357553134500245590935010104","series":"1.2.826.0.1.3680043.8.498.77771300823080189174043221865246457754","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25558526735614916746398919603493693652","series":"1.2.826.0.1.3680043.8.498.32892100099142242842321710711033735831","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25616615589791001678067099907823838797","series":"1.2.826.0.1.3680043.8.498.71267934639404970881744880271151396705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25650384476012633612528314070894186451","series":"1.2.826.0.1.3680043.8.498.13696643879322301958022154558687268353","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25663827732039068221184137540713755716","series":"1.2.826.0.1.3680043.8.498.20302883874262945466541880307983057917","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25697539332996211770844878649053316629","series":"1.2.826.0.1.3680043.8.498.57292875564238378715215793673935415716","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25765711614507755527825654507920391096","series":"1.2.826.0.1.3680043.8.498.44199369362169513517131858265063627759","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25792216289349936450377008453709520260","series":"1.2.826.0.1.3680043.8.498.10368434985030998353308368406291867699","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25818541516137783435601858274495597736","series":"1.2.826.0.1.3680043.8.498.12693717000757170588275113770103841223","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25904473139226718487560978103077541725","series":"1.2.826.0.1.3680043.8.498.12613831064111235491488745794649765246","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25928779639851562853500186715690156268","series":"1.2.826.0.1.3680043.8.498.87392387749408229209028993215825433146","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26002972602557466222471260313257965551","series":"1.2.826.0.1.3680043.8.498.19252443779891975273509912881345386233","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26006372873438224908989402529530833528","series":"1.2.826.0.1.3680043.8.498.11098294144429090729873382324403176159","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26072622476167889001255150030772906081","series":"1.2.826.0.1.3680043.8.498.67814767012570828693756172301396776830","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26257425287121280175418060407044075705","series":"1.2.826.0.1.3680043.8.498.10274770750288713015714303148393672097","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26325951519908477683595799515094966069","series":"1.2.826.0.1.3680043.8.498.56065842918240444180055329328137750919","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26373657290121957625263322898744712730","series":"1.2.826.0.1.3680043.8.498.63486486003116911942702103321824186032","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26431974442245797095978351976773354669","series":"1.2.826.0.1.3680043.8.498.11012848063166174188089485397679094837","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26485916396240515006042297316506505324","series":"1.2.826.0.1.3680043.8.498.10662579038093138779984220387670806412","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26542435427465415008860869796504112727","series":"1.2.826.0.1.3680043.8.498.80534597305312767338883847929616047368","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26698126402727061400851465377574886112","series":"1.2.826.0.1.3680043.8.498.94225761605047268297376438423355771205","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465","series":"1.2.826.0.1.3680043.8.498.76639110060009477040774057211843609294","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26843088620128166809417361066548447201","series":"1.2.826.0.1.3680043.8.498.18950019902615356428287843550739668548","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26880562799522930458745136318439622601","series":"1.2.826.0.1.3680043.8.498.35113723618015908336407205204536279210","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26978027400525885904983020900996228004","series":"1.2.826.0.1.3680043.8.498.72580786429084863941402332620616945441","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27023892731021470476500135946317920771","series":"1.2.826.0.1.3680043.8.498.36655115818017898805723081923913331794","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27034189091164693165076510006436405194","series":"1.2.826.0.1.3680043.8.498.13069885116801347519211054944908683990","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27161418589407291713644384596541786246","series":"1.2.826.0.1.3680043.8.498.24464822604368868885944643725135678972","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27186001621007168581611489439038285508","series":"1.2.826.0.1.3680043.8.498.79696589746719738671704061214442246966","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27192678730060203972082511503480566620","series":"1.2.826.0.1.3680043.8.498.17396428235108103979562208846344519258","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27224327142578332905227939015479767285","series":"1.2.826.0.1.3680043.8.498.11612119772984227014971167956922181157","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27311413427137994435573903541974122355","series":"1.2.826.0.1.3680043.8.498.49822624596745433983578061102145295981","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27324284859290040148888098857880927353","series":"1.2.826.0.1.3680043.8.498.62566401017985565680067495197067306024","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27398034409980350921804398626171135707","series":"1.2.826.0.1.3680043.8.498.10719273385974317401786009147249312120","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27456622705361831116491505615162380604","series":"1.2.826.0.1.3680043.8.498.87989434377287930754964431872497919417","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27474440682441854303485403899867525864","series":"1.2.826.0.1.3680043.8.498.40168019612595937304846984998559349020","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27644321991318957631114727895883331622","series":"1.2.826.0.1.3680043.8.498.29660068881991262492269807077322774266","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27655410895168998898506695863000928400","series":"1.2.826.0.1.3680043.8.498.31256818205528869346739936832622381201","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27681651277422558344520227187021583924","series":"1.2.826.0.1.3680043.8.498.42472038773573179227871679105768097718","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27723413941379681806317930349613970721","series":"1.2.826.0.1.3680043.8.498.12994612592612102757879623625750784814","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27737101979486411869208236811885597351","series":"1.2.826.0.1.3680043.8.498.10517009190595642577653292741718010192","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27883371281368994280894952749865160316","series":"1.2.826.0.1.3680043.8.498.65884412657021545147938597260330267770","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27921111050680927418542588425335218896","series":"1.2.826.0.1.3680043.8.498.57520043663633962269046960414931401322","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27934397251575987048967084987994202973","series":"1.2.826.0.1.3680043.8.498.80288323492836220469622767719309267398","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28011057282528460306257420499560926354","series":"1.2.826.0.1.3680043.8.498.71036629567317742417520120382248032131","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28028890493668487238845659163252231337","series":"1.2.826.0.1.3680043.8.498.57063452162924679989798166251180468331","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28132506470634542229651932141193521661","series":"1.2.826.0.1.3680043.8.498.37539052914478368929039320716422037037","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28153286315438977482271934270979508349","series":"1.2.826.0.1.3680043.8.498.58612477617378653043043836679400605260","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28237068093947300427040925444622840795","series":"1.2.826.0.1.3680043.8.498.18203423874958594406292912997398334734","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28289838818003822677994912735532050038","series":"1.2.826.0.1.3680043.8.498.12769794541700728524905691038829769995","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28333127304382930639614800697959296306","series":"1.2.826.0.1.3680043.8.498.11391511934660341391081239353550887863","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28360174615866226538417948042981668998","series":"1.2.826.0.1.3680043.8.498.10375589571692726534629929584234582678","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28488216314048990109149780234417422948","series":"1.2.826.0.1.3680043.8.498.39935617674212694528825781600591575352","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28539130184243787339297053654373783542","series":"1.2.826.0.1.3680043.8.498.90901496223099642463487949009913592769","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.28595729988867300278425411189173775382","series":"1.2.826.0.1.3680043.8.498.45762806670293118643294465887256498725","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28634248756653012357434232845788268719","series":"1.2.826.0.1.3680043.8.498.52997879340576030342633474808444804219","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28646304373057947277669139986982208473","series":"1.2.826.0.1.3680043.8.498.65221817625949921732727751733141061563","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28686498221359515429184385385077600687","series":"1.2.826.0.1.3680043.8.498.12942293505517503500635830823879178126","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28746954364434312909368235768936980607","series":"1.2.826.0.1.3680043.8.498.78775065568289180100033531284212751033","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28848511016500781798837621261660844237","series":"1.2.826.0.1.3680043.8.498.59269635849791275362629708621917632088","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28888941572409703700315590339486528256","series":"1.2.826.0.1.3680043.8.498.50372843936846591946463949251143137060","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28946722216615442736212359468410891872","series":"1.2.826.0.1.3680043.8.498.49334423973384921589145949606220828229","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29039872266866623711060910233238440882","series":"1.2.826.0.1.3680043.8.498.46146334943773743742332896045920541882","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29090158364700006304493717187346344597","series":"1.2.826.0.1.3680043.8.498.41859559941160943257107577755687143289","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29292398668403593100514493322849770181","series":"1.2.826.0.1.3680043.8.498.10705232377807841085399262656899568357","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29354469534263517460113311852821643665","series":"1.2.826.0.1.3680043.8.498.11360036156872003339760525212064834715","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29371692234061436334815897970798625595","series":"1.2.826.0.1.3680043.8.498.69136802965764332759892441372485449086","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29384016102624554484300773136383156153","series":"1.2.826.0.1.3680043.8.498.10251820498342157206687004068530631558","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29473278891551292295080887778714088878","series":"1.2.826.0.1.3680043.8.498.41793296260579575581648125706848694680","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29579560656789178333629609998252952856","series":"1.2.826.0.1.3680043.8.498.87655002688607398104414949254891574237","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29739171278960083313048266628194182569","series":"1.2.826.0.1.3680043.8.498.46474378292524299529210685011971323081","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29764868091238287072166823522853419550","series":"1.2.826.0.1.3680043.8.498.93049579888925063520771775891465039949","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29826129803278661744835136993610646069","series":"1.2.826.0.1.3680043.8.498.70765589626944821951326715248561349022","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29971686675947075024168289004002445393","series":"1.2.826.0.1.3680043.8.498.33445751949407509140511659599531727523","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30066410959464367576375502522994033142","series":"1.2.826.0.1.3680043.8.498.12636071194952765762679366004383986396","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30131339052241668380936148362472245259","series":"1.2.826.0.1.3680043.8.498.11418206484416708635755495875682447373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30167179890902431464942556467698238073","series":"1.2.826.0.1.3680043.8.498.12798906103197057399985288341668753560","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30222189977108980260768952691472020722","series":"1.2.826.0.1.3680043.8.498.12279501514754075749189809797794868260","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911","series":"1.2.826.0.1.3680043.8.498.13364655615747190896475164649441911015","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30264599677934095429161619981918944098","series":"1.2.826.0.1.3680043.8.498.61411627988380390309405021854458039167","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30330001893437233719772129404417384472","series":"1.2.826.0.1.3680043.8.498.32254979018700276243650983047510536747","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30423643741419627789701272644909134762","series":"1.2.826.0.1.3680043.8.498.22027362853261208543414576126216240703","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30490857770169699336468319880130884989","series":"1.2.826.0.1.3680043.8.498.11425095346697087387836589151202503451","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30520444048368652235940346583890702451","series":"1.2.826.0.1.3680043.8.498.94827831760687985007886743554256974191","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30613528335797277620843542280889823612","series":"1.2.826.0.1.3680043.8.498.11933943832775909235288889053347467927","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30672390215843894382866701432377109444","series":"1.2.826.0.1.3680043.8.498.11920911535587736088357677838878254777","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30746167840159843733412534591988948270","series":"1.2.826.0.1.3680043.8.498.11360770261665698001535522224605906325","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30763696628968291934469221453461209579","series":"1.2.826.0.1.3680043.8.498.11566927067821570149615976856298983633","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30832771419882354941430609179589560374","series":"1.2.826.0.1.3680043.8.498.45474944400886137645359521764276575706","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30842406081180392627544336691360658070","series":"1.2.826.0.1.3680043.8.498.77565407518088221505844454637580647744","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30942874102560162331831960649006411381","series":"1.2.826.0.1.3680043.8.498.13186486821274451807528555876730110114","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30998941785989496591079894962766360976","series":"1.2.826.0.1.3680043.8.498.78436895603792252411726734003124858956","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31080452089421812812019413302222437434","series":"1.2.826.0.1.3680043.8.498.22617645152631299626024454472364257544","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31099878224304681441631176766255029271","series":"1.2.826.0.1.3680043.8.498.24718845335586396638422314959291258939","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31171050331393253655760166990008263793","series":"1.2.826.0.1.3680043.8.498.11774137672139032701185196359964557366","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31193772347652083986607183406944630716","series":"1.2.826.0.1.3680043.8.498.11088943311181685684522554712025605019","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31220970065722563280311764269386457090","series":"1.2.826.0.1.3680043.8.498.12919496990401970668406752551441415420","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31280487369922340170695453174997912050","series":"1.2.826.0.1.3680043.8.498.11753709004566234898338787712410640602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31352187408311685267245948214616940728","series":"1.2.826.0.1.3680043.8.498.11805791514276215517082885739144668015","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31461240229123701082444627629841504099","series":"1.2.826.0.1.3680043.8.498.10604214058599070418340239291579123435","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31555333355990829772911515362502000209","series":"1.2.826.0.1.3680043.8.498.10531545665261055833730518433910793948","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31640159014711750808214354395404624815","series":"1.2.826.0.1.3680043.8.498.21558075734190038620294485036708129414","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31792234576444017394182974951507838927","series":"1.2.826.0.1.3680043.8.498.10077711367735159580771782249334425786","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31827987476038668074615045841102711882","series":"1.2.826.0.1.3680043.8.498.91587614899914924277508114922047791057","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31851627014639252866084081561126592250","series":"1.2.826.0.1.3680043.8.498.51288997156339120720955190187792117113","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31949224274318638796196151105617844024","series":"1.2.826.0.1.3680043.8.498.59972482238366030752939890298847397291","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32078592787308498852570411454111970699","series":"1.2.826.0.1.3680043.8.498.74004561473301462597588985304127837661","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.32172672595181417103935593872811182816","series":"1.2.826.0.1.3680043.8.498.11750683369296325522209569034700069733","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32226253011052584155674401943276961907","series":"1.2.826.0.1.3680043.8.498.13067017606487067671093070932245983766","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32299120582647333604770626110392750631","series":"1.2.826.0.1.3680043.8.498.11791243560392575912181192881782860059","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.32328699307307661181881949763360098439","series":"1.2.826.0.1.3680043.8.498.60929010224737779975473004529718079023","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32413534416513678586709935453506443461","series":"1.2.826.0.1.3680043.8.498.22725163833998912255899261647148764332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32488933488399304214586978746578986823","series":"1.2.826.0.1.3680043.8.498.70520475770229948761245414251779149172","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32513010438338970418994141548516096399","series":"1.2.826.0.1.3680043.8.498.35713469664887938586978338909425520480","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32571577475600416042179474541176580933","series":"1.2.826.0.1.3680043.8.498.11573982916068025198985100256026926374","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32611545463217490575457831874695918229","series":"1.2.826.0.1.3680043.8.498.20028234198241050550064868368828111409","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32704159300269838731722725240928883385","series":"1.2.826.0.1.3680043.8.498.63442204518291265647779028019882557967","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32742383730417896527915284404039812924","series":"1.2.826.0.1.3680043.8.498.32445896157027226467480295515746588108","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32862727544234948830522036721585095753","series":"1.2.826.0.1.3680043.8.498.26914818447097167301829757322929006836","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32916544979513354644075485240094539251","series":"1.2.826.0.1.3680043.8.498.98980203208974551948452032956450234606","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33045687849853291066101186962073873042","series":"1.2.826.0.1.3680043.8.498.92246696273993776490824084188909160807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33215387283623093392227819856970670071","series":"1.2.826.0.1.3680043.8.498.12621145057217394629541457313774661072","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.33273874534046658504935877940537027702","series":"1.2.826.0.1.3680043.8.498.86660623961749410755729062401011621267","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.33324606756355240783076971858231422510","series":"1.2.826.0.1.3680043.8.498.13291573991407864263836444114134503536","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33387430823462011959198830568803117633","series":"1.2.826.0.1.3680043.8.498.10271016864369318810721343160072928827","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33427080769740419588952225424281396684","series":"1.2.826.0.1.3680043.8.498.52308304912748264325045755971111753545","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33479806419966566464727310013949313521","series":"1.2.826.0.1.3680043.8.498.77107254815354533085668213715912329531","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33620268340279901383196638228030589893","series":"1.2.826.0.1.3680043.8.498.32985471088688625488920205108424636075","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33707148112045883958228656491167647755","series":"1.2.826.0.1.3680043.8.498.10911706226640540019759775693048311898","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33834227140201599591317512187502720859","series":"1.2.826.0.1.3680043.8.498.51297956533110434921063885960487705794","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33880013318702285028756482543453984598","series":"1.2.826.0.1.3680043.8.498.68633105141236050965957455528612424146","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33947148016115721563601115463364787679","series":"1.2.826.0.1.3680043.8.498.52605439280782483027129040081386632260","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33997019840409023169542161093833920630","series":"1.2.826.0.1.3680043.8.498.68594150914710439486448277725149636782","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34040631818813472592501398076075999835","series":"1.2.826.0.1.3680043.8.498.13092435215559461478334909327855839293","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34096149829586714035779314398766243514","series":"1.2.826.0.1.3680043.8.498.60426264222482977060481164971736879627","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34126843286927697018624309801437341727","series":"1.2.826.0.1.3680043.8.498.12524672905606729087169483239588086577","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34294522885703132478119231619149118745","series":"1.2.826.0.1.3680043.8.498.34067880595096226682562855862738123640","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34379812274299658721362253287768041613","series":"1.2.826.0.1.3680043.8.498.11697769467818821002763368737480739022","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34433343440016591348979084714300062954","series":"1.2.826.0.1.3680043.8.498.12740372321201741722649177304133702243","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34461302710296907289543656738174044465","series":"1.2.826.0.1.3680043.8.498.35173248906102070841454753868707117458","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34572444983416782410188788741787280444","series":"1.2.826.0.1.3680043.8.498.23936134682964283696612573410065068586","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34636178772986348716257313987840123906","series":"1.2.826.0.1.3680043.8.498.10424136630372831497479450454535622686","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34685905030370793639196564723935583035","series":"1.2.826.0.1.3680043.8.498.10354958299917644989297776650748117510","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34740783941206992242367609204835771599","series":"1.2.826.0.1.3680043.8.498.74701941058555500246844006291374646784","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34791780913322886484171421115861604853","series":"1.2.826.0.1.3680043.8.498.46726962731665696181756166670054400185","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.34826100703045042103752460066224521343","series":"1.2.826.0.1.3680043.8.498.38306391410221340200183992809680611435","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34895742092137516664772577251243433579","series":"1.2.826.0.1.3680043.8.498.94225958398724704097288074508978677897","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35017973257585430544005763350601015208","series":"1.2.826.0.1.3680043.8.498.10213289126459350939330224107930907312","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35027750475162395113025853006474122989","series":"1.2.826.0.1.3680043.8.498.28773984598120489171310807313001187276","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35090276406828326770336478753776261776","series":"1.2.826.0.1.3680043.8.498.44925325763767388953226773430518999393","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35210451107308788915675910467662426761","series":"1.2.826.0.1.3680043.8.498.11192977875824263878450583189801587799","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35257158249961136373696255071228322165","series":"1.2.826.0.1.3680043.8.498.10151175395030773776410110426248004517","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35330802038595377723999341992570259909","series":"1.2.826.0.1.3680043.8.498.90728587017606964942869606881463154670","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35542843700847404584397532402313561503","series":"1.2.826.0.1.3680043.8.498.62951588096541114385451136953757545754","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35572306796137927247415883445612561056","series":"1.2.826.0.1.3680043.8.498.33158072264603832796708355779259113632","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35664741425496814074030858561751287211","series":"1.2.826.0.1.3680043.8.498.58189191146667494760084783199003464050","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35717294165699019392845830077937768611","series":"1.2.826.0.1.3680043.8.498.86367310919973598186458360281745808249","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35790589718590759254855277551289842944","series":"1.2.826.0.1.3680043.8.498.14939780782061274360991806524510418394","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35826402289277331555736683320175912112","series":"1.2.826.0.1.3680043.8.498.44557846774158669149479411405190623307","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35836928928058091851993989259747528067","series":"1.2.826.0.1.3680043.8.498.12411233573720774329809665851041032419","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35892665011493087075493127252825546314","series":"1.2.826.0.1.3680043.8.498.12237366139005282165982861925179144786","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35923146679164443588381195543533271939","series":"1.2.826.0.1.3680043.8.498.23995491990008360127618975285175857846","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35932368810339127867306228699757809449","series":"1.2.826.0.1.3680043.8.498.12050472475496662150882104629121166658","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35961087303991442751747064990497975939","series":"1.2.826.0.1.3680043.8.498.28354795846379763657564060957042850637","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35991881811942962264382690224767291006","series":"1.2.826.0.1.3680043.8.498.10923816872661432410273569044203497916","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36083794505435625841225237415909011547","series":"1.2.826.0.1.3680043.8.498.12467961735622174112971843951043001552","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36131369236532190338276701919104598475","series":"1.2.826.0.1.3680043.8.498.60364098257536513347102799167925432994","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36185754164795053628650116141620545139","series":"1.2.826.0.1.3680043.8.498.48327364781947758088996755130611397214","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36221680997794815863088530960610069278","series":"1.2.826.0.1.3680043.8.498.48220433806595051377220673578358704290","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36293583160636884934622362743062825202","series":"1.2.826.0.1.3680043.8.498.11053099860384402544704153284740138971","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36431753977367847587395456605556703040","series":"1.2.826.0.1.3680043.8.498.70384250579865988647021503930426887172","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36488342368050295711098478501722217415","series":"1.2.826.0.1.3680043.8.498.38023644154830439961044290789269963934","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36588608184578739092409073014150374185","series":"1.2.826.0.1.3680043.8.498.15765981510242737982721107482685330755","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36648377536792309029975432419163533533","series":"1.2.826.0.1.3680043.8.498.17174068714383047715613379784318513617","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36688959138536319644194458211515765203","series":"1.2.826.0.1.3680043.8.498.12045673228486496507664990188337587397","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36812284468308169396147366740331573100","series":"1.2.826.0.1.3680043.8.498.33094556885647399753090115511794572661","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36846391753332367835804699576399553980","series":"1.2.826.0.1.3680043.8.498.13071822073236872968578665171442427222","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36924091633099385100942228071545017565","series":"1.2.826.0.1.3680043.8.498.12485672050104621997889690793437434509","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36995206854684735704099881847485073991","series":"1.2.826.0.1.3680043.8.498.20734790499635135151774210809038477414","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37030593568962187244853961673610373876","series":"1.2.826.0.1.3680043.8.498.45192915720447386830565323256342832795","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37046961643473249831189497076699393868","series":"1.2.826.0.1.3680043.8.498.60937131125500428965739508229860177259","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37147839676713817130306478240352601535","series":"1.2.826.0.1.3680043.8.498.38337809598829511418980399043447595339","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37257728513136112802363958568529758895","series":"1.2.826.0.1.3680043.8.498.48821816986591220637405880719690619726","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37332768072749723117839591872457536315","series":"1.2.826.0.1.3680043.8.498.10944281190307064396601712592036009544","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37390494423735156928337187351900345080","series":"1.2.826.0.1.3680043.8.498.82208691180097120666448664169603782158","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37413349933694832641902637847629969234","series":"1.2.826.0.1.3680043.8.498.11429153337473666956466996099813459389","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37438631765770807892914198396852544668","series":"1.2.826.0.1.3680043.8.498.38397474135403526399862835841730646141","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37463970179664003092832463732043124908","series":"1.2.826.0.1.3680043.8.498.73413009928147915260677164086026165738","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37510478969810784660124747242051166554","series":"1.2.826.0.1.3680043.8.498.74152360720063855786706298968843316371","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37558426555271635608942179071280432530","series":"1.2.826.0.1.3680043.8.498.12365536133209712669262500410493029600","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37591186611460576939009950935820113903","series":"1.2.826.0.1.3680043.8.498.28614376838634012743589474210728322224","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37633035024065331400434438202028794597","series":"1.2.826.0.1.3680043.8.498.75889965532316101598570724907839431351","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37713008080610827760634302982073393454","series":"1.2.826.0.1.3680043.8.498.58782993977377596186625467063869560287","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37782918081276484321055690005029111967","series":"1.2.826.0.1.3680043.8.498.77221123034013258041733858238672116361","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37846940236952007502147453013489359606","series":"1.2.826.0.1.3680043.8.498.10216685221704468701923371753850119821","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37879265071349098151803248077034280617","series":"1.2.826.0.1.3680043.8.498.68828232655620055874109853252848614663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37936061471366103885185528435327650981","series":"1.2.826.0.1.3680043.8.498.95941055006982860811063365203644837682","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38010459551726691601986492657296948013","series":"1.2.826.0.1.3680043.8.498.13308610931319892021730448563266739803","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38039631093067555454700266397073080430","series":"1.2.826.0.1.3680043.8.498.70073707803775716546213676851185016662","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38085359363401170553191582038767900323","series":"1.2.826.0.1.3680043.8.498.11910920592147986874246468642413922040","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38104228771638294693272534500734878990","series":"1.2.826.0.1.3680043.8.498.10719605272964681712173995663524518061","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38233845434092134418342290639986261345","series":"1.2.826.0.1.3680043.8.498.18916587549319171583244754760178104574","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38272345519163551901865251730721530169","series":"1.2.826.0.1.3680043.8.498.52068254480549997354787465024921026056","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38306262614552497897681886368029860358","series":"1.2.826.0.1.3680043.8.498.29724081171457094649581066844424006902","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38333934623775971943734069301776225018","series":"1.2.826.0.1.3680043.8.498.81324280869183943203869232075390810189","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38373204312077453422262347769111605003","series":"1.2.826.0.1.3680043.8.498.12654860042650224597001503850913380398","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38427680930662301114332563903957471851","series":"1.2.826.0.1.3680043.8.498.13319642355522565525276906947304901021","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38485168845116671268554752030254772359","series":"1.2.826.0.1.3680043.8.498.61061785190029658981331842669148186481","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38528511947818361078957998546327084024","series":"1.2.826.0.1.3680043.8.498.63003634217601976738808993258889942994","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38584486469155131627808759862008577200","series":"1.2.826.0.1.3680043.8.498.11055220035039504899977001101979878478","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38593351637971248239815933638950449587","series":"1.2.826.0.1.3680043.8.498.12319300144991064893410092229888090247","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38655145317957160440692625317823254736","series":"1.2.826.0.1.3680043.8.498.89410977455869933116335844015030477823","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38717329577625255474350472405748992219","series":"1.2.826.0.1.3680043.8.498.64171670014856614581835149341683538932","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38774313460931748278644354956408578813","series":"1.2.826.0.1.3680043.8.498.11935411618727132867829530964949478942","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38844149478205202385909887748058004820","series":"1.2.826.0.1.3680043.8.498.65528128832174778747337805278051641655","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38870467957974446108337293912930572060","series":"1.2.826.0.1.3680043.8.498.76974611283354223096468006527198467738","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38900640030483599614088224967855884221","series":"1.2.826.0.1.3680043.8.498.10585372755860325949452680776809534693","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.38925755872951351471027638563742132144","series":"1.2.826.0.1.3680043.8.498.81072097652111149791698225721457471399","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38940868544480800333574786964232944756","series":"1.2.826.0.1.3680043.8.498.36216429487429576251885091915520284219","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39020025030013167602991688044932114354","series":"1.2.826.0.1.3680043.8.498.80250661619408693646552957650959238859","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39025433784711757736168236829157583043","series":"1.2.826.0.1.3680043.8.498.63108154505248189452220048875044789288","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39070169128115370701189151138307035053","series":"1.2.826.0.1.3680043.8.498.58542643826756717410079464478563773719","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39197976722851762826265579540678924584","series":"1.2.826.0.1.3680043.8.498.36742885588622385864522586117424142386","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39264696726258404531295961801831089064","series":"1.2.826.0.1.3680043.8.498.10956649926454054904038863719797492789","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39375796600811714870169173423682043709","series":"1.2.826.0.1.3680043.8.498.13335884453995398771458089442957924864","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39399668753650041254826892657562722764","series":"1.2.826.0.1.3680043.8.498.62094862380533150116683443106682015540","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39507184510584030713933330702725991027","series":"1.2.826.0.1.3680043.8.498.11088087215558738738739006785195091956","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39578126940710463296572383286587298912","series":"1.2.826.0.1.3680043.8.498.11345870820804598146539493113448439160","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39636073039676814228254715705296535819","series":"1.2.826.0.1.3680043.8.498.96277132044074620533664927393948748602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39799597341675508044808282472428242917","series":"1.2.826.0.1.3680043.8.498.29174522500069339662546892377483550632","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.39875636232483792306109110284167439592","series":"1.2.826.0.1.3680043.8.498.85343616597492362090427915275961040809","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39932424305167611414527482641898907759","series":"1.2.826.0.1.3680043.8.498.33571595257520229551523312409299550822","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39969071491383654382076256213867635377","series":"1.2.826.0.1.3680043.8.498.11726982363242891659910075327138057706","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40041164441539690764994039690598305675","series":"1.2.826.0.1.3680043.8.498.79320491945302658005862273346229265161","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40154708689850542672289658249059660788","series":"1.2.826.0.1.3680043.8.498.10745827767614990017599750143627330199","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40223239401390462911823193489179244754","series":"1.2.826.0.1.3680043.8.498.61365682055132123383310854539472331095","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40254123984175499564145728084464192446","series":"1.2.826.0.1.3680043.8.498.10310175537935591892484509015474430664","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40308838286515017547707130085914948230","series":"1.2.826.0.1.3680043.8.498.36561577375713117556142318739832900753","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40372643537313871452054321931253684619","series":"1.2.826.0.1.3680043.8.498.16347612013726692196702901325042948045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40436886205373732563782477802719203679","series":"1.2.826.0.1.3680043.8.498.33555202182083465060739957403625126071","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40441879276927383122480339472406105572","series":"1.2.826.0.1.3680043.8.498.56458102240266280818477062172951630753","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40533893516114346078820881178450524201","series":"1.2.826.0.1.3680043.8.498.36064638631435791234876242094198393057","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40608233778918249560056537030108548037","series":"1.2.826.0.1.3680043.8.498.21357856941188379066377641943179997618","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40627043074325458726412346974457616938","series":"1.2.826.0.1.3680043.8.498.99597812831959900025117464688036602654","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40703974473944295875248011843170371394","series":"1.2.826.0.1.3680043.8.498.10353507183645747509706957254539263059","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40723835064354784133289444204866484476","series":"1.2.826.0.1.3680043.8.498.72420586450147923490112998045713805936","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40823842819404259265489417470859248075","series":"1.2.826.0.1.3680043.8.498.91494856702384212339401827785618009677","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40898220266193841898869905714214454387","series":"1.2.826.0.1.3680043.8.498.13326787438999959463074781776659238601","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40930045208346676628855922755791787949","series":"1.2.826.0.1.3680043.8.498.30274070662444577886120892707831534321","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40994616246264625946610553907227975893","series":"1.2.826.0.1.3680043.8.498.88052307116668483076272506647589925461","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41108421820714108307078614688022098388","series":"1.2.826.0.1.3680043.8.498.11566285769547543571121167787012522957","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41129090437522031508960438281451672212","series":"1.2.826.0.1.3680043.8.498.87995145545138838232262576153518544705","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41254472410580125134596211010943842213","series":"1.2.826.0.1.3680043.8.498.86274802270539049011420809084004361759","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41326575785634778198143222323166756210","series":"1.2.826.0.1.3680043.8.498.11168811957095003572885516032087944829","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41366889271483886167284378157366992074","series":"1.2.826.0.1.3680043.8.498.96552084221529327972702114860465505155","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41440206520769979754790425253631809212","series":"1.2.826.0.1.3680043.8.498.71362715290433874967711982214327848862","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41564229176277158115386256329157870722","series":"1.2.826.0.1.3680043.8.498.38146495697655925758060465202790099887","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125","series":"1.2.826.0.1.3680043.8.498.21445139509346804994276778291771624587","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41623512571866040054960002151226127775","series":"1.2.826.0.1.3680043.8.498.63874667259007594752093398739442601391","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41657361598083583047831423768034379414","series":"1.2.826.0.1.3680043.8.498.97332974008144584905635222628806391407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41696746845418651508748386392327996143","series":"1.2.826.0.1.3680043.8.498.42771547752886908025601203699527943159","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41717437729988689273718049162153247404","series":"1.2.826.0.1.3680043.8.498.24472762518497759435303800817925171824","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41769275806689876192405091865340286535","series":"1.2.826.0.1.3680043.8.498.25924359941745982488992274066925179758","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41837648239243141231469586963862388683","series":"1.2.826.0.1.3680043.8.498.53373748292577815026244432957620243833","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41854610297797849683195403204758363476","series":"1.2.826.0.1.3680043.8.498.10192355622021821678692960962606450280","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41990779875873214154237496750949660480","series":"1.2.826.0.1.3680043.8.498.12111812107625294940835043597693676340","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42018683052544325209168649309206539721","series":"1.2.826.0.1.3680043.8.498.82443844526611229845455566195822167605","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42057349781448469704191384429694247725","series":"1.2.826.0.1.3680043.8.498.62764992569230170429134853337192526679","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42130173624033946465704266507948122088","series":"1.2.826.0.1.3680043.8.498.94819125116394675884211129632955030295","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42225165085995728538373532053227662192","series":"1.2.826.0.1.3680043.8.498.54044870661855122573212744845860968489","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42389063782063360163798575455522539633","series":"1.2.826.0.1.3680043.8.498.62785065277922913180543675215803750217","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42414269167296543219685897036829241885","series":"1.2.826.0.1.3680043.8.498.13277003715175106085353226524764547097","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42486421970612099441197418307958561877","series":"1.2.826.0.1.3680043.8.498.13393397033425094661536635226898481556","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42580107838226807862930203257414265276","series":"1.2.826.0.1.3680043.8.498.73188318763407156908108687791517535531","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42609322631348207494943622999125822017","series":"1.2.826.0.1.3680043.8.498.39931565369935202744181255304983589986","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42719592405241698450757802131459694665","series":"1.2.826.0.1.3680043.8.498.11154856714595400511481727388749216907","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42776652100045349404152232788825670056","series":"1.2.826.0.1.3680043.8.498.63259173970963051349800113151044329748","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42829490507052740062127365747846030214","series":"1.2.826.0.1.3680043.8.498.58066752621718323204037303835575855898","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42887028634948156598417204377884297339","series":"1.2.826.0.1.3680043.8.498.13349825914649819002018713598087154247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42921593713351589642747633505679170597","series":"1.2.826.0.1.3680043.8.498.43209600700224981347005422372706678355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42967970121147528831983176261453235744","series":"1.2.826.0.1.3680043.8.498.97117964712012474919481941264606896960","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43046713539641995841029654207663845887","series":"1.2.826.0.1.3680043.8.498.99056301755255912366031650549635154995","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43117007553940146763245418710268028741","series":"1.2.826.0.1.3680043.8.498.11248293529657977508104948565598663692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43256282639827216310036033378453939750","series":"1.2.826.0.1.3680043.8.498.12231893101289284108575112072982333496","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43300763182450509440662392516941842389","series":"1.2.826.0.1.3680043.8.498.76299011898542650593443200061916301257","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43351215447181735764667371462573550267","series":"1.2.826.0.1.3680043.8.498.12632673997770665726158582403995674768","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43412154740072739455825163326814980838","series":"1.2.826.0.1.3680043.8.498.91688502205805231746324292208084535052","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43442933855807685981061557274166869461","series":"1.2.826.0.1.3680043.8.498.89002646770000543281136726113833959726","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43587191909319947256183198986671106743","series":"1.2.826.0.1.3680043.8.498.68093366684243560971762969321520123650","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43709699920591091261137846976272625163","series":"1.2.826.0.1.3680043.8.498.22727945834613820615118676508786582929","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43797037142729611453626414920554246615","series":"1.2.826.0.1.3680043.8.498.48411991882504481570837029408153769115","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43967752580489865323542041346927244370","series":"1.2.826.0.1.3680043.8.498.96833661815444498731459162295808264268","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44024350801191362011463613321559949052","series":"1.2.826.0.1.3680043.8.498.91667014962419541436951796248183124616","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44259687212868598275713310246316207497","series":"1.2.826.0.1.3680043.8.498.11157139793007334379760978363364185956","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44323131330689365651417881352766725366","series":"1.2.826.0.1.3680043.8.498.28184689114707581367242660725227695010","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44350638600163778456163342454087669278","series":"1.2.826.0.1.3680043.8.498.23876817236344700315144156320383610551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44417768181231991989438194369583907493","series":"1.2.826.0.1.3680043.8.498.13258600070972176826943509166521531604","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44441813293962885868294317830142850713","series":"1.2.826.0.1.3680043.8.498.96796299947034725933575150238042301500","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44629238055006732101555855930071163428","series":"1.2.826.0.1.3680043.8.498.10451520973595108643625974789544423445","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.44670636848627131733157841382853275817","series":"1.2.826.0.1.3680043.8.498.43537681312703475981705578536794755227","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44758766790156297849747618145458535803","series":"1.2.826.0.1.3680043.8.498.11797722988381721920594017550304854467","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44863396093454506630473164990623458425","series":"1.2.826.0.1.3680043.8.498.86427873029974701792081548023759341941","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44879085997210091159117906674172231025","series":"1.2.826.0.1.3680043.8.498.77555777292200845911748471955545863638","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44941174190375343987327566550204051010","series":"1.2.826.0.1.3680043.8.498.10451735019721570876749870234530777264","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45033326048521572717771708818514772861","series":"1.2.826.0.1.3680043.8.498.44176471797293319902682057832146935480","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45082393649917309927267478072569185694","series":"1.2.826.0.1.3680043.8.498.29965917981323773789333926940148056458","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45170517567841608067443743961197123093","series":"1.2.826.0.1.3680043.8.498.49993499256364478139145323859485740072","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45273393220765922863913956738614783632","series":"1.2.826.0.1.3680043.8.498.86443066165180412900568910074079520519","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45304570280714567513866386380356988876","series":"1.2.826.0.1.3680043.8.498.67229524640924916887788375138057170938","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45332185607602495786497298658937575959","series":"1.2.826.0.1.3680043.8.498.10526170811675804310874509841822790037","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45368532788186860828867663606651818835","series":"1.2.826.0.1.3680043.8.498.55112280627913170504997454188727926291","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45406212529450909512649737062964080190","series":"1.2.826.0.1.3680043.8.498.41890838122407165528009939002970963605","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45462389009834095091353507592651342877","series":"1.2.826.0.1.3680043.8.498.67089387632615979198523119438719435543","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45504348071514081636909771148066956410","series":"1.2.826.0.1.3680043.8.498.94739157380274463386905665270117561267","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45630051398709774279703407401383545642","series":"1.2.826.0.1.3680043.8.498.72305863333073517062013130091214126775","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45678969592219293243366784683191701674","series":"1.2.826.0.1.3680043.8.498.12884067990708982095895775451184421094","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45811964949115542204306831813385709206","series":"1.2.826.0.1.3680043.8.498.22967718706862862898351940166099484446","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45865541491386404818767748310365919685","series":"1.2.826.0.1.3680043.8.498.11134965108851768144115568844311659363","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45968957856865486437256472516713628614","series":"1.2.826.0.1.3680043.8.498.89024802095301090748277602382096429521","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46016776602328665980436110567854612595","series":"1.2.826.0.1.3680043.8.498.43573827710098200422611514599732533071","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46052469424267861803032899866428547156","series":"1.2.826.0.1.3680043.8.498.56724155173531059890315200187226158920","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46172451572836863687073454997158931662","series":"1.2.826.0.1.3680043.8.498.15047501682855032768298074497690380853","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46259494895672269475940484605821837768","series":"1.2.826.0.1.3680043.8.498.54832276293459416264596647665819051162","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46297226877524853395345229615300658523","series":"1.2.826.0.1.3680043.8.498.63247075494216368292403383909243242723","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46309540159823359540686782651264853894","series":"1.2.826.0.1.3680043.8.498.46624709699260789563256899120911804532","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46427879286435198611282055405541286766","series":"1.2.826.0.1.3680043.8.498.11947252699812647640195945492252611090","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46491051379327985199309365945879841064","series":"1.2.826.0.1.3680043.8.498.10962015803858024802094074984979633504","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46700895961751940291334058396794206479","series":"1.2.826.0.1.3680043.8.498.10276532137994952414963775390103924461","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.46715670142711601593792518196789347552","series":"1.2.826.0.1.3680043.8.498.10966783420002397464207033024806577117","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46776367027887983105817631582518625639","series":"1.2.826.0.1.3680043.8.498.11207432943155587917380714631080130468","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46842734152337184482152392783260161845","series":"1.2.826.0.1.3680043.8.498.49467686100278346446287480550130201283","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46893956584698923808118623450380169212","series":"1.2.826.0.1.3680043.8.498.15073077928725208119297629911284608296","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46925959240037679795735705370016021232","series":"1.2.826.0.1.3680043.8.498.58103528662223727346669910246362454587","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47061252152535181371772020997295000029","series":"1.2.826.0.1.3680043.8.498.12975581483755176772277067305073150152","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47080537724499166054609508776778189902","series":"1.2.826.0.1.3680043.8.498.51833828646584338541340942010906127834","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47239511009810375185594384849101241004","series":"1.2.826.0.1.3680043.8.498.13166581922023988521364529469581574220","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47267174205472073784818634988669693642","series":"1.2.826.0.1.3680043.8.498.10714321484268227411520497958784943414","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47339093185444044578247608022398494955","series":"1.2.826.0.1.3680043.8.498.12417960061246836868416703967581546207","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47449893776425045020340604662487159684","series":"1.2.826.0.1.3680043.8.498.22596217280330389597914181894255301241","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47531034194300620953276025401018051246","series":"1.2.826.0.1.3680043.8.498.29861532428881128890417097061835113448","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47655269404919497034805209368250028981","series":"1.2.826.0.1.3680043.8.498.12288712857960450174387026653622752801","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47678121014268671436574141288012829166","series":"1.2.826.0.1.3680043.8.498.25326084156910558360537652538920970154","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47784065888386216103165400551121771174","series":"1.2.826.0.1.3680043.8.498.49524895932570954854395033043587071028","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47870074144891988520855535250618228390","series":"1.2.826.0.1.3680043.8.498.11067090872887218615761829181167425496","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47925688677871325890380893763458019552","series":"1.2.826.0.1.3680043.8.498.50184111248572597456516210989163877486","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48068031587855513429324313927339963541","series":"1.2.826.0.1.3680043.8.498.10314913733045728769740114321737115704","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48160569779413970442472500116176876660","series":"1.2.826.0.1.3680043.8.498.52842432862031892477596696350328156269","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48367523758673249284754003765836877239","series":"1.2.826.0.1.3680043.8.498.52941386328195085285817335332472959379","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48417903971145834309474934717837250776","series":"1.2.826.0.1.3680043.8.498.31033146119467896135711080419512703588","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48520238688122533624681796433258211236","series":"1.2.826.0.1.3680043.8.498.59428768609930505492586614372676391417","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48531746264504523370547117933271161633","series":"1.2.826.0.1.3680043.8.498.54768629829189679029279045086896157944","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48697800450463192944610588533290320545","series":"1.2.826.0.1.3680043.8.498.39491015429862297181038425739357226230","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48771187298294322173131630438467163244","series":"1.2.826.0.1.3680043.8.498.92586527665650634882527212589494799409","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48818931767017230678539144135307491171","series":"1.2.826.0.1.3680043.8.498.15047894924743222704920785766795746728","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48855562507887692822284056171390459619","series":"1.2.826.0.1.3680043.8.498.65468533689415717151500246891203291119","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48933020429865163393792899740045803587","series":"1.2.826.0.1.3680043.8.498.88901493655451865879965778758265177317","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48977699132767849812826572970218325717","series":"1.2.826.0.1.3680043.8.498.12004976236821487486328249981202969009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49048739684287632681242054218807611262","series":"1.2.826.0.1.3680043.8.498.13370537507100956681944133980755399343","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49094276335794811195117175603556929507","series":"1.2.826.0.1.3680043.8.498.11325743321113612770057272713539646005","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49186859342644783675130322288585477727","series":"1.2.826.0.1.3680043.8.498.64336887286026816100748763171355654787","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49347024975371539869871751159907613638","series":"1.2.826.0.1.3680043.8.498.52153675959243105071842195278008143575","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49394404966674633152360206835416205330","series":"1.2.826.0.1.3680043.8.498.94203631381709152674308128677694860404","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49443962795025989994487968185070137217","series":"1.2.826.0.1.3680043.8.498.23336662828457726199693407574107884693","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49495644703047055149965226028097259207","series":"1.2.826.0.1.3680043.8.498.93621927139303104926388272885521866710","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49583348226932515919984811293816232513","series":"1.2.826.0.1.3680043.8.498.63063926099346432639850495874833510907","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49660659440268002826070581802625582554","series":"1.2.826.0.1.3680043.8.498.39372420185810450287053917149438247171","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49703744074809050221928706774732375079","series":"1.2.826.0.1.3680043.8.498.81860087379169190640618842497930668077","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49847104080351361589960954776757484668","series":"1.2.826.0.1.3680043.8.498.32736232611459023563207529313478386206","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49866252311077397175807461727885923657","series":"1.2.826.0.1.3680043.8.498.12759835862426511299400462393950814166","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49884243466758407760126403944842240776","series":"1.2.826.0.1.3680043.8.498.56958173450301824914389366788316639339","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49942556484410764813681696157580687854","series":"1.2.826.0.1.3680043.8.498.13327706287090329891677776868875464815","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.50051845967696843074391018845332480653","series":"1.2.826.0.1.3680043.8.498.11329535386073203259378044302144210248","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50089015780087008332657793589955395952","series":"1.2.826.0.1.3680043.8.498.94583717349695127458677581497114348955","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50109255875403812845391217314726500341","series":"1.2.826.0.1.3680043.8.498.13094088499754703510330438282212546206","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50183004658202912273462671442829139110","series":"1.2.826.0.1.3680043.8.498.37475115608375846650720167892975384807","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50248717336781045512984550029203786086","series":"1.2.826.0.1.3680043.8.498.11490536291115990448352644588327589484","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50261788831539215224639813005933285052","series":"1.2.826.0.1.3680043.8.498.32069151288254303813745246581743069688","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50342726517205151681842826322659275894","series":"1.2.826.0.1.3680043.8.498.12739205221694360905025522451380401430","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50424421287869498746932271082980000521","series":"1.2.826.0.1.3680043.8.498.91626327872531742978210815590550360442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50501020658943435430736717390350999395","series":"1.2.826.0.1.3680043.8.498.12632123688185685663218096317942891326","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50571692547974449396635351108344029536","series":"1.2.826.0.1.3680043.8.498.52850207749530461544658496894193397506","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50666048236278074641860077012751071758","series":"1.2.826.0.1.3680043.8.498.62575885736538635798097195127880425102","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50687743970686088872471288288960579306","series":"1.2.826.0.1.3680043.8.498.90909907572078717983627311856735013707","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50694733929204626284366818623993960684","series":"1.2.826.0.1.3680043.8.498.77485888020163831709026811358580133463","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50777252749942894325007417893973309310","series":"1.2.826.0.1.3680043.8.498.59214942495588901180505286008102604766","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50813123035221058777203670043892828582","series":"1.2.826.0.1.3680043.8.498.50844537972898509706433037649852133053","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50823279853002320074080586693604976784","series":"1.2.826.0.1.3680043.8.498.45475965718111597143938147252339646419","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50880891504121357724276091096406248845","series":"1.2.826.0.1.3680043.8.498.42523798858933003281890810973725343855","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50935247372419775856440294835587827419","series":"1.2.826.0.1.3680043.8.498.10856697109921479776655562906712725198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51006813205605862503064447831381009425","series":"1.2.826.0.1.3680043.8.498.72587115870847425960513987452779931152","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51126807380879212261057967703844943102","series":"1.2.826.0.1.3680043.8.498.65549244517546642058870881972313661789","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51179760508587500025973916664101120069","series":"1.2.826.0.1.3680043.8.498.10897811642720173672710575398317294058","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51297777726540529860433097009654453742","series":"1.2.826.0.1.3680043.8.498.22092265626493596927174038608858796130","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51426696939581135481763379246368970360","series":"1.2.826.0.1.3680043.8.498.53607220529281949797362954725791401391","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51469564445249084310991187819410005207","series":"1.2.826.0.1.3680043.8.498.67397065289716525357184800607142103040","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.51581188134109925758379838961317006116","series":"1.2.826.0.1.3680043.8.498.94316589287722797157628530107952973877","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51684117251092766010524179062007524257","series":"1.2.826.0.1.3680043.8.498.65921469492228813546390647050426064428","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51768733026033198941231586462443503827","series":"1.2.826.0.1.3680043.8.498.38683524561100870856992131429954837270","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51809092313496476873962641559638828608","series":"1.2.826.0.1.3680043.8.498.10633884126575655603941921289603932446","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51828180600648126726057100018010765613","series":"1.2.826.0.1.3680043.8.498.12132358451987176221771753637599498023","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51867133375333949773527521703402111601","series":"1.2.826.0.1.3680043.8.498.76443636990399817146390239281171710656","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51910852739185117358682097116614556910","series":"1.2.826.0.1.3680043.8.498.46364007321003496788733093171197863059","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51981019698056120061499970572334688555","series":"1.2.826.0.1.3680043.8.498.12062135721467100389660514765250909582","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52010165788889734743850946282043771445","series":"1.2.826.0.1.3680043.8.498.10403040086911086184967378140854174658","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52059522418267239652374290810994711032","series":"1.2.826.0.1.3680043.8.498.72336706585970309490959982447673309241","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52098249444968803324423421624756966294","series":"1.2.826.0.1.3680043.8.498.11978173470409208127514384487455889131","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52138967568492774495485454357106269286","series":"1.2.826.0.1.3680043.8.498.35392145422463752364359981419473750124","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52204390785696944340286127542242341202","series":"1.2.826.0.1.3680043.8.498.90364979762221424237047277498562258217","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52347547859131274836026480407333415929","series":"1.2.826.0.1.3680043.8.498.10708518326502733190262614248000307580","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52435631723314823200374985065756182593","series":"1.2.826.0.1.3680043.8.498.99805999968792582215708609260212940965","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52456272643822591755257802747431239152","series":"1.2.826.0.1.3680043.8.498.83323328564104911658335475989003883277","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52520288400853832269613213204540385687","series":"1.2.826.0.1.3680043.8.498.10776490881942993378873295355361089963","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52599302991259061057950834864441634066","series":"1.2.826.0.1.3680043.8.498.39469884521673892999946967839772615207","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52745530063404662086111315503240395535","series":"1.2.826.0.1.3680043.8.498.53300183656931048211533423594173804877","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52846023545870016653608198538437574931","series":"1.2.826.0.1.3680043.8.498.26808211961467025432293359474413259912","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52849154011665864497632297320961015159","series":"1.2.826.0.1.3680043.8.498.28884885696929789485707724476936615289","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53025489296610571044246428509154950855","series":"1.2.826.0.1.3680043.8.498.20078593552822643792834799869525444470","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53078461307849490495536549806091931824","series":"1.2.826.0.1.3680043.8.498.39725013172244891927784745599040486023","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53114802640679819441521055029024037716","series":"1.2.826.0.1.3680043.8.498.95420191451719757319964552109997514182","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53137953094305471774552883960361922832","series":"1.2.826.0.1.3680043.8.498.79107412395832978543532528279116678714","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53174810138171412044410050984734412146","series":"1.2.826.0.1.3680043.8.498.71468315597313492575069840434776999291","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53262958748075737242820629127001253146","series":"1.2.826.0.1.3680043.8.498.10050717991446997908905340020744994412","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53337684515627962545396495541158882262","series":"1.2.826.0.1.3680043.8.498.14123191288058159040937732864909053406","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.53355961816756477601843499985089590263","series":"1.2.826.0.1.3680043.8.498.78323054368542979585590974582740305248","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53393741562201846141435914434030220216","series":"1.2.826.0.1.3680043.8.498.12483642605054559485072628559651979699","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53490080963614272234072703760363052476","series":"1.2.826.0.1.3680043.8.498.18654559453568022065201362295054382181","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53579416035647672067965892020505016454","series":"1.2.826.0.1.3680043.8.498.49422652592385966719705189641408974106","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53649895712172408476570770924813501118","series":"1.2.826.0.1.3680043.8.498.13153176526338194713356610523565156051","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53792159922324410857755054939229546555","series":"1.2.826.0.1.3680043.8.498.30949861515304839918995580690338838155","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53884715696186989798696544661396487883","series":"1.2.826.0.1.3680043.8.498.14921322393910937247312585249703109618","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53963759852017695930808272438606800422","series":"1.2.826.0.1.3680043.8.498.43828251525023455544875841298170204534","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54030363830629950591221870797594277092","series":"1.2.826.0.1.3680043.8.498.21635427746580086018655110867497693935","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54099199514076260519579096546600505763","series":"1.2.826.0.1.3680043.8.498.63229790897893044249093341382414944224","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54132814503202891434380002378663763655","series":"1.2.826.0.1.3680043.8.498.17128387370432881739420706316221597414","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54142170846176833568961426171739782214","series":"1.2.826.0.1.3680043.8.498.10436180569919265505217102173661907226","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54270039578867689669466339980682684126","series":"1.2.826.0.1.3680043.8.498.13193761973759463498302688820761300319","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54306467780023777062920737767581126383","series":"1.2.826.0.1.3680043.8.498.46961299081206110017595760703229529532","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54374735423703474541190464716308417985","series":"1.2.826.0.1.3680043.8.498.10375083427323469200902015746436914703","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54427039222108917725902677899081931510","series":"1.2.826.0.1.3680043.8.498.75135563193865292738707663621434338240","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54482848766921329667469114743547203929","series":"1.2.826.0.1.3680043.8.498.27959764558557496493896092754223740260","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54539703415639765348239790227512479583","series":"1.2.826.0.1.3680043.8.498.43080382950641429877727618567314149107","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54619696938940618755334224550745735815","series":"1.2.826.0.1.3680043.8.498.11708370949774373476519107193591427712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54669065204808162914386927559244866557","series":"1.2.826.0.1.3680043.8.498.66021965095691735877088024010462110349","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54699196872730350880681018674893216245","series":"1.2.826.0.1.3680043.8.498.13049631940438706818080314721551552248","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54755249425443469939742671497042113665","series":"1.2.826.0.1.3680043.8.498.59846291883505263555598454701461620766","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54792838438079861454532183998999249213","series":"1.2.826.0.1.3680043.8.498.53782289927457682890211035001030975281","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450","series":"1.2.826.0.1.3680043.8.498.80377175672460372451948006708970551356","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55022631713888448062201243978634363406","series":"1.2.826.0.1.3680043.8.498.10047519984694514846911312497416515211","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55065105486648830377200570585789726800","series":"1.2.826.0.1.3680043.8.498.81346662939160329156906033503019675356","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55183153701332792661581741475876132309","series":"1.2.826.0.1.3680043.8.498.26884178716447119463235249681223419755","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55300053252365037422745072128951402593","series":"1.2.826.0.1.3680043.8.498.12307369211638356472792314278156355146","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55344353927048912237233441917779875587","series":"1.2.826.0.1.3680043.8.498.89412143433105717158564074073359204154","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55397602116058381866334455456729470963","series":"1.2.826.0.1.3680043.8.498.96221785098194152692388582074336677957","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55583505147281684100202399148099429721","series":"1.2.826.0.1.3680043.8.498.25906427507784198561455755480089770041","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55653770127824574123236841437998533498","series":"1.2.826.0.1.3680043.8.498.91333643382892182690258494899824414661","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55700532609077428304194916132759352518","series":"1.2.826.0.1.3680043.8.498.11615634807475575753590476939580398294","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55771181380353653474896884649540028856","series":"1.2.826.0.1.3680043.8.498.40440268197391034665758025447681980596","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55850385284038070407841702672827533728","series":"1.2.826.0.1.3680043.8.498.11800837804482391545403582037841630080","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55861544093014499478739864837675161978","series":"1.2.826.0.1.3680043.8.498.47073609739850374988763821193411639243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55930031395651221279170435293900769500","series":"1.2.826.0.1.3680043.8.498.11819015429467282868987962386751164594","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55971241672517044174019654998425819738","series":"1.2.826.0.1.3680043.8.498.12932223535328246665431462492864827152","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56045687004558569005240775213510192805","series":"1.2.826.0.1.3680043.8.498.10409164003604402487727621523310819731","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.56061505201159269536816810086342635391","series":"1.2.826.0.1.3680043.8.498.56248775220010251451334692923881943226","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56146582420561901778012523946835033628","series":"1.2.826.0.1.3680043.8.498.57860423285952094209241463687946583492","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56188634971310070481285525830793231849","series":"1.2.826.0.1.3680043.8.498.10557759512647022814414471488791093637","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56266280744044680626884459696038109616","series":"1.2.826.0.1.3680043.8.498.65823399854402610888254161321388024107","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56385473079196655120993854770623075553","series":"1.2.826.0.1.3680043.8.498.41632060043217454033131357502208977891","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56497268548664917337130798701015887461","series":"1.2.826.0.1.3680043.8.498.30682963291486954938728585789048812347","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363","series":"1.2.826.0.1.3680043.8.498.10759295590848082080643815881145133309","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56575942633137496743464648679775042979","series":"1.2.826.0.1.3680043.8.498.17226931316477403275233775597745616135","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56686055146646210144964207684044815006","series":"1.2.826.0.1.3680043.8.498.10727378916523855718297602296628076337","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56762592839620371588520593573846414860","series":"1.2.826.0.1.3680043.8.498.64733738422280080444024435059775144353","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56796622077427019119540733986070534126","series":"1.2.826.0.1.3680043.8.498.15819786718901945127437930378042676491","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56839374698050163738039760889156066445","series":"1.2.826.0.1.3680043.8.498.31158399704298896601731353015262078879","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56882015310689536014061315110969323429","series":"1.2.826.0.1.3680043.8.498.28924059465467669170707241474212125592","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56915079210789596803972894081756325026","series":"1.2.826.0.1.3680043.8.498.35103288201101174528179528331850897537","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56945039212126810522718038626526579591","series":"1.2.826.0.1.3680043.8.498.11088402108675184130203993110782166580","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56991590802231436015843619633497607680","series":"1.2.826.0.1.3680043.8.498.29430991290614692861088244400199589725","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57064618560116981041722203028857007300","series":"1.2.826.0.1.3680043.8.498.11807352063964066453507732088856171137","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57084445429567494093960261980941917499","series":"1.2.826.0.1.3680043.8.498.95833655757099904116388739000851388346","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57239450717434070615919278253974598533","series":"1.2.826.0.1.3680043.8.498.11358651886695157613605922265761344557","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57342523828166309133982222435655662667","series":"1.2.826.0.1.3680043.8.498.11656694227605366396371234438259038118","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57491375928585600499323136123872536597","series":"1.2.826.0.1.3680043.8.498.10152926793192573096907395500887922500","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57703608122587911055931213603182679454","series":"1.2.826.0.1.3680043.8.498.66460221588329450026320913601580753498","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57775389384191131322519457853517177210","series":"1.2.826.0.1.3680043.8.498.16714540967227270186782766630077513000","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57802887045514681373294932749517914959","series":"1.2.826.0.1.3680043.8.498.21449608028323326592660396987442765079","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57836524675365700674244122764523161699","series":"1.2.826.0.1.3680043.8.498.25837117313310872675556128111534360313","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57882532838957577543516026386452055977","series":"1.2.826.0.1.3680043.8.498.10236092938030295755571901205523736606","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57951268100539350409054562078476694936","series":"1.2.826.0.1.3680043.8.498.11716042125602402039289541056322623732","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58051085126748057986491277367254533832","series":"1.2.826.0.1.3680043.8.498.49821306845640700000927450678919476236","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58250779958866595505585700517927225230","series":"1.2.826.0.1.3680043.8.498.18888266971492661612611722897465574140","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58280111882249056715877316026191844548","series":"1.2.826.0.1.3680043.8.498.11514628179713640198471497644264977383","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58346459085553680879151588730895136332","series":"1.2.826.0.1.3680043.8.498.11064047332639925426490370226250982512","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58404766256608483217059706069436050741","series":"1.2.826.0.1.3680043.8.498.24652291177505920810085900658330131692","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58475208045272914292874190001248518687","series":"1.2.826.0.1.3680043.8.498.36254084500153720340521322221492002103","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58482779112389764165599701941310919660","series":"1.2.826.0.1.3680043.8.498.79964789328831028945147728924181671684","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58585441284202143792468863402617212167","series":"1.2.826.0.1.3680043.8.498.17350738305519323448843001976592557882","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58626635384305289939700801774393536415","series":"1.2.826.0.1.3680043.8.498.76657371903103751887730549652782362032","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58685941646495891824559391997783302656","series":"1.2.826.0.1.3680043.8.498.46506504890013522948759620373158669941","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58728013757221328590898166513995249334","series":"1.2.826.0.1.3680043.8.498.25564165072182083666174403272943566284","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58768083117076186082335984587845964130","series":"1.2.826.0.1.3680043.8.498.64223842933546631043333249386542681527","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58811528407321581206563412079643094538","series":"1.2.826.0.1.3680043.8.498.31211599999554435527206811040769801422","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58857477075095368361468658035369975572","series":"1.2.826.0.1.3680043.8.498.32988246184613967752026556284375051359","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58925843611945635442423587804169242644","series":"1.2.826.0.1.3680043.8.498.23985899728554599447797682838060574714","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59021913092219789232890327780064145948","series":"1.2.826.0.1.3680043.8.498.10033233627873733099696523751624550439","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59073122184069676919857712290159511305","series":"1.2.826.0.1.3680043.8.498.20434918563195649613860001151906875954","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59126324546257448818397467078224944768","series":"1.2.826.0.1.3680043.8.498.76454022460265932700789766974049210896","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59262240088132401543519466539885561578","series":"1.2.826.0.1.3680043.8.498.67228262938354756991375678714858230177","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59394856717216352192037635065925607933","series":"1.2.826.0.1.3680043.8.498.18775492952396399000409497075257066451","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59449702487184611096113620550399616423","series":"1.2.826.0.1.3680043.8.498.75339102783200406483599868336206590192","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59476657510212258499047356518058209126","series":"1.2.826.0.1.3680043.8.498.44295136673260745222663041783084644605","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59527896591755229337596505685198957637","series":"1.2.826.0.1.3680043.8.498.48819746971436440712877185218223909649","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59680220055968160813801840287391339651","series":"1.2.826.0.1.3680043.8.498.64271145049424146898265533593143173810","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59758232061190668535098343218115599228","series":"1.2.826.0.1.3680043.8.498.72743288703313087533508916772576678970","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59835677513721120155185666132462195647","series":"1.2.826.0.1.3680043.8.498.37961130981838818662447683161502519019","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59923507410103964137819195926587517759","series":"1.2.826.0.1.3680043.8.498.27548247957631392800435544161038801768","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59938475193950583662125713924156951751","series":"1.2.826.0.1.3680043.8.498.10438582831354769122234870417485433370","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59954207362160100069708115719895503197","series":"1.2.826.0.1.3680043.8.498.86787629181412410654632640516065753783","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60064080350033586611530748685370040101","series":"1.2.826.0.1.3680043.8.498.12428803357983578145196137818246710592","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60094182309575937331152278560900767484","series":"1.2.826.0.1.3680043.8.498.19573214455858287496686432524663492439","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60227430483444032531479844365716485643","series":"1.2.826.0.1.3680043.8.498.53104905499400183040718216611141670466","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60259828181878278590614805828898992241","series":"1.2.826.0.1.3680043.8.498.28816342203813782971581844670544918325","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60394858470917657607846893765818799720","series":"1.2.826.0.1.3680043.8.498.48382973425942981863755088867299295682","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60439571175176992075370053747045234796","series":"1.2.826.0.1.3680043.8.498.68606088686697103746337721971050816098","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60465293103574559546982440125572490190","series":"1.2.826.0.1.3680043.8.498.44108499896957137171051162620897793905","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.60477027832057450004004942199870813886","series":"1.2.826.0.1.3680043.8.498.90080900268630947126296493478778326106","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60548477793215320533051194155539307724","series":"1.2.826.0.1.3680043.8.498.63058901296363262835605473389537022762","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60614975796208617794180432816390721331","series":"1.2.826.0.1.3680043.8.498.79262843269575806248945073880364528049","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60661676782543347731000874708337132276","series":"1.2.826.0.1.3680043.8.498.41894234155764693481286410628022204713","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60840220723824894619895264673401129352","series":"1.2.826.0.1.3680043.8.498.60318451824687242374703712652302187524","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60931716414402855657852057585917605786","series":"1.2.826.0.1.3680043.8.498.14669497474227751923553287579774342145","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61043690805522768840878574610198420558","series":"1.2.826.0.1.3680043.8.498.10416675879245714289377813576391375285","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61140619849833544411463750382539885141","series":"1.2.826.0.1.3680043.8.498.57969852562476735416267875223773736647","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61148667330113929729568621580194520169","series":"1.2.826.0.1.3680043.8.498.35928787060123399320039487932772127782","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61227644940421894951950605627016308160","series":"1.2.826.0.1.3680043.8.498.11855225305003373570139538149523199206","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61333047594780344399535845209036589563","series":"1.2.826.0.1.3680043.8.498.13992713271338888572398305480243453245","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.61435096069921474761812731150911059491","series":"1.2.826.0.1.3680043.8.498.76174831307323277603419929538072255200","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61490475141024845836646569993549106628","series":"1.2.826.0.1.3680043.8.498.13201401834416240781255539328821254442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61554384977916135091630433680780524386","series":"1.2.826.0.1.3680043.8.498.10567070191655458179141017617245074818","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61581513083114643407028382481113097015","series":"1.2.826.0.1.3680043.8.498.67060949063748878094669404862574047066","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61715630583919056026446344483871058799","series":"1.2.826.0.1.3680043.8.498.11531355268162358798591343963628948954","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61864939963243675140060416958055914981","series":"1.2.826.0.1.3680043.8.498.11857725286198246889819996035141410586","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61956863718716027369068406125965902569","series":"1.2.826.0.1.3680043.8.498.62376045124303878761282277574479567179","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62025036385410820041698409408092924893","series":"1.2.826.0.1.3680043.8.498.37978923216521237012220597401468407545","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62154969360158054310282857093665911261","series":"1.2.826.0.1.3680043.8.498.88181180570388260086465393133569186473","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62205799963299902160268565319924399775","series":"1.2.826.0.1.3680043.8.498.58814440948401346682543681286596944306","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62218684089117864632544493263539598403","series":"1.2.826.0.1.3680043.8.498.29943074035297606205411793935681392842","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62329283530383340787096585492217632179","series":"1.2.826.0.1.3680043.8.498.91067726617483342696198688693545985329","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.62400371770962739002393364782172386752","series":"1.2.826.0.1.3680043.8.498.10820091427350505719536158686351199630","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62420632149863295140175740643640707423","series":"1.2.826.0.1.3680043.8.498.50358795766731888499738643681069958553","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236","series":"1.2.826.0.1.3680043.8.498.69443004644346250053604983692181738675","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62693151030819598514729637644804213983","series":"1.2.826.0.1.3680043.8.498.32842275987423162568824927135486587721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62776347582771469289738234607517546504","series":"1.2.826.0.1.3680043.8.498.85155113896060167383531866859338663790","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62830031603985608774418534497520663269","series":"1.2.826.0.1.3680043.8.498.15794212900797493301185895466158807561","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62928513795250694194589804784951257414","series":"1.2.826.0.1.3680043.8.498.53546505254333103996226354903373489914","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62983291359021407345388012674614537807","series":"1.2.826.0.1.3680043.8.498.82248045275201583468364274543365065102","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63068687840516335034631353955761036465","series":"1.2.826.0.1.3680043.8.498.74689085820343461972981681628031351335","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63096952391833769395204944538276970114","series":"1.2.826.0.1.3680043.8.498.42261065008542682457659336881629038829","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63151733210715530368323422018984705316","series":"1.2.826.0.1.3680043.8.498.69920525552531989902121922993928908148","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63197503121546251937003511751049515360","series":"1.2.826.0.1.3680043.8.498.10514085544877598891685399569326901825","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63223220827342079068491460847713457238","series":"1.2.826.0.1.3680043.8.498.89940503076232730597865598406874149424","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63242029756172180508167933519465902827","series":"1.2.826.0.1.3680043.8.498.10843770122530101656157296365427782193","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63320397279666280420359587173167195895","series":"1.2.826.0.1.3680043.8.498.12926111901776247979669304033144120057","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63377901538320536934403969965624202519","series":"1.2.826.0.1.3680043.8.498.10945401932329783147606712514253414847","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63426442500798189991842988966227275166","series":"1.2.826.0.1.3680043.8.498.54435231701662767414172341059620987988","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63519003265636561965405570006747767668","series":"1.2.826.0.1.3680043.8.498.87160307006814717685275493413825837082","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63521726454879203860288713455843702564","series":"1.2.826.0.1.3680043.8.498.11402199844627729464380956162487413348","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63550258873504696667112631869753870779","series":"1.2.826.0.1.3680043.8.498.11003454443317349041966651408688306880","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63565089119380386830816381560603756258","series":"1.2.826.0.1.3680043.8.498.79995054666505401781199000548398178497","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63673733836218886397484265807806548473","series":"1.2.826.0.1.3680043.8.498.10103829221207766316467387736296654671","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63797588614544518196205411126336672334","series":"1.2.826.0.1.3680043.8.498.77186234555140108296222942762153561118","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63884838982826315257233262184190435906","series":"1.2.826.0.1.3680043.8.498.47464808993720632953507743637146344320","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64022112049842890696609934970222024490","series":"1.2.826.0.1.3680043.8.498.70692094148559982491748201044251703569","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64034149345689391514924451724232196567","series":"1.2.826.0.1.3680043.8.498.33420519692968549181409920530080655943","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64109231850522212435708176045583779359","series":"1.2.826.0.1.3680043.8.498.30453616258766845693612493955308604015","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64172939203926034064618447795131041647","series":"1.2.826.0.1.3680043.8.498.94284373427192175022504978348793882084","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64283618749539713756220539395944025878","series":"1.2.826.0.1.3680043.8.498.11047432462337301703400216886201611671","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64381606512330581256470279413868645789","series":"1.2.826.0.1.3680043.8.498.68537507668813346698715687535739864292","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910","series":"1.2.826.0.1.3680043.8.498.27195687106566182788779817031043206458","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64443149121726771900576109439951701105","series":"1.2.826.0.1.3680043.8.498.12570387373207814940098438397224721687","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64526757783166918696475501252110447010","series":"1.2.826.0.1.3680043.8.498.11694149834541607458384610387335824604","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64590392444259942263746282444188079724","series":"1.2.826.0.1.3680043.8.498.97176687980761205328653220930966315105","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64652085172505930317660036583152917472","series":"1.2.826.0.1.3680043.8.498.10898338682561376234969171143836341293","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039","series":"1.2.826.0.1.3680043.8.498.51087602398228796162207544837869472387","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64865300853258390816475799272221318005","series":"1.2.826.0.1.3680043.8.498.42994676746980871405084648595024651589","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64946212153370539647301308676689632525","series":"1.2.826.0.1.3680043.8.498.68480754941571710710249766540336360351","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65077085081642080527129904737510276643","series":"1.2.826.0.1.3680043.8.498.94411135832150680324459059775720885068","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65123055741109319778634264103224173252","series":"1.2.826.0.1.3680043.8.498.87718311149592737135750559019329224643","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65187850119346448625449915281726338455","series":"1.2.826.0.1.3680043.8.498.25642017242043975468130908980192060051","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65224808465738083245406235000047993647","series":"1.2.826.0.1.3680043.8.498.93362838673513213214444556940881051615","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65322445533737673246312151513913460816","series":"1.2.826.0.1.3680043.8.498.11829124784828638733974609543552088411","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65403984120528711726317048984986599537","series":"1.2.826.0.1.3680043.8.498.23434287304727560740322237013451173706","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65524487168129064460609645228098420719","series":"1.2.826.0.1.3680043.8.498.85013237143397356277274601596177126633","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65543472488214519069910341591668399186","series":"1.2.826.0.1.3680043.8.498.12146481844904820936876316123478837079","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65607807464576681752987746458209389847","series":"1.2.826.0.1.3680043.8.498.12072748722758310488145211632755788874","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.65703265163052901613211590083675186626","series":"1.2.826.0.1.3680043.8.498.48482618291259542087011676396650838981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65756786850574169314934708606155033765","series":"1.2.826.0.1.3680043.8.498.94465854040333173010536307054579808478","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65783500506568702195262661666345236666","series":"1.2.826.0.1.3680043.8.498.65259994782268037285115841363755395706","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65809115193576546782858254430325338674","series":"1.2.826.0.1.3680043.8.498.29416637354232193708721440341771467650","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65855247249608663104822603172047250057","series":"1.2.826.0.1.3680043.8.498.10434437183731503926318092924693084202","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65939700606136711089565313916789162980","series":"1.2.826.0.1.3680043.8.498.57443517056101635822812436566437597699","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65977711270942843864266358352913175672","series":"1.2.826.0.1.3680043.8.498.11845701396401829270308980769058514482","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66031142975529812579758652269360365398","series":"1.2.826.0.1.3680043.8.498.87284649391386701446146728309673267852","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66083855405250624607521810369199218887","series":"1.2.826.0.1.3680043.8.498.69262929617177288921994441448260522400","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66199248385068685539872840826351152652","series":"1.2.826.0.1.3680043.8.498.12622946053385835557757486294242164712","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66264392187277583747691643146920431469","series":"1.2.826.0.1.3680043.8.498.62955814298634355217806239702930673825","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66305771884507759587182077473863996177","series":"1.2.826.0.1.3680043.8.498.12031989382361785419422348400784064963","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66327751482076404294387513492434577338","series":"1.2.826.0.1.3680043.8.498.91879255522306531038106524135395731573","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66364462115470823900176770413527217503","series":"1.2.826.0.1.3680043.8.498.11109102802066388476251810890449005609","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66489634638847834550694493585147280754","series":"1.2.826.0.1.3680043.8.498.91642555162994513516171608364390786970","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66596090141798159058634289358454183351","series":"1.2.826.0.1.3680043.8.498.48427518176664000264788648397666668523","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66694519343321893230666089565893349853","series":"1.2.826.0.1.3680043.8.498.51753734752980278586507411755708872694","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66805139646641294853509958688742554525","series":"1.2.826.0.1.3680043.8.498.40724760637866088010591755643237868000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66960891947495601371173426277430301127","series":"1.2.826.0.1.3680043.8.498.52893368581115562905193669868263774425","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66993076561387452114335652021593805727","series":"1.2.826.0.1.3680043.8.498.40719455560690551969644937543319761155","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67157948144868269956661697924341234924","series":"1.2.826.0.1.3680043.8.498.37731036038834573338866394194682561848","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774","series":"1.2.826.0.1.3680043.8.498.10345232795159127591934880165802825806","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67208608901061914528631080268819069203","series":"1.2.826.0.1.3680043.8.498.64115589406349254885377271313332221784","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67284681623851829916403311739987186159","series":"1.2.826.0.1.3680043.8.498.15423333056365846753634300181023910407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67397756161011684282778294948252031539","series":"1.2.826.0.1.3680043.8.498.10451211969702044259390020364364342586","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67456402757136973484427155324664966595","series":"1.2.826.0.1.3680043.8.498.52251973666622218390842154477477131526","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67498362978754615554767658151787188674","series":"1.2.826.0.1.3680043.8.498.51189152465788945842610303295331940788","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67550262192397377344258276170992188234","series":"1.2.826.0.1.3680043.8.498.11585053900441817668830529397638832855","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67620512607520186685421323991166816451","series":"1.2.826.0.1.3680043.8.498.99573350535331921752113801889743447720","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67690090498828405347380696553164435363","series":"1.2.826.0.1.3680043.8.498.86643845995139184994981624575208656172","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67716063997837501476742746033172868748","series":"1.2.826.0.1.3680043.8.498.61576452162813973122318055257971363756","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67772676243889350087068112324015156848","series":"1.2.826.0.1.3680043.8.498.96854963247211404531720069553535009538","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67788047649994083591931245362169694306","series":"1.2.826.0.1.3680043.8.498.53669264156952182878699812022686951163","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67875980316053973061252844698884229168","series":"1.2.826.0.1.3680043.8.498.20465639743494776386775574219405346857","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67902658411525109462089381503197501728","series":"1.2.826.0.1.3680043.8.498.34156081006180887137660133923856861767","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67958551704692248657133951054036903879","series":"1.2.826.0.1.3680043.8.498.25024225309554510540136421990851408126","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67981539513814867568631217960473023750","series":"1.2.826.0.1.3680043.8.498.16343445549539481755519163448569026379","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68098431505945078581285587720599762771","series":"1.2.826.0.1.3680043.8.498.10837445897988155992456399495468543596","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68115220080063728900711167735074276147","series":"1.2.826.0.1.3680043.8.498.32680304026159112944971779892184823627","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68142054543837964766571710567214307830","series":"1.2.826.0.1.3680043.8.498.33773407446354478131794991906422255519","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68168197665046786209369894696801397179","series":"1.2.826.0.1.3680043.8.498.76957950699604799925267549177459439178","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68219942161081583831276769978472014317","series":"1.2.826.0.1.3680043.8.498.66482666282269169537517381155430118778","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68263925314265871578642679583159577645","series":"1.2.826.0.1.3680043.8.498.95651389826687504771229711336993791950","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68304853223764870957854101604223809218","series":"1.2.826.0.1.3680043.8.498.32414470756169453235461020424975559311","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68434116759606759586218589635360502038","series":"1.2.826.0.1.3680043.8.498.82995769603163917588244496670937982863","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68435927323644457286635872824563629385","series":"1.2.826.0.1.3680043.8.498.16334274526730435025418634487503830522","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68504220056564878621959394439671931747","series":"1.2.826.0.1.3680043.8.498.12125228732557727182389933108210414797","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68570907895010847368926943092385685462","series":"1.2.826.0.1.3680043.8.498.33797471634711441637257221387352927234","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68578659816221770558506780445608768600","series":"1.2.826.0.1.3680043.8.498.42143806600151682776736264021170372455","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68668620295929607459355154952035421306","series":"1.2.826.0.1.3680043.8.498.18671963842004643963211420397081168457","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68744926735490285461495285547312308522","series":"1.2.826.0.1.3680043.8.498.66155430914423055110590704094086036851","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68777783869253673804970857981124324335","series":"1.2.826.0.1.3680043.8.498.76412692497023093942213314965868219960","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68862684196934468400953020419415008157","series":"1.2.826.0.1.3680043.8.498.10033301774034198032502564349466321155","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68931510756844060683643298583132884960","series":"1.2.826.0.1.3680043.8.498.10946241467695410024086842966081374479","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68971603226593154050514800137550171537","series":"1.2.826.0.1.3680043.8.498.63390024700729293956624486251817556804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69077351372476086930472412978307038819","series":"1.2.826.0.1.3680043.8.498.32163952244093092703082130355714704791","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69156800026549034619316278954442147703","series":"1.2.826.0.1.3680043.8.498.11944652956070021879099813683049960063","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69236835611860340913671698884308946455","series":"1.2.826.0.1.3680043.8.498.44015617361920870298203544267594308573","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69311065409015951283858403030798356475","series":"1.2.826.0.1.3680043.8.498.29880260708218384726750211100161120881","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69397705155385306818511499752200941968","series":"1.2.826.0.1.3680043.8.498.12756028361832858364419017102299443510","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69429384137362531712760113705595820981","series":"1.2.826.0.1.3680043.8.498.56105677746793814371552845164099043764","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69514709065182531331987458626496800737","series":"1.2.826.0.1.3680043.8.498.51196376719464660915556743678991837745","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69596668032633976121623939445795750384","series":"1.2.826.0.1.3680043.8.498.47209235992417625554242714107231608812","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69654898267139720912940475772303404772","series":"1.2.826.0.1.3680043.8.498.12281331509312291715468758188201900965","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69717507929047897681405659943474469018","series":"1.2.826.0.1.3680043.8.498.11544062061044122388811776040245241698","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69730556680449158538173708009262014812","series":"1.2.826.0.1.3680043.8.498.14811929940989003099023863711252705513","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69846596572083911408018130524077120494","series":"1.2.826.0.1.3680043.8.498.10784545760206952221334296395258457482","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69930569170269472474738549886709596023","series":"1.2.826.0.1.3680043.8.498.61563599547411757103026635355865438507","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70125113032990356507120856030706104508","series":"1.2.826.0.1.3680043.8.498.12710488366033404406081667014401062590","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70212141059019236877414201247791694101","series":"1.2.826.0.1.3680043.8.498.45607135727376915117029353136433145395","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70291739279544255146458190540233023337","series":"1.2.826.0.1.3680043.8.498.76235348580562598791926309137195697094","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70358024114589255428192032298530338260","series":"1.2.826.0.1.3680043.8.498.64838829784712492833061908438024805164","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70427947434642957729711329062828233047","series":"1.2.826.0.1.3680043.8.498.48636903289741796663687200141056824751","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70464717849928808075676221329034929056","series":"1.2.826.0.1.3680043.8.498.12189295720389954806750738213385450625","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70507536279875475164400849670959748314","series":"1.2.826.0.1.3680043.8.498.30391891915219476985292348600311986237","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70532653867267627708152426358320452518","series":"1.2.826.0.1.3680043.8.498.10086196160360308376842789661115907599","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70691305452244378255646875206864340458","series":"1.2.826.0.1.3680043.8.498.89944518003787665217253365897105803779","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70796360440320543674384511859375295769","series":"1.2.826.0.1.3680043.8.498.35975602371341556569743208347285886136","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70865607713855272648070757909359669526","series":"1.2.826.0.1.3680043.8.498.47701523725362820458786016685176581565","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70880546533848103776823598726224805522","series":"1.2.826.0.1.3680043.8.498.12052767470888437825268866312288941607","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70913971933753517251340687745718346828","series":"1.2.826.0.1.3680043.8.498.14200636049596928073780885237155339405","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70931968429039283348328653687636803315","series":"1.2.826.0.1.3680043.8.498.41521349100082257616860527177785779317","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70955936086115471632446529979621948261","series":"1.2.826.0.1.3680043.8.498.42378855519197013348846492412668292684","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70984418257970716669463829979819981861","series":"1.2.826.0.1.3680043.8.498.11147238314616592227093300539889961895","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71063550263564056105844586167361750567","series":"1.2.826.0.1.3680043.8.498.96130267576375844096054483146813346156","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71180975285730441100082825359032677931","series":"1.2.826.0.1.3680043.8.498.11108618727209009321224346577603029752","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71311031275189592818091380416268522846","series":"1.2.826.0.1.3680043.8.498.18855543083213269127436205972402936846","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71405492944866967669069984757332849184","series":"1.2.826.0.1.3680043.8.498.98352139529443178809487367280297460136","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71622107620007706778976946784787365505","series":"1.2.826.0.1.3680043.8.498.19092188268997525116568396533639656682","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71660070729774772658965427776586988585","series":"1.2.826.0.1.3680043.8.498.54027099055984580203288261970980580088","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71686717249431292334159336816742076478","series":"1.2.826.0.1.3680043.8.498.12535635941566437105949137018817750977","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71707317731065645278250956442336515504","series":"1.2.826.0.1.3680043.8.498.87784033772506347488656043122518240500","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71736046309441396676679830874326797437","series":"1.2.826.0.1.3680043.8.498.10726227390932737041685128759190262110","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71753603955749189961638324806586359888","series":"1.2.826.0.1.3680043.8.498.24329565745226215835696326258945983959","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71792468450491064417953341293663396190","series":"1.2.826.0.1.3680043.8.498.67460400431031664504392902773108896239","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71836684300308056297405632410209248276","series":"1.2.826.0.1.3680043.8.498.32403005715854732114951168469944937985","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71898908031968158312013185809213669768","series":"1.2.826.0.1.3680043.8.498.41958482528745788559548964806703313845","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71985417286226511458068110832784023627","series":"1.2.826.0.1.3680043.8.498.93029383265993951006674736578557105140","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72017096768214637927798648334301077145","series":"1.2.826.0.1.3680043.8.498.62544423153862088684409804548041646385","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72095214441332094906401331319197122529","series":"1.2.826.0.1.3680043.8.498.96172941926191591429028150337200082125","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72172265826902387784016098389863005666","series":"1.2.826.0.1.3680043.8.498.28030351153713494240770751919659635870","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72191353854496806844040272237700346774","series":"1.2.826.0.1.3680043.8.498.10791491432226042488570455620442780344","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72248728319398595146365363062799556637","series":"1.2.826.0.1.3680043.8.498.10185012997837128985732936735230480723","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72270915206075693056344475438055463113","series":"1.2.826.0.1.3680043.8.498.28105566009145756793454200338184006657","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72313325634455670216423864504103231505","series":"1.2.826.0.1.3680043.8.498.63740759365614787422712546066998551906","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72375103774239787721371778221019402834","series":"1.2.826.0.1.3680043.8.498.77337675224309273961892971688419996488","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.72453426317225885428514595399496580896","series":"1.2.826.0.1.3680043.8.498.88878690142615467712708924289514130529","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72495590491484814220060275823761767516","series":"1.2.826.0.1.3680043.8.498.19530208440713101948880088825267824749","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72576069358772864331810785663326683534","series":"1.2.826.0.1.3680043.8.498.32678544363825790700269969675174700383","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.72653666963736486779382314125782779004","series":"1.2.826.0.1.3680043.8.498.22771961232465656825703912526215745300","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72697172162233919540021967137047646179","series":"1.2.826.0.1.3680043.8.498.98759837861547376952136290053614119317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72781608662300877077691853670862414309","series":"1.2.826.0.1.3680043.8.498.79070831684334097532761270893405461049","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72800659596099468282135738543971261904","series":"1.2.826.0.1.3680043.8.498.44964619588839538586785395159044074368","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72880034930990967339546757487847803284","series":"1.2.826.0.1.3680043.8.498.17164796227218190978557189144046681222","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72896586188735703676995894830815493004","series":"1.2.826.0.1.3680043.8.498.78491143578046910050987135676859825930","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72943191222961132416678221885831944435","series":"1.2.826.0.1.3680043.8.498.10614862175000191942994132823627296242","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73033521237848260228708170280586043408","series":"1.2.826.0.1.3680043.8.498.85244684265551850687266267150944302681","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73070182506490199787361298167985971577","series":"1.2.826.0.1.3680043.8.498.11345030916585262812472936117995401179","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73083786204651635096412213233334934941","series":"1.2.826.0.1.3680043.8.498.78761284108275412084554957512217156926","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73170541731840260065335942048010387599","series":"1.2.826.0.1.3680043.8.498.79963081900954972499529584603164252329","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73205916973275504240156555900460720571","series":"1.2.826.0.1.3680043.8.498.26925243648239229819963083722405549166","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73241519025825862373621891294281253158","series":"1.2.826.0.1.3680043.8.498.10174173132588087384637582359948276428","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73254365064404586162477089636000427751","series":"1.2.826.0.1.3680043.8.498.99365616068612847017451898228434484165","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73314900301724532218347713512381622074","series":"1.2.826.0.1.3680043.8.498.17309036938299884171457263204874175814","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73356831368999268488355650751758108912","series":"1.2.826.0.1.3680043.8.498.73651767620997919036292383599407680898","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73477249559479870799427355988138990349","series":"1.2.826.0.1.3680043.8.498.10808438620116488851204220472197184936","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73511703934549662392765328175356230615","series":"1.2.826.0.1.3680043.8.498.12540961570322621327942375905254770095","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220","series":"1.2.826.0.1.3680043.8.498.12239267188323122300736742237473727431","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73567807253694728043552196136704821457","series":"1.2.826.0.1.3680043.8.498.48710758865226348967988187082722434871","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73682608760611536443754406204042210115","series":"1.2.826.0.1.3680043.8.498.54969036959660317835427187967168325236","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73748007182569068263280098473557128226","series":"1.2.826.0.1.3680043.8.498.11827629928775899679950344029040995538","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73869775636181013203432845292462040922","series":"1.2.826.0.1.3680043.8.498.82058414464041163318632098011929976287","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995","series":"1.2.826.0.1.3680043.8.498.78991165425199309794359588807608652481","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74052473719853343027548507306186291134","series":"1.2.826.0.1.3680043.8.498.13189808124000815380723566175332214029","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74095402538454760179339842746572802563","series":"1.2.826.0.1.3680043.8.498.29644220842272195089160095338424980559","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74140806279409854828032496578072554031","series":"1.2.826.0.1.3680043.8.498.11195764477495822110193359777420239748","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74206939384612669977629625310277500302","series":"1.2.826.0.1.3680043.8.498.35773764155702984083853272851544095324","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74222284584657051737421776919599206132","series":"1.2.826.0.1.3680043.8.498.67087396943324419881058153518293950036","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74243110959141459268838217468465880820","series":"1.2.826.0.1.3680043.8.498.71471161947618739724934427258972504756","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74274855596893595390880617896069524465","series":"1.2.826.0.1.3680043.8.498.13352739194155997839380929094945516595","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.74332475620045145049628676751586693385","series":"1.2.826.0.1.3680043.8.498.33441574962708649552786733748020308493","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74394784271003789564066563841806954326","series":"1.2.826.0.1.3680043.8.498.72879537541595753047219601099574641737","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74461421748415064477531528884191890661","series":"1.2.826.0.1.3680043.8.498.55423237211682778141707490653228429370","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74622031396885197580453304563088863451","series":"1.2.826.0.1.3680043.8.498.75406624340989145453920956214533216482","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74688733778107958348746767768064374883","series":"1.2.826.0.1.3680043.8.498.30922608911708960383802926414132141395","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74749071555443582954158991248361308204","series":"1.2.826.0.1.3680043.8.498.10094781245693702203234027303975497416","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74864145321761783409935288797327597738","series":"1.2.826.0.1.3680043.8.498.65637841134449342254374838995241671261","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74914646210033032631540802996162814268","series":"1.2.826.0.1.3680043.8.498.69227934226904137115183523764466785772","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74952769952682175467667812958859010329","series":"1.2.826.0.1.3680043.8.498.11162086423421508899451183896047524506","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75168480371573015758963049782687952716","series":"1.2.826.0.1.3680043.8.498.66826768020197733797316286926564407670","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190","series":"1.2.826.0.1.3680043.8.498.98272753977677818069076712502597328523","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75282577107802752393765861202996650638","series":"1.2.826.0.1.3680043.8.498.95493491246982485532806137894977846198","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75384600248630012126841890313149674691","series":"1.2.826.0.1.3680043.8.498.12044665211032604623709502539368933287","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75419034794059601820999512200601915669","series":"1.2.826.0.1.3680043.8.498.25770033293028324871335077940837533642","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75518253718111135068488417046278011303","series":"1.2.826.0.1.3680043.8.498.11644169166882424342717893186413304230","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75523541256169382504121567002441615798","series":"1.2.826.0.1.3680043.8.498.49312870099917996672062214310434490369","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75577846246542841539613557336025271434","series":"1.2.826.0.1.3680043.8.498.13345539619619583385481595616599321842","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75627574440860942023321071382766359827","series":"1.2.826.0.1.3680043.8.498.12829920936294976359712059979288536042","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75670996863043899245632198549551296292","series":"1.2.826.0.1.3680043.8.498.11890256300245105367200479057897138345","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75742195884147990442829552992649204078","series":"1.2.826.0.1.3680043.8.498.29526295681061650310181811505369966009","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75760698509105388883811155432691667447","series":"1.2.826.0.1.3680043.8.498.99245479833799426246462204698253071897","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75800418125597325281807386716279941068","series":"1.2.826.0.1.3680043.8.498.12729906681465461073576644020461384909","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75926011938952404257302292495731660728","series":"1.2.826.0.1.3680043.8.498.29479668125183415410589040755055134341","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75980374267002793040832157122311396174","series":"1.2.826.0.1.3680043.8.498.10120851566934647467741443199105830080","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76065434429437758192282562351089010088","series":"1.2.826.0.1.3680043.8.498.16102324771846052242462059297726471957","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76107330932071966465869847071480517541","series":"1.2.826.0.1.3680043.8.498.46877833053431753523369149387922571283","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76167153566894354739332364376483514559","series":"1.2.826.0.1.3680043.8.498.12520260525085903169007972090517330378","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76222770668660274805993655800271653712","series":"1.2.826.0.1.3680043.8.498.12540303068436671410700465746415446923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76344826994380951498651448092980303697","series":"1.2.826.0.1.3680043.8.498.12790746304889263359684564732794216988","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76514504104121634569792846980451896030","series":"1.2.826.0.1.3680043.8.498.12073428653601333795057284607877417404","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76543241221133987658552440969465372120","series":"1.2.826.0.1.3680043.8.498.86722822449498134854821215470747338063","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76620177268652814425535081607587091903","series":"1.2.826.0.1.3680043.8.498.14642773838188812082441412831203216478","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76680092665218514711300086707222109323","series":"1.2.826.0.1.3680043.8.498.13246188781588477465107497226762948660","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76772148369531098768751348676329945789","series":"1.2.826.0.1.3680043.8.498.77152692753258163277013306325966965746","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76862911430418639277808760660745763387","series":"1.2.826.0.1.3680043.8.498.18432255815871147230239007124461696660","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76909970028125318280995002129970165723","series":"1.2.826.0.1.3680043.8.498.12873607961255672617573093985275367042","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76993922254007307143291254174266755701","series":"1.2.826.0.1.3680043.8.498.10686518326965401956641141661862799734","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77037121474756969001683126693104350509","series":"1.2.826.0.1.3680043.8.498.12761501371002041139663943225109517505","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77060858802024387619459645469871885412","series":"1.2.826.0.1.3680043.8.498.82630629865191944825575585635731723969","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77155817997584903787597001328868124855","series":"1.2.826.0.1.3680043.8.498.27408493115565595699172936885691361489","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77183368292104509335580564463404842046","series":"1.2.826.0.1.3680043.8.498.99436810570834680261801979132941597558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77261014467268552758826551926128217025","series":"1.2.826.0.1.3680043.8.498.11625689593740731204505271632835501187","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77283213969268676660453937180363826917","series":"1.2.826.0.1.3680043.8.498.73825822682646554055659868113608771671","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77290477043075828139630734968573461136","series":"1.2.826.0.1.3680043.8.498.11766863910025714885289254628962736093","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77319661699409997424800377883567027839","series":"1.2.826.0.1.3680043.8.498.57095784498129923786403432361370038210","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77370119801633353780677378448761665685","series":"1.2.826.0.1.3680043.8.498.57848158698812622851640997895347680380","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77391801789875882109036004511558148658","series":"1.2.826.0.1.3680043.8.498.60406176722020536395313472914865767931","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77433065090757862487482575794398458402","series":"1.2.826.0.1.3680043.8.498.70672095858536792651143054131135393078","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77498121906981512168120474112422492971","series":"1.2.826.0.1.3680043.8.498.12125008226099049116132831719890843246","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77513348456432770689938407624768928320","series":"1.2.826.0.1.3680043.8.498.13364138719412618459028916239206479044","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77546769004367585830455474984769277713","series":"1.2.826.0.1.3680043.8.498.96510371406003588017814583709591472588","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77584925163022886785284486387921312146","series":"1.2.826.0.1.3680043.8.498.64135917598269113455255878698504714356","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77626578713099735143745396709359607445","series":"1.2.826.0.1.3680043.8.498.56973034242233196716222827197430362930","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77716260980321491689913730053309892120","series":"1.2.826.0.1.3680043.8.498.12525026835187772263603381447869469281","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77740672313593763051900703940745812581","series":"1.2.826.0.1.3680043.8.498.24622631747447867583478060190348379131","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77800740811302686332282410929816056283","series":"1.2.826.0.1.3680043.8.498.85432342959062182023460095914540023968","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77823946901144772346251781172043480398","series":"1.2.826.0.1.3680043.8.498.10068571245104766581894385340662791471","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77889800516853576877948593628420228178","series":"1.2.826.0.1.3680043.8.498.68768017651219510520001794668119962852","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77933128806366660573360274958258917742","series":"1.2.826.0.1.3680043.8.498.13670229115015396985784595445422020257","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.78012314847102407375138809615977711653","series":"1.2.826.0.1.3680043.8.498.35132341930026740799596819414281897548","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78083057157844664959843364584162587026","series":"1.2.826.0.1.3680043.8.498.94325583194782969439770626548487119491","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78119223228067919465880137763593966456","series":"1.2.826.0.1.3680043.8.498.49589416877737982954473491237653687100","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78198993378521219839327914679211902952","series":"1.2.826.0.1.3680043.8.498.13002441693694896192886151270419050050","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78282552802075313852355443512555224586","series":"1.2.826.0.1.3680043.8.498.13230367408177330010209362415519933593","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78296478954955171036523274995555716268","series":"1.2.826.0.1.3680043.8.498.36341344904769640106588471249976309082","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78475145549489837635803968565380247517","series":"1.2.826.0.1.3680043.8.498.10829902296028198059607095249671370750","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828","series":"1.2.826.0.1.3680043.8.498.92631810116127614014780193051312108215","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78646473961447523033955716877524794707","series":"1.2.826.0.1.3680043.8.498.12312731078704231844740986266284284509","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78667999747216717251638754803697959701","series":"1.2.826.0.1.3680043.8.498.52678372774399774584566616895218773547","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78757011872917563830995996191850541453","series":"1.2.826.0.1.3680043.8.498.74129260922005983690782907865769676798","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78800983923143929298359366572020370031","series":"1.2.826.0.1.3680043.8.498.50506775968118078475103908405616311906","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78853931928960361455025737444267199231","series":"1.2.826.0.1.3680043.8.498.55903800781895708135401997930613947115","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79021499290691125513421477204330532755","series":"1.2.826.0.1.3680043.8.498.87894860685820013348102757642071555355","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79077219037094025366585138112899760966","series":"1.2.826.0.1.3680043.8.498.74776212473384372236475642810735008514","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79087227780630716132147615846840128474","series":"1.2.826.0.1.3680043.8.498.70308429783838754326143554815706486727","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79165531588966776246377098630025444815","series":"1.2.826.0.1.3680043.8.498.56499686866694625282161981189331298152","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.79225094031423353857449795712402404317","series":"1.2.826.0.1.3680043.8.498.12921195805635550811364487141402856865","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79247871688929479411597859576284194165","series":"1.2.826.0.1.3680043.8.498.12366162056092619824995238373253055574","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79274948549030940175536603063980168192","series":"1.2.826.0.1.3680043.8.498.38655045533665712348318726629981137845","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79292862882980756453890778996691993351","series":"1.2.826.0.1.3680043.8.498.10094804580185303413150752598948199980","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79341955397030543255969676347554165142","series":"1.2.826.0.1.3680043.8.498.12806100301143525879172973084948648255","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79551456190098389081551755482042135791","series":"1.2.826.0.1.3680043.8.498.11532754827112896038346820719643503615","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79565393323323007178523850093527250842","series":"1.2.826.0.1.3680043.8.498.24370456519764002841561390995439205373","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79636443202100988025435536780567446582","series":"1.2.826.0.1.3680043.8.498.48514135176333586149731254352385270915","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79672421957132993896448375311641998688","series":"1.2.826.0.1.3680043.8.498.76867591169736790211097631216692457884","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79798779812052605703861699094268016306","series":"1.2.826.0.1.3680043.8.498.13376506175279969490685474159780845938","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79977436705480715175480363175316375576","series":"1.2.826.0.1.3680043.8.498.25550822983519136385602595070736494736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80100785345864355815698147663964005920","series":"1.2.826.0.1.3680043.8.498.30646625998032280400787874845819002386","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80161127293548940216292842707683784128","series":"1.2.826.0.1.3680043.8.498.40214391464167502509965701964016380341","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80207056225476761432712249807198836187","series":"1.2.826.0.1.3680043.8.498.17056829945034252933762640553104305658","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80275294885165896124489476449616436102","series":"1.2.826.0.1.3680043.8.498.59500119362922787437450717128676246771","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80292580287849976694993628109721422478","series":"1.2.826.0.1.3680043.8.498.10028049225658146919936353067470558456","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80313580693550912024103221258669390487","series":"1.2.826.0.1.3680043.8.498.13204774805688968926221487550895640935","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80391378873630377472192017862302619947","series":"1.2.826.0.1.3680043.8.498.39774280718843931706638222730907681081","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80458507774711142534182397228254444268","series":"1.2.826.0.1.3680043.8.498.76005864791787691140225902505176386909","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80556515221794602430701573588498144286","series":"1.2.826.0.1.3680043.8.498.47847126325153674593232674399754424325","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80589796918823112368758999678369292754","series":"1.2.826.0.1.3680043.8.498.55711637064780371051050208348381478595","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80629565747112614845746475820152813307","series":"1.2.826.0.1.3680043.8.498.10606437737421286455565523325423668556","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80643480554795312280333659144941545925","series":"1.2.826.0.1.3680043.8.498.13084323451878816167379438588307033143","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80661392139668653552110941896475066204","series":"1.2.826.0.1.3680043.8.498.30298134786899685387838512603649529774","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80726797831040227285328789648680519664","series":"1.2.826.0.1.3680043.8.498.44324972524014279490754731266472011904","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80758336123908522542787716219939751704","series":"1.2.826.0.1.3680043.8.498.51683058394397045685093146745477886901","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80861108320341696163987804408524915999","series":"1.2.826.0.1.3680043.8.498.93139468516191732600953070231674414982","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80886525412417948114478368374120704834","series":"1.2.826.0.1.3680043.8.498.40004125387519878723909691289044160372","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81012548926053584036386949658594985125","series":"1.2.826.0.1.3680043.8.498.83120070369255204594938179931784605823","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81031057654953109295788159826841315360","series":"1.2.826.0.1.3680043.8.498.48035613347478337183026020095740677392","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81103809943451790668501089584752866199","series":"1.2.826.0.1.3680043.8.498.11927016466735197091195763814669406371","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81248820934296461385324385812581970340","series":"1.2.826.0.1.3680043.8.498.12977795954929514023557764866772825872","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81327951282145199298572980264557740321","series":"1.2.826.0.1.3680043.8.498.11694546796694822500685536388211170190","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81347841116019279082226143308588516787","series":"1.2.826.0.1.3680043.8.498.11511345429186716681341225820069122887","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81407504276375197122586620633759174421","series":"1.2.826.0.1.3680043.8.498.18223757901898769428630212394867134831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81466575994184158172641823163654418018","series":"1.2.826.0.1.3680043.8.498.30058757417820372879651519363586786142","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81540134120507822899514611863725372563","series":"1.2.826.0.1.3680043.8.498.76489631330208234413277894637525235403","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81590494282169434250974061642267472166","series":"1.2.826.0.1.3680043.8.498.10131476627692785270610024227964258831","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81631925197682464058124413934958934086","series":"1.2.826.0.1.3680043.8.498.27836604411955606648504559764951218908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81692996137620934043276253858381812472","series":"1.2.826.0.1.3680043.8.498.33161472925160877415643524959927536016","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81738571920712657790861120957086010772","series":"1.2.826.0.1.3680043.8.498.50662716624860950107973769066962655604","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81790990482142045640192729399175106255","series":"1.2.826.0.1.3680043.8.498.15445774213462654244419734144396133360","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81797437102217585216278133168092577952","series":"1.2.826.0.1.3680043.8.498.11654006190642773933646946766396491212","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81832287065207835637503039077685315236","series":"1.2.826.0.1.3680043.8.498.62505248537820765535894997419438244820","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81927409938015158663782061022054347955","series":"1.2.826.0.1.3680043.8.498.86506438995822965018676401023124007000","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81964000393640770822199916465038759069","series":"1.2.826.0.1.3680043.8.498.72589928460235427991574917649631613829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81984200870681776955493738987078746894","series":"1.2.826.0.1.3680043.8.498.62425806606678717384923368762514956290","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82053853036886620816821481139413594407","series":"1.2.826.0.1.3680043.8.498.98623547060573183858325841715013398154","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82144602300666174450721045102799681426","series":"1.2.826.0.1.3680043.8.498.11205328139598874243169270140152389117","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254","series":"1.2.826.0.1.3680043.8.498.10436783375077399659860593777522979072","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82272793832748838882752586678639564181","series":"1.2.826.0.1.3680043.8.498.41961979535597508405704716333976481648","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82440268452598207215458064865523988222","series":"1.2.826.0.1.3680043.8.498.13363925189876349494491157924530196935","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82481056562260046738834822905709207128","series":"1.2.826.0.1.3680043.8.498.72178874561594188578353682293925773817","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82548251499640077572026404183402934330","series":"1.2.826.0.1.3680043.8.498.48180971530937408459950159038227118964","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82582558624896662901163827145277490232","series":"1.2.826.0.1.3680043.8.498.77909010308247329813886619122842685275","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82591599611540849522609405219902904134","series":"1.2.826.0.1.3680043.8.498.12725312486952325042086842699865776140","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82744514729782879778828512295184306538","series":"1.2.826.0.1.3680043.8.498.65064301086839674297861376201579157258","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82858981777405241601610241542173548956","series":"1.2.826.0.1.3680043.8.498.94394482160369741879330962335215834182","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82967086493280898295669930903240677729","series":"1.2.826.0.1.3680043.8.498.15338873980989064268537718068377093935","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83028939839767027342482365275441078777","series":"1.2.826.0.1.3680043.8.498.97009040630091346141430792875334586524","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83055401642066749019091368358783700783","series":"1.2.826.0.1.3680043.8.498.71573832810660461510453881616155100087","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83108474538388106770920972885900596508","series":"1.2.826.0.1.3680043.8.498.97906866752714228642116375785635738625","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83220896521302064187926748035734200732","series":"1.2.826.0.1.3680043.8.498.12831891567242694506865239421534581321","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83237706274929081175032033288272528919","series":"1.2.826.0.1.3680043.8.498.45704455793665057997348381631848926532","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83271280040777227673434567660749430744","series":"1.2.826.0.1.3680043.8.498.64953540258530461194043983346688690327","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83456649088980869429062498344457162631","series":"1.2.826.0.1.3680043.8.498.12353569826935775526131268886029588875","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83509478059288902959149676250911654986","series":"1.2.826.0.1.3680043.8.498.22561712693907290559584400802888880144","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83602205818477323486169938182117122488","series":"1.2.826.0.1.3680043.8.498.12258709458724372070339877561870438167","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83747418172021065605877274971789603771","series":"1.2.826.0.1.3680043.8.498.40727656654725982280997715544385150567","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83956129905762530953192730040209719910","series":"1.2.826.0.1.3680043.8.498.99500653389160477538089966598156853288","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84047134932018909998648460546980598926","series":"1.2.826.0.1.3680043.8.498.55422884995119816156004547416446996878","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84062980592611347665739105270731842220","series":"1.2.826.0.1.3680043.8.498.10779279686050346512541058812264531082","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84123495463663584148839831457665741488","series":"1.2.826.0.1.3680043.8.498.91211297985209342221361798190671328503","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.84238814030979264404768702945290704724","series":"1.2.826.0.1.3680043.8.498.57157585685689683821905084104857480803","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84347051904829814300575579670995018991","series":"1.2.826.0.1.3680043.8.498.65367090315555047706272263715481004183","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84483080389344581106129884869678629612","series":"1.2.826.0.1.3680043.8.498.47589844573115635635947161400862967336","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84612593835203397696029057940345763682","series":"1.2.826.0.1.3680043.8.498.77436296258471961003881520725298917372","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84661341697154104963062177064495044195","series":"1.2.826.0.1.3680043.8.498.10710517364627046292532532406461470673","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84748479740284893375192048321707057482","series":"1.2.826.0.1.3680043.8.498.29299364877184273478984172621088660462","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84855341377363129926967387137216814518","series":"1.2.826.0.1.3680043.8.498.79728494714572796028919424648818268860","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84896756080952941537556818257863912066","series":"1.2.826.0.1.3680043.8.498.24474710828019883103282399955578907138","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85031127696558971105465423134104223548","series":"1.2.826.0.1.3680043.8.498.44663619361920327336257097529205987315","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85095389649021071892478435298763928925","series":"1.2.826.0.1.3680043.8.498.86291053100199431598261264758521988730","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.85137704768728261642637391657270006448","series":"1.2.826.0.1.3680043.8.498.79380868105359858408664610704122401024","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85294435004325211584698794441083866921","series":"1.2.826.0.1.3680043.8.498.11532056423427645519022831152918714584","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85333821516157983299252782678788406019","series":"1.2.826.0.1.3680043.8.498.13335659559326939760046613953511005440","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85391052076760686301971262244561025345","series":"1.2.826.0.1.3680043.8.498.42758607033474245103952263630977329499","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85416107585703628549041626609272707431","series":"1.2.826.0.1.3680043.8.498.11003432376261546407453277110049150283","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85454884397223412543050876117416146750","series":"1.2.826.0.1.3680043.8.498.23284035277276366928446377205346723215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85499036441349866810703945289793057098","series":"1.2.826.0.1.3680043.8.498.70517504785892335698865664328256289826","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85566900782101872452970957532250214590","series":"1.2.826.0.1.3680043.8.498.41730844389731853321546912199638895375","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85646781189339988389190891868001374533","series":"1.2.826.0.1.3680043.8.498.28855904501955688550099935362535144235","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85663162480758065951465617999516770647","series":"1.2.826.0.1.3680043.8.498.18854112789212226545254189514038359680","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85719235937472088594405415004352845221","series":"1.2.826.0.1.3680043.8.498.39754968057808358437958115598013048458","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85735729179216019275685564112535593988","series":"1.2.826.0.1.3680043.8.498.89504731996576821954046411050092720536","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85849908712977285006159496192673293501","series":"1.2.826.0.1.3680043.8.498.78815865333181868838740721419161341941","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85879572979834912642178551098379145607","series":"1.2.826.0.1.3680043.8.498.25713691644011383038827574959926507401","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85890890545815190189345870912381925290","series":"1.2.826.0.1.3680043.8.498.31508181633282198598334424124106457013","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85943737587102032370951456635167782343","series":"1.2.826.0.1.3680043.8.498.53133903077645286051698317814063642175","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85976668143927055109184433855426232055","series":"1.2.826.0.1.3680043.8.498.10008266800101012018024124219906684387","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86020966816754359840982061252012700091","series":"1.2.826.0.1.3680043.8.498.10021275280365286643025336096155733093","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86042942187650835661744645315369706668","series":"1.2.826.0.1.3680043.8.498.85339241854355273939031164265419688659","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86117639434986594266345773277419881144","series":"1.2.826.0.1.3680043.8.498.89020365744889614472168677438590815014","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86239793008693746155478763153154040128","series":"1.2.826.0.1.3680043.8.498.28734032347562465141142921312219116808","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86263264208309369957750915520115390972","series":"1.2.826.0.1.3680043.8.498.62017634797215100238942388727374499946","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86285715293767810771609523819042157072","series":"1.2.826.0.1.3680043.8.498.95920161880285938100850609078176940969","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86332035495941070455209594003499488889","series":"1.2.826.0.1.3680043.8.498.54085398830686309541973596060325241712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86405969333062807387894608315897106350","series":"1.2.826.0.1.3680043.8.498.44972552972248977326688599998758397768","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86416984885007871128151024515780322767","series":"1.2.826.0.1.3680043.8.498.43248899960023045794753360902147850124","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86435890312227285216810145658548164773","series":"1.2.826.0.1.3680043.8.498.89767874283411320208589587786038173712","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.86532643576908316499388520854820002133","series":"1.2.826.0.1.3680043.8.498.13323939871384294038125113393001664575","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86625876521226465945114246972542881955","series":"1.2.826.0.1.3680043.8.498.62349292802623752502024271327351749709","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86715997235054599024919893656486735623","series":"1.2.826.0.1.3680043.8.498.96065538613520261350531162200293494075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86881633316011036335786865282908198831","series":"1.2.826.0.1.3680043.8.498.94629383375509851421655979917672780041","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86974853023888933977090539174537083409","series":"1.2.826.0.1.3680043.8.498.71005486601352243975023540142548789506","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87046380372347615219261898797043742937","series":"1.2.826.0.1.3680043.8.498.45302098568492865101576931194641381981","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87101276883427329522365304528449003589","series":"1.2.826.0.1.3680043.8.498.26378723220157109852447946816607709689","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87198191797585172912979594638355685739","series":"1.2.826.0.1.3680043.8.498.56713417954624590986017915929622839464","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87214831397290763080243320084625847148","series":"1.2.826.0.1.3680043.8.498.13205345053673784110354641437447055572","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87241650441242408799533998101943024283","series":"1.2.826.0.1.3680043.8.498.25884670958401216509104734155791538012","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87262205711949358943515326577886508585","series":"1.2.826.0.1.3680043.8.498.75597487935784258259344219955199149664","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87404730020015145302086817782843205171","series":"1.2.826.0.1.3680043.8.498.90476457720649962309047158701964003344","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87415625824966124550466651648244553892","series":"1.2.826.0.1.3680043.8.498.13854897225755380055000827760979723076","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87516159856486535118712846576943170757","series":"1.2.826.0.1.3680043.8.498.53192222133946709266151473397827893942","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87560916423885591029120178291827033868","series":"1.2.826.0.1.3680043.8.498.11943900250842666704550057813255762090","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87650297321648368205324227429768256110","series":"1.2.826.0.1.3680043.8.498.10002854609799107704163984063758739358","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87680825247567493080163504511649789348","series":"1.2.826.0.1.3680043.8.498.12144325867630537047963938440779454987","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87761573479019057997334508266888503487","series":"1.2.826.0.1.3680043.8.498.34832358161754442886734081864813029817","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87799106106199445436621174506197684035","series":"1.2.826.0.1.3680043.8.498.75721737770771898000181830974700260011","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87876800169766592603600598167358214111","series":"1.2.826.0.1.3680043.8.498.25312182521554565832106364394721018502","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87998832540756872371878613339628615880","series":"1.2.826.0.1.3680043.8.498.33447832840203674996795211373294285212","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88070896604159829163161019117970440189","series":"1.2.826.0.1.3680043.8.498.10367054773456936567787798597239735908","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435","series":"1.2.826.0.1.3680043.8.498.30657822634139739188979013992800834075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88211355128040927831407007101167868685","series":"1.2.826.0.1.3680043.8.498.45851067658006175559357702352441520093","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88368930449171804348215887291291274615","series":"1.2.826.0.1.3680043.8.498.97614516343707857416927166626523895037","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88466268868654468582661984391239692466","series":"1.2.826.0.1.3680043.8.498.74179315631369412454565579166912906525","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88523737187772411911621088333701306485","series":"1.2.826.0.1.3680043.8.498.42320199910296371276691572957880895576","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88570871026022705696415758052703394453","series":"1.2.826.0.1.3680043.8.498.36432071232842492156394547475394482217","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88609906210780568637209291182572648571","series":"1.2.826.0.1.3680043.8.498.67944590684174955298611097616045731834","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88653085234639096911379842849193874077","series":"1.2.826.0.1.3680043.8.498.11961547933292499817919736726123583089","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88682169866098509464628948503453852928","series":"1.2.826.0.1.3680043.8.498.18924250666088131110308412598704611827","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88711356400820562340756013729440417015","series":"1.2.826.0.1.3680043.8.498.70282109130987037598004570915152213420","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88753415430913291809543943690942744997","series":"1.2.826.0.1.3680043.8.498.56131399386501990720907909378216877384","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88827276474694165071267273766912871459","series":"1.2.826.0.1.3680043.8.498.75220792547031947652188148595580731525","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88872155446935518174304990308380906112","series":"1.2.826.0.1.3680043.8.498.97542156257832829173003387137728090946","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88899519473612920868099376939780201067","series":"1.2.826.0.1.3680043.8.498.67610333890666165583980840002133197075","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.88964371740619947193957393682095448256","series":"1.2.826.0.1.3680043.8.498.15714740765615489035764771553664236990","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89048907638162719840222508855779172366","series":"1.2.826.0.1.3680043.8.498.29502582041163780468851107693322474160","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89110271437785174288044757208973128908","series":"1.2.826.0.1.3680043.8.498.10683414454273810085173686414614693971","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.89190511197564127079697151213577962649","series":"1.2.826.0.1.3680043.8.498.11695344721391685327217675783426604913","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89247482662611898098489676481780144186","series":"1.2.826.0.1.3680043.8.498.72987718811829980654561919380066974877","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89307235192406983202868051697765144920","series":"1.2.826.0.1.3680043.8.498.83022921455895055965147306763792827867","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89346975863478164334871204856160071946","series":"1.2.826.0.1.3680043.8.498.27291957818276321626461896874806018407","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89357823844365374776758395347764601030","series":"1.2.826.0.1.3680043.8.498.11444330045656161439902639298456724803","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89423600201830685256510321877396547497","series":"1.2.826.0.1.3680043.8.498.11244404913868331637421697086402685108","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89569762088162031469299314053759758927","series":"1.2.826.0.1.3680043.8.498.13290180478653698205120215030695601227","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89584627944947813918297925813120247226","series":"1.2.826.0.1.3680043.8.498.65829605973892233967641683224951349962","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89729486637365590877469535082675069395","series":"1.2.826.0.1.3680043.8.498.14850214653674811808490914553316538285","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89829843399218627892046362324057240535","series":"1.2.826.0.1.3680043.8.498.12276557310495585517870872891996770026","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89857801102910604766959864464865910585","series":"1.2.826.0.1.3680043.8.498.15691359843868808510087320561768874925","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89953731420879076053772747816096629231","series":"1.2.826.0.1.3680043.8.498.27361911129336777417798052247772351037","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89987866402625167201267936974120824712","series":"1.2.826.0.1.3680043.8.498.12855615115406461383509135158499651324","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90057130466053377217652879986591534745","series":"1.2.826.0.1.3680043.8.498.91795948530650188788359402423527966702","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90159195278632080234412316065469499332","series":"1.2.826.0.1.3680043.8.498.11587113107887118432700439151667095391","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90193654806390401129887447441469319421","series":"1.2.826.0.1.3680043.8.498.72771623742953306690339962364774388646","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90227414485305096825508911050013279704","series":"1.2.826.0.1.3680043.8.498.78481198491132173697693002556443578496","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90264300606047579088362539636050224478","series":"1.2.826.0.1.3680043.8.498.11826580423130736642070604555322620024","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767","series":"1.2.826.0.1.3680043.8.498.10851414972318193746245591053862925239","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90299615112155255774026824400461236845","series":"1.2.826.0.1.3680043.8.498.12499793666821447790127441255429860966","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90340114291852810896219650388952790164","series":"1.2.826.0.1.3680043.8.498.56827323449032315876089943915400808762","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90433488790413321102007716267340900056","series":"1.2.826.0.1.3680043.8.498.88967144440686012823021873764073919076","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90483774039833567637730989859822082272","series":"1.2.826.0.1.3680043.8.498.22229978303175729030769728979072360790","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90508461713737190527021531642732695353","series":"1.2.826.0.1.3680043.8.498.48617377907026015563814719607602447455","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90573769367031841574114648773801425781","series":"1.2.826.0.1.3680043.8.498.60010287430380635952578377164831277752","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90668646188784648688693620159738574680","series":"1.2.826.0.1.3680043.8.498.60832056848659567714466064262862061363","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90751504670572401790063555695835439317","series":"1.2.826.0.1.3680043.8.498.68426977468611641283160320618515729609","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90764713690393618577990540005126771712","series":"1.2.826.0.1.3680043.8.498.54665683653539939600592935127031644440","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90893882071188409362296099956462540506","series":"1.2.826.0.1.3680043.8.498.12638469077901420797944209374003491783","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90985037792656682958281230441654772668","series":"1.2.826.0.1.3680043.8.498.44206448801945329553245596126829355968","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91122873808284731048016330734204992698","series":"1.2.826.0.1.3680043.8.498.53568642311904006488469225254206908573","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91125742927944703615970502025043944089","series":"1.2.826.0.1.3680043.8.498.11920052156775850630715515266315411180","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91168270990167204047014161851340054645","series":"1.2.826.0.1.3680043.8.498.10192555998978011468092099858688350599","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91230059913252604666971822137553690811","series":"1.2.826.0.1.3680043.8.498.75419609407747356048249894576353544063","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91262503278090288446189575519187653125","series":"1.2.826.0.1.3680043.8.498.46561780341968307745572741679387467385","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91346221361264140052760282748336176394","series":"1.2.826.0.1.3680043.8.498.65719248490445636856795271831537869981","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91380337141071062166571333367919885984","series":"1.2.826.0.1.3680043.8.498.88421313148497313021719003634381533418","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91391446167553025113367459399831345295","series":"1.2.826.0.1.3680043.8.498.89137093393967512219335072909822671125","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91497176426480836856097456438014037999","series":"1.2.826.0.1.3680043.8.498.83771116319571434169160202281532642122","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91581717063042044543371480238854954692","series":"1.2.826.0.1.3680043.8.498.50012142568264629332097037167418911223","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91660278231317256353374962767408532443","series":"1.2.826.0.1.3680043.8.498.93932670313239792194209248365717902295","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91742054227008690570256869200116133922","series":"1.2.826.0.1.3680043.8.498.71437442235543247291351242091601594333","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91839696756303525362262251587171486273","series":"1.2.826.0.1.3680043.8.498.64104806459673864498051173908595386301","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91897419008563027309752122760214275182","series":"1.2.826.0.1.3680043.8.498.93577622165138039547391958680744231193","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91940000162714863613139612302783091395","series":"1.2.826.0.1.3680043.8.498.87822711464412650386602141522312376974","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91991985280680856830060431668414014465","series":"1.2.826.0.1.3680043.8.498.83470143466087765200596313650414432246","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92071199603599654827155753204585082303","series":"1.2.826.0.1.3680043.8.498.98758954024839659489611228500426066953","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92127426417720107708158713612888411025","series":"1.2.826.0.1.3680043.8.498.13152663673168221125810835673989602431","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92285140961160393284537044926797259671","series":"1.2.826.0.1.3680043.8.498.82139069831116224081239148534479868644","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.92326467051364762529119362944559591665","series":"1.2.826.0.1.3680043.8.498.12132346591110288207569905791200966145","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92407834172018110142589158281575319154","series":"1.2.826.0.1.3680043.8.498.86421273711917246726993816513365063433","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92449204325697143336618138322280397765","series":"1.2.826.0.1.3680043.8.498.10141627530321625633100754242326318568","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92581137608969847756039955144478716506","series":"1.2.826.0.1.3680043.8.498.65606634471674348800906629085846649923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92621720172216795135762712303124932801","series":"1.2.826.0.1.3680043.8.498.11451040277877573049115654421893669148","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92655291659311409459093330375014445330","series":"1.2.826.0.1.3680043.8.498.26535713752594360973811897204696052172","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92715232057122971236003034574127234433","series":"1.2.826.0.1.3680043.8.498.33620833092004492608106647909390017547","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92839397144213404004277739877191283401","series":"1.2.826.0.1.3680043.8.498.70410912011731586117083555596432363510","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92902692647544828045768164058889405369","series":"1.2.826.0.1.3680043.8.498.46393817427064969334261530086679148892","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93026010065162534828966791702007067882","series":"1.2.826.0.1.3680043.8.498.69796242803634543793290450097767290971","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93096918345463865518663434763610016449","series":"1.2.826.0.1.3680043.8.498.17406000632496904587414645546858349874","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93177444420568709574758402836371200945","series":"1.2.826.0.1.3680043.8.498.14066364851192692642197287030747595038","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93188773180403601524488403580611534053","series":"1.2.826.0.1.3680043.8.498.14187447663931740523076772707797779851","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93230313557251171119475173996942942855","series":"1.2.826.0.1.3680043.8.498.51611137538319854326864290108043801788","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93344237572402384894445814841725878706","series":"1.2.826.0.1.3680043.8.498.37395650365235462400975159231991829562","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93385433312281352893457225415812880948","series":"1.2.826.0.1.3680043.8.498.13341521410256326813903015962232847425","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93555726419626687118723083838655757151","series":"1.2.826.0.1.3680043.8.498.53215791983841335496130129585789849157","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93655619137163635289231840294252859840","series":"1.2.826.0.1.3680043.8.498.62684162473061135330599500834648504870","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93722352425168280299901366261374293722","series":"1.2.826.0.1.3680043.8.498.63745190551153949540496571911559687856","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93774667493652492119129601342627338680","series":"1.2.826.0.1.3680043.8.498.49006692182550551287409389729060453851","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93902628119199416703899320994283313812","series":"1.2.826.0.1.3680043.8.498.20417575944064446496275915763026594189","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93985093305024344873337249433396141838","series":"1.2.826.0.1.3680043.8.498.52659794158233936327227068593212479306","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94027547274004055924993074754507802303","series":"1.2.826.0.1.3680043.8.498.13256948215040347146796346677701737544","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94117540970936035429846288243624086693","series":"1.2.826.0.1.3680043.8.498.23377313391395985222476613461841919560","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94252884278314596961426629318911909075","series":"1.2.826.0.1.3680043.8.498.83800650880078783193342720503385776025","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94296943940827583101208235208463322140","series":"1.2.826.0.1.3680043.8.498.10779618068700868413205443781177875879","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94335780236370744781443631231792792083","series":"1.2.826.0.1.3680043.8.498.59240970341534630559382903344275997828","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312","series":"1.2.826.0.1.3680043.8.498.19808351390262770843383719107538938411","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94553664348118184200007654265863262684","series":"1.2.826.0.1.3680043.8.498.13306075200739671927704715029730147076","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.94594009911808653224418094594012527944","series":"1.2.826.0.1.3680043.8.498.19525394035488600069576706189863701527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94625454484290996254750481989036812516","series":"1.2.826.0.1.3680043.8.498.10246654810967499834621506583435366904","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94658317716039594639343437780115140531","series":"1.2.826.0.1.3680043.8.498.25909945610099659963463302260548449575","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94729634553835820944505159643192054175","series":"1.2.826.0.1.3680043.8.498.42086919465223711971737477091424480853","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94761755148413531477206243804832613511","series":"1.2.826.0.1.3680043.8.498.47270208637957219708492401721124163388","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94798723424795346945893598929864335472","series":"1.2.826.0.1.3680043.8.498.51895130262128464195537488252990174847","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94921929374845793599006362190119159537","series":"1.2.826.0.1.3680043.8.498.21978189593939718765390300892765894419","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94954489705497211843190734684710878341","series":"1.2.826.0.1.3680043.8.498.11305489946818423705991880605253921753","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95026524886584356226576936309513335214","series":"1.2.826.0.1.3680043.8.498.44280686858591280561642730138598966808","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95080449640993916851075185393444497308","series":"1.2.826.0.1.3680043.8.498.75004838567294993812105981464719302970","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.95113114010658169441709314948308683982","series":"1.2.826.0.1.3680043.8.498.11603598324893760667244502908827829816","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95196756237527666678816095708949563380","series":"1.2.826.0.1.3680043.8.498.12461856792047009298769312736684503906","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95256518245659249144569318023469597946","series":"1.2.826.0.1.3680043.8.498.33874727546620349126401182540392352002","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95324318295506605557933083155792587297","series":"1.2.826.0.1.3680043.8.498.26038516600328506746395574208733769666","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95351925765761617117057799846763403963","series":"1.2.826.0.1.3680043.8.498.10107892099353987154877697804511144517","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95433650759419050864053270236035947981","series":"1.2.826.0.1.3680043.8.498.10984947269636505605889762116206618013","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95497572581157858069692837655072209020","series":"1.2.826.0.1.3680043.8.498.11717909564308502556445940550984222590","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95560861705581165570145459410262387310","series":"1.2.826.0.1.3680043.8.498.69514547033550129202491163361810779021","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95586477246200285088746818821808017494","series":"1.2.826.0.1.3680043.8.498.90470843071950787726822700796269112964","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95601371640669355936785637162110963091","series":"1.2.826.0.1.3680043.8.498.11840091397064461153576700724937879703","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95653228095828585867177997067051877351","series":"1.2.826.0.1.3680043.8.498.25734532720822452029346389887882545888","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95720534093121754146413060811354489247","series":"1.2.826.0.1.3680043.8.498.14970588894482357260975441182948103257","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95728724960383464194568391302477328450","series":"1.2.826.0.1.3680043.8.498.26593851196701162807184729874410399863","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95796009319232592137564145023096016872","series":"1.2.826.0.1.3680043.8.498.10442670743617551998965617347586180152","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95882731821729917461275694847968503951","series":"1.2.826.0.1.3680043.8.498.47944502948138267686826775362533050516","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95918254432056542701730614049432316728","series":"1.2.826.0.1.3680043.8.498.10649599094673397488976877225417313298","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96054287426681123062488435620352206601","series":"1.2.826.0.1.3680043.8.498.48057088056782824624221527379738051859","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96146854841734718524530740965296119916","series":"1.2.826.0.1.3680043.8.498.12390443803286049118533124254817054325","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96158169859491339496094295508194547199","series":"1.2.826.0.1.3680043.8.498.92565540815571060151431599460806588850","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96293531288862652713666056362328893428","series":"1.2.826.0.1.3680043.8.498.12089436998657865214662796518812392299","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96425049974772570897047688166336162674","series":"1.2.826.0.1.3680043.8.498.82031797675668574357894500242558588097","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96483513352751590579274577125356217169","series":"1.2.826.0.1.3680043.8.498.83076762802327682574190308388540279977","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96561978771090020509310535265236331946","series":"1.2.826.0.1.3680043.8.498.10215373994982244069875252005804351235","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96564709711765982786409987515735443801","series":"1.2.826.0.1.3680043.8.498.32946887228488740981740017928444177745","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96614828054790815108190478795539254570","series":"1.2.826.0.1.3680043.8.498.71631320898326695691856544638269900298","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96791010280967772394523567484937888186","series":"1.2.826.0.1.3680043.8.498.12323183025850610308221798768964711661","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96859317329616917022799512740280754327","series":"1.2.826.0.1.3680043.8.498.24374326547110018289693935892660087278","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96944926470854551064948200936284887742","series":"1.2.826.0.1.3680043.8.498.17089031015917282788228356171917995134","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97003157807630958179276831877591411623","series":"1.2.826.0.1.3680043.8.498.37820665935645070230527833857086913326","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97020520267554584684160936968569132254","series":"1.2.826.0.1.3680043.8.498.15019496205007136532467277726094586807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97087958215554026148585108910609248523","series":"1.2.826.0.1.3680043.8.498.51349609715383800795416727380546674685","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97135552871905937467775630206570358085","series":"1.2.826.0.1.3680043.8.498.42370360164489905470833956383477117178","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97234348688223610934999923021953923903","series":"1.2.826.0.1.3680043.8.498.25291674038625885029554863087789815810","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97259766233250760343109933426776941754","series":"1.2.826.0.1.3680043.8.498.56706745060936296635447230672669540356","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662","series":"1.2.826.0.1.3680043.8.498.96963728552261384768664457442339822107","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97303893645051596044169857585815133097","series":"1.2.826.0.1.3680043.8.498.46110754392252165049326817387997687394","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97400307568415006244714024769069514497","series":"1.2.826.0.1.3680043.8.498.17601578813530971974345590985808765484","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97488501634729541923597957717971376373","series":"1.2.826.0.1.3680043.8.498.62362651647692710455623166714678685627","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97500632594529766269369977508352975364","series":"1.2.826.0.1.3680043.8.498.54449704104675415974120334089881685670","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97536230315600409983600607358927775480","series":"1.2.826.0.1.3680043.8.498.53662577695045559646722740685140207686","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97579738268290461886850261405719501384","series":"1.2.826.0.1.3680043.8.498.86612978937797429850646575476186348723","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97689748704772735695349883980413529091","series":"1.2.826.0.1.3680043.8.498.12739078443617777692476602089301541013","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97705920250069816684861966612302390614","series":"1.2.826.0.1.3680043.8.498.52513566286710537636929998374818718620","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97759980486481979519068526243106370997","series":"1.2.826.0.1.3680043.8.498.24823850635665269086360535525111416295","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97830923192341890652952842494018952920","series":"1.2.826.0.1.3680043.8.498.12949366467981045900243975360840451907","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97944261713902372188894456162512065499","series":"1.2.826.0.1.3680043.8.498.98742974851969969364935682059292090345","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98022738778461179495257509215244053198","series":"1.2.826.0.1.3680043.8.498.74778310044625381375405720020290395257","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98039072306618106705944452906395982381","series":"1.2.826.0.1.3680043.8.498.10412368104116500331014759259777905798","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.98081486779628580946589119341754353354","series":"1.2.826.0.1.3680043.8.498.72122742593414964273342820523451455808","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98107112397588450345153336193341117589","series":"1.2.826.0.1.3680043.8.498.12059819020529186254768447906836458060","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98146155044178593810385943365143374814","series":"1.2.826.0.1.3680043.8.498.31145361798136458903283678003595838792","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98181040478767526925214101502401448074","series":"1.2.826.0.1.3680043.8.498.41875167312752839502958293656379024914","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98315778542448798493600485116505234132","series":"1.2.826.0.1.3680043.8.498.74490336994702105524045318078944852534","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98343185597651455245996667389303915779","series":"1.2.826.0.1.3680043.8.498.32690826589467452468032253511599221531","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98454206323401382140655387224146512225","series":"1.2.826.0.1.3680043.8.498.10021416149166512820013390377829661744","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98581537941208552065033680853629122764","series":"1.2.826.0.1.3680043.8.498.54732965723779041839885549730737158662","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98676079331571304021387833410440136393","series":"1.2.826.0.1.3680043.8.498.46133157625901643761455340721759210597","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98781117432577714521231232440503778261","series":"1.2.826.0.1.3680043.8.498.47166337429963042718849265061622655908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98828753472895685940998888332947119084","series":"1.2.826.0.1.3680043.8.498.11567766964041195470722250777310793819","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98883718766501717651651876978607853803","series":"1.2.826.0.1.3680043.8.498.66239374265853329797447949310205204101","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98958651320983196982606333548522846934","series":"1.2.826.0.1.3680043.8.498.60101573180771667676144724691221097017","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99077241529686301278201179461896593659","series":"1.2.826.0.1.3680043.8.498.16112574584701202383065705160509289323","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99155175553749543768821558247962844052","series":"1.2.826.0.1.3680043.8.498.32064846033423974135930576445099674024","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99243902497374852166139435719549686908","series":"1.2.826.0.1.3680043.8.498.40961925256640246405476310150445918118","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99303079292320085374389086609761974820","series":"1.2.826.0.1.3680043.8.498.24700385801146576572340687965149447050","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99414163787650487720542840540379270468","series":"1.2.826.0.1.3680043.8.498.71124967928791576590781592560747597773","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99441773159476751698123795913150660766","series":"1.2.826.0.1.3680043.8.498.12416352477144175733916009485252637740","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99454657594216441037514296544803013407","series":"1.2.826.0.1.3680043.8.498.73432435645629705459780964145813296604","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99493257395588095270208592176887619190","series":"1.2.826.0.1.3680043.8.498.30586916288184089949364224280112178476","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99578894341571394220300473118855007095","series":"1.2.826.0.1.3680043.8.498.10735819076115257884811842648005978846","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99651725851820083142440965238449655973","series":"1.2.826.0.1.3680043.8.498.11923843963999687880127266084725136777","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99725523560254732094432244556623376711","series":"1.2.826.0.1.3680043.8.498.11779575322707075398677134300943825902","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99768154649604643975127490783989114506","series":"1.2.826.0.1.3680043.8.498.18865150805207093116215223644364468212","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99827460918477409069820779642161448917","series":"1.2.826.0.1.3680043.8.498.13846715339512218812564202626249574472","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99880021280666757817587834481314366555","series":"1.2.826.0.1.3680043.8.498.57075026057016289771660811830704812249","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99926624968240681772735102303531613209","series":"1.2.826.0.1.3680043.8.498.17200236650969779012824845263770515396","plane":"Sagittal","fluid":1,"slot":"SAG_FS"}]')
print('series 320 rgb, UID figes depuis crop130_rgb25d :', len(jobs))
assert len(jobs) == 1777
assert jobs and {'study', 'series', 'plane', 'fluid', 'slot'} <= set(jobs[0])


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def _to_u8(sl: np.ndarray) -> np.ndarray:
    sl = cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w) volume ORIGINAL trie

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    # memes 24 centres que crop130 (linspace dans la bande),
    # mais voisins ±1 dans le volume original, pas dans les 24
    idx_band = np.linspace(0, b - a, N_SLICES).round().astype(int)
    idx_orig = np.clip(a + idx_band, 0, n - 1)

    out = np.empty((N_SLICES, 3, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    last = n - 1
    n_clamp = 0
    for i, k in enumerate(idx_orig):
        k = int(k)
        km = 0 if k <= 0 else k - 1
        kp = last if k >= last else k + 1
        if km == k or kp == k:
            n_clamp += 1
        out[i, 0] = _to_u8(vol[km])
        out[i, 1] = _to_u8(vol[k])
        out[i, 2] = _to_u8(vol[kp])

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
        "n_clamp": int(n_clamp),
        "rgb": "dicom_neighbors",
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_rgb25d_320_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / f"manifest_rgb25d_320_part{SHARD_ID}.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "shape": [N_SLICES, 3, IMG_SIZE, IMG_SIZE],
    "rgb": "dicom_neighbors",
    "note": (
        "1 serie fluid-prioritaire par plan. Crop 130 mm (PixelSpacing) sinon "
        "centre 70 %. Bande 20-80 %, linspace 24 centres. Pour chaque centre k "
        "dans le volume ORIGINAL : R=k-1, G=k, B=k+1 (1 coupe DICOM d'ecart, "
        "clamp aux bords). Resize 320, uint8. PAS les voisins des 24 linspace."
    ),
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
            "n_clamp": None, "rgb": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        assert vol.shape == (N_SLICES, 3, IMG_SIZE, IMG_SIZE), vol.shape
        assert vol.dtype == np.uint8, vol.dtype
        rg = np.abs(vol[:, 0].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        bg = np.abs(vol[:, 2].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        print(
            name, vol.shape, vol.dtype,
            "min", int(vol.min()), "max", int(vol.max()),
            "mean", float(vol.mean()),
            "|R-G|", round(float(rg), 2), "|B-G|", round(float(bg), 2),
        )
print("attendu : shape (24, 3, 320, 320), |R-G| et |B-G| > 0 (voisins DICOM reels)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
